In [ ]:
# Scimago 파일을 분석용 CSV로 변환하는 중입니다. 먼저 지표 안내와 저널 정보 시트의 나머지 구조를 확인하겠습니다.Scimago 파일을 CSV 5종과 대조 스크립트로 정리했습니다. 파일을 전달한 뒤 교정한 현황과 S2 방향을 정리하겠습니다.# 1. 만든 파일

# Excel의 6개 시트를 분석용 CSV로 풀었습니다. 값은 바꾸지 않았고, 파생 열은 `d_` 접두사로 구분했습니다.

# | 파일 | 내용 | 행 |
# |---|---|---|
# | `scimago_journal_year.csv` | **주 분석표.** 저널×연도에 18개 지표, `fixed`, `in_window`, `small_n_lt30` 플래그, 파생 열 | 610 |
# | `scimago_quartile_long.csv` | 분야별 사분위를 세로로 정리(분야명 포함, 비어 있는 구조적 열 제거) | 1,824 |
# | `scimago_journal_meta.csv` | 저널별 ID, ISSN, 출판사, Scopus 색인 구간, 2025 스냅샷, 분야, 범위 설명문 | 34 |
# | `scimago_missing_reasons.csv` | 빈칸 사유(연도 행 없음 317, 원문 빈 값 100) | 417 |
# | `scimago_metric_dictionary.csv` | 열마다 정의, 시간 단위, 계획서상 용도, 역할, 주의 | 30 |
# | `scimago_MANIFEST.csv` | 파일 해시와 원본 xlsx 해시 | 5 |
# | `s1_scimago_merge.py` | 설치, 코퍼스 대조 2종, CHANGELOG, 선택적 커밋·태그 | |

# - 푸터 안내문 7행은 제거했고, 빈칸은 0으로 채우지 않았습니다.
# - `Cites per document`는 610행 전부 3년 창 값과 같아서 중복 열입니다. 사전에서 "사용 안 함"으로 표시했습니다.
# - **`Est. APC`와 `Est. value`는 SJR에서 계산된 모형값**이라 분석 의미가 없습니다. 같은 이유로 "사용 안 함"입니다.
# - `d_sjr_pct_fixed_ref`는 같은 해 고정 저널 14개의 SJR 분포 안에서의 백분위입니다. 기준집단이 고정이라 해마다 비교할 수 있습니다.
# - 스크립트는 가짜 코퍼스로만 실행해 보았습니다. 오류 없이 돌았지만 실제 자료로는 아직 실행하지 않았습니다.

# 설정은 제안값이고 결과를 보기 전에 정해 둔 것입니다. 연구자 확정이 필요합니다.
# - 코퍼스 n ÷ Scopus Documents의 표시 임계값: 0.80 미만 또는 1.25 초과
# - 비교에서 제외할 문서 수: 30편 미만
# - Spearman 참고선: 0.80

# ---

# # 2. 교정한 진행 현황 (2026-10-13)

# ## 완료
# | 단계 | 상태 | 핵심 |
# |---|---|---|
# | S0 | 완료·동결 | F1X 111,416편, 고정 저널 14개, 주 시나리오 NONE, 보정 PREREG `f16e76b4`(커밋 `90416c0`) |
# | S1 | 완료·동결 | 창 2007, 전수 수집 81,485편, 수작업 200/200, 원문 대조 41/41 |
# | S1 복구 | 동결 | 규칙 해시 `d4a87d357217`, 복구율 96.2%, 태그 `S1-final-2026-10-13`(커밋 `b0a658d`) |
# | 연도별 UNK | 기술 | 복구 후 0.7%에서 0.4% 안팎으로 평평. **2024년 0.9%, 2025년 1.9%는 높게 남음** |

# ## 새로 추가된 것: Scimago (S1 보강, 미실행)
# - 수집일은 2026-10-12이고 S5 이후 결과는 아직 보지 않은 상태입니다.
# - **S1.9의 JIF 수집 항목이 대체로 해결됩니다.** 고정 저널 14개는 2007~2025년 SJR이 빠짐없이 채워져 있습니다. 다만 Clarivate JIF가 아니라 Scimago 지표(2년 Cites/Doc는 JIF와 같은 계산이라고 Scimago가 설명)이므로 CHANGELOG에 대체 사실을 적어야 합니다.
# - **S1.5와 S1.6이 외부 출처로 보강됩니다.** 새로 생기는 외부 출처는 두 가지입니다.
#   - 코퍼스 논문 수와 Scopus Documents의 비
#   - 국가 층의 다국적 공저 비율과 International Collaboration의 대조. 이것은 Lens 생략의 부분 대체일 뿐 같은 수준의 검증은 아닙니다.
# - **pending 사건 두 건에 보조 증거가 생겼습니다.** 사건 목록은 동결 파일이라 수정하려면 CHANGELOG가 필요합니다.
#   - AMAR는 Scopus 색인 시작이 2014로 Crossref(2014)와 일치합니다.
#   - TR-E는 Scopus coverage 필드가 1997부터라 코퍼스 진입(1997)과 일치합니다. 이것은 코퍼스 진입이 색인 시작과 겹친다는 설명이지, 창간연도 1972를 확인해 주는 것이 아닙니다.
# - 예전에 대화에서 본 6칸의 예비 대조에서 TranspPol 2008은 32편 대 40편(0.80)이었습니다. S1 audit에서 0.800이던 바로 그 칸이 독립 증거로도 얇게 나왔습니다.

# ## 맥락으로 새로 보이는 것 (Scopus 문서 수, 34개 저널 안에서)
# - TRR의 문서 비중은 2007년 52%에서 2025년 11%로 줄었습니다.
# - 고정 저널 14개의 비중은 97%에서 64%로 줄었습니다. 저널 수는 17개에서 34개로 늘었습니다.
# - 고정 저널 SJR 중앙값은 1.11에서 2.05로 올랐습니다. 이는 S4.5의 구성 효과 해석에 직접 필요한 맥락입니다.
# - Overton 합계가 2025년에 57로 떨어지는 것은 정책 인용의 지연 때문일 가능성이 큽니다. 원인은 확인하지 못했으니 S8에서 최근 연도 해석에 주의하세요.

# ## 남은 것
# | 항목 | 성격 |
# |---|---|
# | push, `unk_share_by_year.csv` 커밋 | 즉시 |
# | **`s1_scimago_merge.py` 실행** | S2 전에 권장(OpenAlex 호출 없음) |
# | 복구판 논문 단위 국가 몫 | S2 첫 작업 |
# | `nauthors_check` | S2.1 전 필요 |
# | 창간연도 확인, 2024~2025 잔여 미상 표본 읽기 | 병행 |

# ## 누적 한계
# 기존 5개에 아래를 더합니다.
# - Scimago 지표는 단일 출처(Scopus)이며 Clarivate JIF가 아닙니다. Female Percent와 International Collaboration은 Scimago의 추정·집계 방식이 공개돼 있지 않거나 확인되지 않았습니다.
# - 이 변수들은 계획서에 없던 사후 추가입니다.

# ---

# # 3. S2 분석 방향 (재제시)

# ## 3.0 원칙: Scimago를 쓰는 선을 먼저 긋기
# | 구분 | 변수 | 허용 | 금지 |
# |---|---|---|---|
# | 검증 | Documents, International Collaboration | S1 보강, S2 부호표의 외부 점검 | 결과 변수나 공변량으로 투입 |
# | 맥락 | SJR, Cites/Doc, Quartile, Overton | 표, 히트맵 옆 병기, S4.5·S4.6 설명 | S4 인용 백분위와 섞기, 토픽 모형에 투입 |
# | 사용 안 함 | Est. APC, Est. value, 중복 열 | | |
# | 부록 한정 | Female Percent | 기술 통계만 | multiverse에 넣기 |

# SJR과 인용은 서로 같은 원천(인용)에서 나오고, International Collaboration은 S7의 지역 결과와 가까운 개념이라 순환하기 때문입니다.

# ## 3.1 실행 순서 (수정)
# | 단계 | 내용 | 변경점 |
# |---|---|---|
# | **0. 선행** | push, `s1_scimago_merge.py`(먼저 `COMMIT=False`), `nauthors_check` | **새 단계** |
# | **A. 기반** | 핵심 목록 고정(3.2) → 변수 사전 → 정리 규칙 → 복구판 국가 몫 | 정리 규칙에 Scimago 점검 추가 |
# | **B. 가중·결측** | 토픽 결측 → 가중치 4종 → 위험군 재현율 | 결측 분해에 Scopus 분모 추가 |
# | **C. 저자 지표** | 포트폴리오 → 잡음 대응 | 변경 없음 |
# | **D. 민감도** | 제목 분류기 → K 민감도 | 변경 없음 |
# | **E. 판정** | 부호표, 불변식, 음성 대조 | 부호표에 구성 효과 병기 |

# ## 3.2 핵심 목록 (제안)
# 이 목록은 결과를 보기 전에 PREREG 부록과 CHANGELOG에 적어야 합니다. 항목은 연구자가 정합니다.

# | 후보 | 출처 | 비고 |
# |---|---|---|
# | 주제 유효수·HHI의 시간 추세 부호 | S5.1(a) | |
# | 연속 노출 Spearman(Δexposure, d_all − d_fixed) 부호 | S5 주 분석 | |
# | 갱신 4항 분해의 진입·재배분 몫 방향 | S6 | |
# | 상위 5개국 몫, HHI 추세 부호 | S7 | 19개 연도만 사용 |
# | 신규 저자 지역 구성 변화 방향 | S7.2 | |

# ## 3.3 단계별 변경점 (원안에서 바뀐 부분만)

# **A-2. 정리 규칙(2.8)에 추가하는 점검**
# - 정리 규칙으로 제거한 건수를 저널·연도별로 만들고, Scimago `d_noncitable_share_3y`와 나란히 놓습니다. 이 값은 3년 창이고 정의가 달라서 목표 값이 아니라 **그럴듯한지 점검하는 참고**일 뿐입니다.
# - 비연구 문서 비중이 높게 나오는 저널은 ITS Mag(0.26), CommTR(0.17), VT Mag(0.15), TranspRev(0.10)입니다. VT Mag 칼럼 규칙이 이 저널에서 걸리는 것이 합리적인지 확인하는 정도로 쓰세요.

# **B-1. 토픽 결측 21%(2.6)에 추가하는 분해**
# - 지금까지는 "토픽 결측"만 있었습니다. 이제 저널×연도 수준에서 **논문 자체 누락**(코퍼스 대비 Scopus 비)과 **토픽만 결측**(has_topic=0)을 분리해서 볼 수 있습니다.
# - 비가 낮은 칸은 두 가지 결측이 겹친 칸입니다. 성향 모형이 이 칸을 보정하는지 먼저 확인하세요.
# - 성향 모형의 기본 공변량은 원안 그대로 두고, `d_log_documents`와 `d_sjr_pct_fixed_ref`를 넣은 모형은 **사전등록한 민감도 변형 1종**으로만 추가합니다. SJR은 토픽 결과 변수와 직접 순환하지는 않지만, 3.0의 선을 지키기 위해 주 모형에는 넣지 않습니다.

# **E. 부호표에 추가하는 열**
# - 부호표에 `구성 효과 병기` 열을 추가합니다. 고정 저널 14개의 비중이 97%에서 64%로 줄었으므로(Scopus 문서 수 기준), "전체 34개"와 "고정 14개"에서 부호가 달라지면 구성 효과로 우선 해석됩니다. 이는 원안의 S4.5와 S5.3이 이미 요구하던 구분을 외부 수치로 뒷받침합니다.

# **S5 준비: E2 민감도**
# - E2는 코퍼스의 연간 비중이 처음 2%를 넘는 해입니다. Scimago `d_doc_share_year_all`로 같은 정의를 한 번 더 계산할 수 있습니다. 단 분모가 그 해 Data에 있는 저널이라 구성이 바뀌므로 **민감도로만** 쓰고, EVENTS_FROZEN을 바꾸지는 마세요.

# ## 3.4 통과 판정 (원안 유지, 보강 점검 추가)
# | 검사 | 기준 | 실패 시 |
# |---|---|---|
# | 부호표 | 핵심 목록이 가중치 4종 모두 같은 부호 | 정의 민감 결과로 보고 |
# | 불변식·음성 대조·K 민감도 | 원안 그대로 | 원안 그대로 |
# | **(추가)** Scopus 대조 | 창 안 고정 저널 칸 중 표시된 칸의 비율을 보고 | 판정 기준이 아니라 부록 표. 표시된 칸은 분리 보고 |

# ## 3.5 주의
# - 이 파일은 Scopus와 OpenAlex 사이의 **집계 차이**를 보여 줍니다. 비가 1에서 벗어나는 이유는 문서 유형 정의, 정리 규칙, 수집 시점 등 여러 가지라서 원인을 단정하지 마세요.
# - `intl_collab_pct`와 우리 값은 **수준이 다를 수 있습니다**. 소속 국가를 모르는 저자가 있으면 우리 쪽이 낮게 나옵니다. 그래서 스크립트의 주 지표는 수준 차이가 아니라 **순위상관**이고, 평균 차이는 보조로 출력됩니다.
# - 영토 처리(HK 등을 모국에 합칠지)에 따라 값이 달라지므로 스크립트가 두 방식을 모두 출력합니다. 어느 쪽을 주로 쓸지는 TERRITORY_RULES(영토는 별도 국가)에 맞춰 **별도 처리를 주로** 보세요.
# - 2007~2025 국가 층(19개 연도)과 2000~2025 주제 층의 창 차이는 그대로입니다.
# - F2, F4, F7과 임베딩·토픽 파일의 구조는 여전히 보지 못했습니다. A 단계 코드는 그 열 목록을 받은 뒤에 맞출 수 있습니다.

# ## 3.6 다음 작업
# 1. `scimago` 폴더를 `~/Downloads/scimago/`에 두고 `s1_scimago_merge.py`를 실행하세요(`COMMIT=False`). 출력의 표시 칸 분포와 풀링 Spearman을 붙여 주시면 판독하겠습니다.
# 2. 임계값(0.80/1.25, 문서 30편, 참고선 0.80)이 마음에 들지 않으면 **실행 전에** 바꾸세요. 실행 후에 바꾸면 사후 변경입니다.
# 3. 판독 후에 핵심 목록 확정과 변수 사전 초안, A 단계 코드를 진행하겠습니다. F2, F4, F7의 열 목록(이름, dtype, 결측률)과 토픽·임베딩 파일 위치를 같이 붙여 주세요.

In [2]:
# -*- coding: utf-8 -*-
# S1 Scimago 후속 진단 (읽기 전용 + 요약 CSV 1개, OpenAlex 호출 없음)
# 목적: 표시된 칸(low/high)이 어디에 몰렸는지, 연도 합계 이탈(2013, 2020~2021, 2025)의 원인 저널, 다국적 공저 비율의 저널별 체계적 차이를 기술한다.
# 이 스크립트는 코퍼스·창·임계값을 바꾸지 않는다(기술 통계). 결과를 본 뒤 규칙을 바꾸면 CHANGELOG에 사후 변경으로 기록해야 한다.
import warnings
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings("ignore"); pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40); pd.set_option("display.max_rows", 200)
ROOT = Path.home() / "Research/Minnesota_Transport"; SC = ROOT / "paper1/01_S1/scimago"
MIN_DOCS = 30
hdr = lambda t: print("\n" + "=" * 90 + f"\n{t}\n" + "=" * 90)
C = pd.read_csv(SC / "coverage_vs_scopus.csv"); I = pd.read_csv(SC / "intl_collab_check.csv")
C["has_scopus"] = C.documents > 0; G = C[C.has_scopus].copy(); G["size"] = np.where(G.documents >= MIN_DOCS, "문서>=30", "문서<30")

hdr("[1] 표시 칸의 규모별·고정 여부별 분포")
t = G.groupby(["fixed", "size"]).flag.value_counts().unstack(fill_value=0); t["칸"] = t.sum(axis=1); t["표시율"] = ((t.get("low", 0) + t.get("high", 0)) / t["칸"]).round(3); print(t)

hdr("[2] 고정 저널 중 표시 칸(전체)")
fx = G[G.fixed & G.flag.isin(["low", "high"])].sort_values(["venue_short", "year"])
print(fx[["venue_short", "year", "n_corpus", "documents", "ratio", "flag"]].round(3).to_string(index=False))

hdr("[3] 저널별 비 요약(창 안): 중앙값이 0.9 미만 또는 1.1 초과인 저널")
S = G.groupby("venue_short").agg(fixed=("fixed", "first"), cells=("year", "size"), median_ratio=("ratio", "median"), n_corpus=("n_corpus", "sum"), scopus=("documents", "sum"),
                                 n_low=("flag", lambda s: (s == "low").sum()), n_high=("flag", lambda s: (s == "high").sum())).reset_index()
S["pooled_ratio"] = S.n_corpus / S.scopus; print(S[(S.median_ratio < 0.9) | (S.median_ratio > 1.1)].round(3).to_string(index=False))

hdr("[4] 연도 합계가 벗어난 해(2013, 2020, 2021, 2025)의 주요 기여 저널(코퍼스 − Scopus)")
for y in [2013, 2020, 2021, 2025]:
    d = G[G.year == y].assign(gap=lambda t: t.n_corpus - t.documents); d = d.loc[d.gap.abs().sort_values(ascending=False).index].head(5)
    print(f"[{y}] 합계 차 {int(G[G.year == y].n_corpus.sum() - G[G.year == y].documents.sum()):+d} | 상위:", [(r.venue_short, int(r.gap)) for r in d.itertuples()])

hdr("[5] 대형 저널 시계열 비: T-ITS, T-IV, TRR")
print(G[G.venue_short.isin(["T-ITS", "T-IV", "TRR"])].pivot(index="year", columns="venue_short", values="ratio").round(2).to_string())

hdr("[6] 고정 저널만 본 연도별 비 (표시 칸 영향 확인)")
fy = G[G.fixed].groupby("year").apply(lambda t: pd.Series(dict(n_corpus=t.n_corpus.sum(), scopus=t.documents.sum(), ratio=t.n_corpus.sum() / t.documents.sum()))); print(fy.round(3).T.to_string())

hdr("[7] 다국적 공저 비율: 저널별 체계적 차이")
U = I[I.usable].copy(); J = U.groupby("venue_short").agg(cells=("year", "size"), median_diff=("diff", "median"), median_abs=("diff", lambda s: s.abs().median())).reset_index()
J["rho_within"] = [U[U.venue_short == v][["ours_multi_pct", "intl_collab_pct"]].corr(method="spearman").iloc[0, 1] if (U.venue_short == v).sum() >= 8 else np.nan for v in J.venue_short]
print(J.reindex(J.median_diff.abs().sort_values(ascending=False).index).head(8).round(2).to_string(index=False))
big = J[J.median_diff.abs() > 10].venue_short.tolist(); V = U[~U.venue_short.isin(big)]
print(f"|중앙 차| 10%p 초과 저널 {big} 제외 시 풀링 Spearman {V[['ours_multi_pct', 'intl_collab_pct']].corr(method='spearman').iloc[0, 1]:.3f} (n={len(V)}) | 전체 {U[['ours_multi_pct', 'intl_collab_pct']].corr(method='spearman').iloc[0, 1]:.3f}  ※ 기술용, 판정 아님")
print("T-ITS 2008 행:"); print(I[(I.venue_short == "T-ITS") & (I.year.between(2006, 2010))][["year", "n_known", "documents", "ours_multi_pct", "intl_collab_pct"]].round(1).to_string(index=False))

out = S.merge(J, on="venue_short", how="left", suffixes=("", "_intl")); out.to_csv(SC / "coverage_followup.csv", index=False, encoding="utf-8-sig"); print("\n저장: coverage_followup.csv")


[1] 표시 칸의 규모별·고정 여부별 분포
flag          high  low   ok    칸    표시율
fixed size                              
False 문서<30     13   13   48   74  0.351
      문서>=30    22   28  107  157  0.318
True  문서<30      1    0    0    1  1.000
      문서>=30    31   23  211  265  0.204

[2] 고정 저널 중 표시 칸(전체)
venue_short  year  n_corpus  documents  ratio flag
        AAP  2007       192      150.0  1.280 high
        AAP  2008       181      253.0  0.715  low
        AAP  2009       215      161.0  1.335 high
        AAP  2013       404      511.0  0.791  low
      CACIE  2007        35       50.0  0.700  low
      CACIE  2009        58       46.0  1.261 high
      CACIE  2011        35       48.0  0.729  low
      CACIE  2016        48       63.0  0.762  low
      CACIE  2021       137      107.0  1.280 high
      CACIE  2025       247      327.0  0.755  low
        JTG  2009        83       56.0  1.482 high
        JTG  2011       109      165.0  0.661  low
      T-ITS  2009        89       70.0  1.27

In [3]:
#

In [5]:
# `schema` 출력을 반영해 스크립트를 수정하겠습니다. `F2.w`가 이미 `1/n` 값처럼 보여서, `prereg`를 동결하기 전에 가중치 정의를 확인할 점검 단계도 추가합니다.`s2_measurement.py`를 `schema` 출력에 맞게 수정했습니다. 문법 검사와 가짜 자료 시험(`f2_check`, `risk`)만 통과했고, 실제 자료로는 아직 돌려 보지 못했습니다.

# ## schema 출력에서 읽은 것
# - **토픽:** `F1.topic150`(결측 21.06%)이 유일한 토픽 열입니다. 설정을 `TOPIC_FILE="F1_papers.csv"`, `TOPIC_COL="topic150"`으로 고정했습니다. `has_topic`이 이미 있어서, `f2_check`에서 두 열이 일치하는지 확인합니다.
# - **위험군:** F7에는 위험군 플래그 열이 없고 `F2_risk_names.csv`에 `ambiguous`, `heavy` 열이 있습니다. `risk` 단계를 이 구조로 다시 썼습니다.
#   - 이름당 OpenAlex ID 2개 이상을 근사 정답으로 둡니다.
#   - 플래그는 `ambiguous`, `heavy`, 둘의 합집합으로 나눠 precision과 recall을 냅니다.
#   - F7에 행이 1개뿐인 이름은 ID 2개가 불가능하므로, 행 2개 이상 이름만 따로 평가한 결과도 병기합니다.
#   - 위험군 변형 7종별 분해는 이 파일들에 변형 정의 열이 없어 하지 않았습니다.
# - **F7의 연도 범위:** F7(141,836행)은 F2(282,040행)보다 작아서 일부 authorship만 담은 것으로 보입니다. `risk`는 F7의 연도 분포를 출력하니 평가 범위로 보고하세요.

# ## prereg 전에 확인할 것
# `F2.w`의 예시가 0.1667로, `n_authors=6`의 1/n과 같습니다. 계획서의 `equal(현 F2 방식)`이 이미 1/n이고, `inv_n_authors`는 위험군으로 빠진 질량을 손실로 두는 `1/(원래 n_authors)`일 가능성이 있습니다. 이 해석이 맞으면 제가 `prereg`에 적은 가중치 정의("equal = authorship당 1")는 계획서와 다릅니다. 지금은 동결 전이라 비용 없이 고칠 수 있으니 `["f2_check"]`를 먼저 실행하세요. 점검 항목은 다음과 같습니다.
# - F2 가중치가 1/n_authors와 일치하는 행의 비율
# - 논문별 가중치 합이 1인 논문의 비율
# - 저자 행 보존율(EDA의 98% → 86%와 대조)
# - F2의 토픽이 F1과 같은지
# - `in_sample`이 False인 논문이 2007년 이후에 있는지(S1 국가 층은 이 열로 거르지 않았습니다)
# - `fixed_venue`가 PREREG의 고정 저널 14개와 같은지

# 출력을 붙여 주시면 `author_w`의 정의와 `prereg`의 weights 문구를 그에 맞춰 고친 뒤 동결하세요. 이 점검이 끝나기 전에 `prereg`를 실행하지 마세요.

# ## 이어지는 순서
# 1. `["f2_check"]`: 출력을 붙여 주세요.
# 2. 정의를 확정한 뒤 `["prereg"]`를 실행합니다.
# 3. `["clean_flags","country_rec","weights_country","unk_resid"]`를 실행합니다.
# 4. `["missing"]`과 `["risk"]`를 실행합니다.

# ## 참고
# - `F4`는 `name_norm` 기준이라 저자 식별이 이름 기반입니다. 동명이인 영향은 `risk` 결과를 범위로 보고하는 설계가 맞습니다.
# - `F4`에는 겹치는 기간 열(`m_P10_14`/`m_P12_14`, `m_P20_24`/`m_P22_24`)이 있습니다. 포트폴리오 단계(2.4)를 만들 때 어느 쪽이 주 기간인지 정해 주세요.
# - `F5`의 `main_venue0`는 계획서대로 폐기 대상이고, F3(저자×기간×저널×토픽)이 포트폴리오 지표의 원료입니다.

# `f2_check` 출력을 붙여 주시면 다음 판독을 같이 하겠습니다.

In [7]:
#

In [12]:
# -*- coding: utf-8 -*-
# =====================================================================================
# 논문 1 / S2 측정 교정 (OpenAlex 호출 없음, 크레딧 소모 없음)
# 실행 순서 (STEPS만 바꿔 여러 번 실행)  [v2: schema 결과 반영, f2_check 추가]
#   0) ["f2_check"]                    F2 가중치 정의·F1 열(in_sample, fixed_venue) 점검. prereg 전에 실행, 출력 붙여 주세요
#   1) ["schema"]                       F1~F7·토픽 파일의 열 구조와 변수 사전 골격 생성 -> 출력을 붙여 주세요
#   2) ["prereg"]                       S2 사전등록 부록(핵심 목록, 정의, 임계값)을 동결. CORE_CONFIRMED=True 필요
#   3) ["clean_flags","country_rec","weights_country","unk_resid"]   지금 가진 자료만으로 실행 가능
#   4) ["missing"]                      토픽 파일이 확인된 뒤 (TOPIC_FILE/TOPIC_COL 지정 또는 자동 탐색)
#   5) ["risk","risk_selection"]        위험군 플래그 재현율 / 위험군 제외의 국가 선택 효과
#   6) ["commit"]                       산출물 해시 기록 + 커밋 + 태그
# 이 파일이 하지 않는 것: 포트폴리오 지표(2.4), 잡음 대응(2.5), 제목 분류기(2.7), K 민감도(2.9), 저자 단위 가중 결과.
#   F2/F4와 임베딩·토픽 구조를 본 뒤 다음 단계로 만듭니다.
# 모든 임계값은 prereg 단계에서 결과를 보기 전에 동결됩니다. 실행 후 바꾸면 사후 변경이므로 CHANGELOG_S2에 기록하세요.
# =====================================================================================
import re, json, hashlib, subprocess, warnings
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40); pd.set_option("display.max_rows", 120)

# ---------------- 설정 (여기만 수정) ----------------
STEPS = ["f2_check"]
ROOT = Path.home() / "Research/Minnesota_Transport"
CORE_CONFIRMED = False       # 핵심 목록(아래 CORE_LIST)을 연구자가 확인·수정했으면 True
WINDOW = 2007                # 국가 층 창(S1 확정)
TOPIC_START = 2000           # 주제 분석 본분석 시작
SEED = 2026
TOPIC_FILE = "F1_papers.csv"  # schema 결과: 토픽은 F1의 topic150(결측 21%)
TOPIC_COL = "topic150"
TOPIC_PID_COL = "pid"
TOPIC_MISSING_VALUES = [-1]  # 토픽 없음으로 볼 값(NaN은 항상 결측). schema 출력의 값 분포를 보고 확정
F7_FILE = "F7_oa_author_ids.csv"; RISK_FILE = "F2_risk_names.csv"   # schema 결과
COMMIT = False               # commit 단계에서만 사용
TAG_NAME = None

CORE_LIST = [  # 결과를 보기 전에 연구자가 확정할 것(3.2 제안안)
    dict(id="C1", stat="주제 유효수·HHI 시간 추세 부호", source="S5.1(a)", status="주제 층 필요: 이 파일에서는 계산 안 함"),
    dict(id="C2", stat="연속 노출 Spearman(Δexposure, d_all-d_fixed) 부호", source="S5 주 분석", status="이 파일에서는 계산 안 함"),
    dict(id="C3", stat="갱신 4항 분해의 진입·재배분 몫 방향", source="S6", status="저자 층 필요: 이 파일에서는 계산 안 함"),
    dict(id="C4", stat="상위 5개국 몫(top5_incl, top5_known)과 HHI(hhi_known)의 연도 추세 부호", source="S7", status="weights_country 단계에서 계산(19개 연도)"),
    dict(id="C5", stat="신규 저자 지역 구성 변화 방향", source="S7.2", status="저자 층 필요: 이 파일에서는 계산 안 함"),
]
# ----------------------------------------------------

S0, S1, S2 = ROOT / "paper1/00_S0", ROOT / "paper1/01_S1", ROOT / "paper1/02_S2"
DATA = ROOT / "analysis_data/final"; SC = S1 / "scimago"
hdr = lambda t: print("\n" + "=" * 90 + f"\n{t}\n" + "=" * 90)
now = lambda: pd.Timestamp.now().isoformat(timespec="seconds")
sha_b = lambda b: hashlib.sha256(b).hexdigest()
sha_f = lambda p: sha_b(Path(p).read_bytes())
class Halt(Exception): pass

# ---------------- 공통 ----------------
def nd(s):
    if not isinstance(s, str) or not s.strip(): return None
    s = s.strip().lower(); s = re.sub(r"^(https?://(dx\.)?doi\.org/|doi:\s*)", "", s); return s or None

def load_any(p, nrows=None):
    p = Path(p)
    if p.suffix == ".parquet": return pd.read_parquet(p)
    return pd.read_csv(p, low_memory=False, nrows=nrows)

def read_tbl(name):
    p = S1 / f"{name}.parquet"
    return pd.read_parquet(p) if p.exists() else pd.read_csv(S1 / f"{name}.csv.gz")

def changelog(line):
    cl = S2 / "CHANGELOG_S2.md"; S2.mkdir(parents=True, exist_ok=True)
    if not cl.exists(): cl.write_text("# CHANGELOG S2 (날짜 | 변경 | 사유 | 영향)\n\n", encoding="utf-8")
    if line in cl.read_text(encoding="utf-8"): return
    with cl.open("a", encoding="utf-8") as f: f.write(line + "\n")

def git(*a, check=False):
    r = subprocess.run(["git", "-C", str(ROOT), *a], capture_output=True, text=True)
    if check and r.returncode: raise Halt(f"git {' '.join(a)} 실패: {r.stderr.strip()}")
    return r

def load_F1X():
    F1 = pd.read_csv(DATA / "F1_papers.csv", low_memory=False); F1["pid"] = F1.pid.astype(str)
    exf = S0 / "EXCLUDE_PIDS_FROZEN.csv"; log = pd.read_csv(S0 / "FREEZE_LOG.csv"); last = log[log.file == exf.name].iloc[-1]
    if sha_f(exf) != last.sha256: raise Halt("EXCLUDE_PIDS_FROZEN.csv 해시가 동결 기록과 다릅니다")
    EXC = set(pd.read_csv(exf, dtype=str, encoding="utf-8-sig").pid)
    return F1[~F1.pid.isin(EXC)].copy()

def spec_hash(spec): return sha_b(json.dumps({k: v for k, v in spec.items() if k not in ("sha256", "created")}, sort_keys=True, ensure_ascii=False).encode())

# ============================================================
# S2 사전등록 부록 (모든 임계값과 정의의 단일 출처)
# ============================================================
def make_spec():
    return dict(version=1, created=now(), seed=SEED, window=WINDOW, topic_start=TOPIC_START, core_list=CORE_LIST,
        weights=dict(
            equal="F2.w 방식: 위험군 이름을 뺀 뒤 남은 저자 수로 1/n_retained(논문 질량 1). f2_check로 F2 논문 100%가 이 정의임을 확인",
            inv_n_authors="1/n_authors(F1의 원래 저자 수). 위험군으로 빠진 저자의 질량은 '손실 질량'으로 보고",
            first_last="원래 저자 순서·수 기준: 단독 1.0 / 2인 0.5씩 / 3인 이상 제1·마지막 각 0.3, 중간 0.4/(n-2). 위험군 제외 시 남은 저자 몫을 논문 질량 1로 재정규화(가정)",
            full_count="저자-논문 쌍마다 1(논문 질량 = 남은 저자 수). 국가 층에서는 소속 국가마다 1(분할 없음)",
            country_layer_note="국가 층은 저자 신원이 필요 없어 위험군 제외를 적용하지 않음 -> 국가 층에서 equal = inv_n_authors(동일 값). 구분되는 가중치는 3종이며 equal과 inv_n의 차이는 저자 층에서만 생김",
            fixed_venues_note="분석의 고정 저널은 PREREG.json event.fixed_venues(14개)를 쓴다. F1.fixed_venue 열은 JPT에서 PREREG와 달라 사용하지 않는다"),
        country_variants=dict(main="OpenAlex 국가(S1 주 분석)", rec="국가 미상을 동결된 복구 규칙(RECOVERY d4a87d357217)으로 보충(민감도)",
                              terr_merge="HK·MO를 CN과 합산한 민감도(주 분석은 영토 별도)"),
        clean_rules=dict(bib_section=r"bibliographic section", vt_mobile_radio=dict(venue="VT Mag", regex=r"\[\s*mobile radio\s*\]"),
                         repeat_title_min=5, dup_title="같은 저널·연도·정규화 제목 중복: 첫 건 제외 나머지 표시", topic_excluded=[18],
                         note="표시만 하고 제거는 하지 않음. 제거 전후 비교는 별도 단계"),
        propensity=dict(clip_pct=[1, 99], positivity_floor=0.05, model="logistic(main effects, C=10)", cv_folds=5,
                        outcome="has_topic = 토픽 열이 결측이 아니고 TOPIC_MISSING_VALUES에 없음",
                        covariates_A="year, venue_short, n_authors 구간, is_solo (2000~2025)",
                        covariates_B="A + 논문 주 국가군(상위 8 + other + UNK) (2007~2025)"),
        portfolio=dict(shrink_k=5, min_topic_papers=3),
        stats=dict(top5="연도별 상위 5개국 몫 합(UNK 제외 국가 중). top5_incl=전체 질량 분모, top5_known=국가 아는 질량 분모",
                   hhi_known="국가 아는 질량 기준 국가 몫 제곱합", trend="연도와의 Spearman 부호(보조: OLS 기울기)"),
        pass_rule="핵심 통계의 부호가 가중치 4종에서 모두 같음(변형·표본·영토 조합마다)")

def check_prereg():
    p = S2 / "S2_PREREG_ADDENDUM.json"
    if not p.exists(): raise Halt("먼저 prereg 단계를 실행해 S2 부록을 동결하세요")
    spec = json.loads(p.read_text(encoding="utf-8"))
    if spec_hash(spec) != spec["sha256"]: raise Halt("S2_PREREG_ADDENDUM.json이 동결 후 변경되었습니다")
    return spec

def st_prereg():
    hdr("[prereg] S2 사전등록 부록 동결")
    S2.mkdir(parents=True, exist_ok=True); p = S2 / "S2_PREREG_ADDENDUM.json"
    spec = make_spec(); spec["sha256"] = spec_hash(spec)
    if p.exists():
        old = json.loads(p.read_text(encoding="utf-8"))
        if old["sha256"] != spec["sha256"]: raise Halt(f"이미 동결된 부록({old['sha256'][:12]})과 현재 설정({spec['sha256'][:12]})이 다릅니다. 사후 변경이면 새 파일로 분리하고 CHANGELOG_S2에 사유를 쓰세요")
        print("이미 동결됨, 동일:", old["sha256"][:12]); return
    if not CORE_CONFIRMED: raise Halt("CORE_LIST를 확인·수정한 뒤 CORE_CONFIRMED=True로 바꾸세요(핵심 목록은 결과를 보기 전에 확정해야 함)")
    try:
        pr = json.loads((S0 / "PREREG.json").read_text(encoding="utf-8")); print("S0 PREREG의 weights 항목:", pr.get("weights"), "| 위 weights 정의와 대조하세요")
    except Exception: pass
    p.write_text(json.dumps(spec, ensure_ascii=False, indent=1), encoding="utf-8")
    changelog(f"- {now()} | S2 사전등록 부록 동결(sha256 {spec['sha256'][:12]}): 핵심 목록, 가중치 4종 정의, 정리 규칙, 성향 모형 설정, 영토 합산 민감도 | S2 시작 전, 결과 열람 전 | 이후 모든 S2 단계가 이 해시를 확인함")
    print("동결:", p.name, spec["sha256"][:12])

# ============================================================
# [schema] 열 구조와 변수 사전 골격
# ============================================================
def st_schema():
    hdr("[schema] 파일 구조 점검")
    S2.mkdir(parents=True, exist_ok=True); rows = []
    files = sorted(p for p in DATA.iterdir() if p.is_file() and (p.suffix in (".csv", ".parquet") or p.name.endswith(".csv.gz")))
    print("DATA 폴더 파일:", [p.name for p in files])
    for p in files:
        try:
            big = p.stat().st_size > 300e6; df = load_any(p, nrows=500000 if big else None)
        except Exception as e: print(f"  {p.name}: 읽기 실패 {e}"); continue
        print(f"\n[{p.name}] {len(df):,}행{' (대용량: 앞 50만 행만)' if big else ''}, {df.shape[1]}열")
        for c in df.columns:
            s = df[c]; ex = s.dropna().iloc[0] if s.notna().any() else ""
            rows.append(dict(file=p.name, column=c, dtype=str(s.dtype), null_rate=round(float(s.isna().mean()), 4), n_unique=int(s.nunique()) if s.dtype == object or s.nunique() < 1000 else -1,
                             example=str(ex)[:60], definition="", base_sample="", derived_from="", topic_papers_only=""))
        sm = pd.DataFrame(rows[-df.shape[1]:])[["column", "dtype", "null_rate", "example"]]; print(sm.head(60).to_string(index=False))
    pd.DataFrame(rows).to_csv(S2 / "variable_dictionary_skeleton.csv", index=False, encoding="utf-8-sig")
    print("\n저장: variable_dictionary_skeleton.csv (definition, base_sample, derived_from, topic_papers_only 열은 직접 채움)")
    others = [str(p.relative_to(ROOT)) for p in (ROOT / "analysis_data").rglob("*") if p.is_file() and re.search(r"topic|embed|specter|cluster", p.name, re.I)]
    print("토픽·임베딩 관련 파일 후보:", others[:30] or "없음")
    tf = find_topic(quiet=True)
    if tf: print(f"토픽 열 후보: {tf[0]} / {tf[1]}")

def find_topic(quiet=False):
    if TOPIC_FILE:
        p = DATA / TOPIC_FILE if (DATA / TOPIC_FILE).exists() else ROOT / TOPIC_FILE
        if not TOPIC_COL: raise Halt("TOPIC_FILE을 지정했으면 TOPIC_COL도 지정하세요")
        return p, TOPIC_COL
    cands = []
    for p in sorted(DATA.iterdir()):
        if not p.is_file(): continue
        try:
            if p.suffix == ".parquet":
                import pyarrow.parquet as pq; cols = pq.ParquetFile(p).schema.names
            elif p.suffix == ".csv" or p.name.endswith(".csv.gz"): cols = list(pd.read_csv(p, nrows=3).columns)
            else: continue
        except Exception: continue
        if TOPIC_PID_COL in cols: cands += [(p, c) for c in cols if re.search(r"topic", c, re.I)]
    if len(cands) == 1: return cands[0]
    if quiet: print("토픽 열 후보(여러 개 또는 없음):", [(p.name, c) for p, c in cands]); return None
    raise Halt(f"토픽 열을 하나로 정하지 못했습니다. 후보 {[(p.name, c) for p, c in cands]}. TOPIC_FILE/TOPIC_COL을 지정하세요")

def load_has_topic(F):
    p, col = find_topic(); T = load_any(p, None); T[TOPIC_PID_COL] = T[TOPIC_PID_COL].astype(str)
    print(f"토픽 파일 {p.name}, 열 {col}: 값 분포 상위", T[col].value_counts(dropna=False).head(8).to_dict())
    ok = T[col].notna() & ~T[col].isin(TOPIC_MISSING_VALUES)
    m = dict(zip(T.loc[ok, TOPIC_PID_COL], T.loc[ok, col]))
    F = F.copy(); F["topic"] = F.pid.map(m); F["has_topic"] = F.topic.notna().astype(int)
    print(f"F1X {len(F):,}편 중 토픽 보유 {F.has_topic.mean():.1%} (계획서 EDA: 결측 약 21%와 대조)")
    return F

# ============================================================
# [clean_flags] 정리 규칙 2.8 (표시만, 제거하지 않음)
# ============================================================
def normt(s): return re.sub(r"[^a-z0-9]+", " ", str(s).lower()).strip()

def st_clean_flags():
    hdr("[clean_flags] 비연구 항목 규칙 표시"); spec = check_prereg(); cr = spec["clean_rules"]
    F = load_F1X(); F["t"] = F.title.fillna("").map(normt)
    F["f_bib"] = F.title.fillna("").str.contains(cr["bib_section"], case=False, regex=True)
    F["f_vtmr"] = (F.venue_short == cr["vt_mobile_radio"]["venue"]) & F.title.fillna("").str.contains(cr["vt_mobile_radio"]["regex"], case=False, regex=True)
    cnt = F[F.t != ""].groupby(["venue_short", "t"]).pid.transform("size"); F["f_repeat"] = False; F.loc[cnt.index, "f_repeat"] = cnt >= cr["repeat_title_min"]
    F["f_dup"] = False; d = F[F.t != ""].duplicated(["venue_short", "year", "t"], keep="first"); F.loc[d[d].index, "f_dup"] = True
    flags = ["f_bib", "f_vtmr", "f_repeat", "f_dup"]; F["any_flag"] = F[flags].any(axis=1)
    try:
        tp, tc = find_topic(quiet=True) or (None, None)
        if tp is not None:
            G = load_has_topic(F); F["f_topic18"] = G.topic.isin(cr["topic_excluded"]).values; flags.append("f_topic18"); F["any_flag"] |= F.f_topic18
    except Exception as e: print("※ 주제 18번 규칙은 토픽 파일 확인 후 적용:", e)
    print("규칙별 표시 건수:", {f: int(F[f].sum()) for f in flags}, "| 하나라도:", int(F.any_flag.sum()), f"({F.any_flag.mean():.2%})")
    print("(참고: 계획서의 '중복 제목 459행'과 f_dup 건수를 대조. 다르면 정의가 다른 것이니 이유를 확인)")
    F[["pid", "year", "venue_short"] + flags + ["any_flag"]].to_csv(S2 / "clean_flags.csv", index=False)
    rng = np.random.default_rng(SEED); S = []
    for f in flags:
        g = F[F[f]]; S.append(g.sample(min(30, len(g)), random_state=SEED)[["pid", "venue_short", "year", "title"]].assign(rule=f, correct=""))
    pd.concat(S).to_csv(S2 / "clean_review_samples.csv", index=False, encoding="utf-8-sig"); print("규칙별 30건 표본 저장: clean_review_samples.csv (correct 열에 1/0 수작업 확인)")
    rp = F[F.f_repeat].groupby(["venue_short", "t"]).size().sort_values(ascending=False).head(30).reset_index(name="n"); rp.to_csv(S2 / "top_repeated_titles.csv", index=False, encoding="utf-8-sig")
    print("반복 제목 상위:\n", rp.head(10).to_string(index=False))
    vy = F.groupby(["venue_short", "year"]).agg(n=("pid", "size"), **{f: (f, "sum") for f in flags}, any_flag=("any_flag", "sum")).reset_index(); vy["any_share"] = vy.any_flag / vy.n
    sp = next(iter(SC.glob("**/scimago_journal_year.csv")), None)
    if sp is not None:
        S_ = pd.read_csv(sp)
        if {"venue_short", "year", "d_noncitable_share_3y"} <= set(S_.columns):
            vy = vy.merge(S_[["venue_short", "year", "d_noncitable_share_3y"]], on=["venue_short", "year"], how="left")
            x = vy.dropna(subset=["d_noncitable_share_3y"]); x = x[x.n >= 20]
            print(f"[참고] 표시 비중과 Scimago 비인용 문서 비중의 Spearman {x.any_share.corr(x.d_noncitable_share_3y, method='spearman'):.2f} (n={len(x)} 칸, 정의가 달라 목표가 아니라 그럴듯함 점검)")
    else: print("※ scimago_journal_year.csv를 찾지 못해 Scimago 병기 생략")
    vy.to_csv(S2 / "clean_summary_venue_year.csv", index=False, encoding="utf-8-sig")
    changelog(f"- {now()} | clean_flags 실행: 정리 규칙 표시만(제거 안 함), 건수 {{{', '.join(f'{f}:{int(F[f].sum())}' for f in flags)}}} | S2.8 | 제거 전후 비교는 별도")

# ============================================================
# 가중치 4종과 복구판 국가 질량
# ============================================================
SCHEMES = ["equal", "inv_n_authors", "first_last", "full_count"]

def author_w(A, scheme):
    n = A["n"].values.astype(float); r = A["rank"].values
    if scheme == "full_count": return np.ones(len(A))
    if scheme in ("equal", "inv_n_authors"): return 1.0 / n   # 국가 층에서는 두 정의가 같은 값(위험군 제외 없음)
    if scheme == "first_last":
        return np.where(n == 1, 1.0, np.where(n == 2, 0.5, np.where((r == 0) | (r == n - 1), 0.3, 0.4 / np.maximum(n - 2, 1))))
    raise ValueError(scheme)

def country_mass(A, col, scheme):
    """A: pid, rank, n, <col>(국가 '|' 구분, 빈 문자열=미상). 반환: pid, country, w"""
    w = author_w(A, scheme); cs = A[col].where(A[col] != "", "UNK").str.split("|"); k = cs.str.len().values
    B = pd.DataFrame({"pid": A.pid.values, "country": cs.values, "w": w if scheme == "full_count" else w / k})
    return B.explode("country").groupby(["pid", "country"], as_index=False).w.sum()

def extra_unk(D, scheme):
    n = D.n_authors.fillna(1).astype(float).values
    return pd.DataFrame({"pid": D.pid.values, "country": "UNK", "w": n if scheme == "full_count" else 1.0})

def build_authorship(F):
    A = read_tbl("authorship_level"); A["pid"] = A.pid.astype(str); A["countries"] = A.countries.fillna("")
    Rr = pd.read_csv(S1 / "authorship_level_recovered.csv.gz", dtype={"pid": str}); Rr["countries_rec"] = Rr.countries_rec.fillna("")
    A = A.merge(Rr[["pid", "rank", "countries_rec"]], on=["pid", "rank"], how="left"); A["countries_rec"] = A.countries_rec.fillna("")
    A["cs_main"] = A.countries; A["cs_rec"] = np.where((A.countries == "") & (A.countries_rec != ""), A.countries_rec, A.countries)
    A["n"] = A.groupby("pid")["rank"].transform("size")
    return A

def st_country_rec():
    hdr("[country_rec] 가중치(국가 층은 equal=inv_n이라 실질 3종) x (주, 복구) 국가 질량 + 불변식"); check_prereg()
    F = load_F1X(); D = F[F.year >= WINDOW]; A = build_authorship(F); ok = True; L = []
    ex = D[~D.pid.isin(A.pid)]; print(f"창 안 논문 {len(D):,} | authorship 있는 논문 {A.pid.nunique():,} | 없는 논문(국가 미상 처리) {len(ex):,}")
    for var, col in [("main", "cs_main"), ("rec", "cs_rec")]:
        for sc in SCHEMES:
            M = pd.concat([country_mass(A, col, sc), extra_unk(ex, sc)]); M["scheme"] = sc; M["variant"] = var; L.append(M)
            ps = M.groupby("pid").w.sum(); nn = A.groupby("pid").n.first().reindex(ps.index).fillna(F.set_index("pid").n_authors.reindex(ps.index))
            if sc in ("equal", "inv_n_authors", "first_last"): t = np.allclose(ps, 1.0)
            else: t = bool((ps.values >= nn.fillna(1).values - 1e-9).all())
            print(f"  불변식 질량 [{var}/{sc}] 논문당 합: {'PASS' if t else 'FAIL'}"); ok &= bool(t)
    L = pd.concat(L).merge(F[["pid", "year", "venue_short"]], on="pid")
    P = read_tbl("country_authorship"); P["pid"] = P.pid.astype(str)
    Mo = L[(L.variant == "main") & (L.scheme == "inv_n_authors")][["pid", "country", "w"]].merge(P[["pid", "country", "w"]], on=["pid", "country"], how="outer", suffixes=("_new", "_old")).fillna(0)
    dmax = float((Mo.w_new - Mo.w_old).abs().max()); t = dmax < 1e-9; ok &= t; print(f"  재현 점검: 재구축(main, inv_n)이 S1 country_authorship과 일치, 최대 차이 {dmax:.2e}: {'PASS' if t else 'FAIL'}")
    u = L[L.scheme == "inv_n_authors"].assign(unk=lambda d: d.country == "UNK").groupby(["variant", "year"]).apply(lambda d: d.w[d.unk].sum() / d.w.sum()).unstack(0).round(3)
    print("연도별 미상 질량 몫(inv_n, 논문 단위):\n", u.T.to_string())
    L.to_parquet(S2 / "country_mass_long.parquet", index=False) if _has_pa() else L.to_csv(S2 / "country_mass_long.csv.gz", index=False, compression="gzip")
    if not ok: raise Halt("불변식 FAIL: 이후 단계를 진행하지 마세요")
    changelog(f"- {now()} | country_rec: 4종 가중치 x (주, 복구) 국가 질량 생성, 불변식 PASS, S1 재현 최대 차이 {dmax:.1e} | S2.1, S7 민감도")

def _has_pa():
    try: import pyarrow; return True
    except Exception: return False

def load_mass():
    p = S2 / "country_mass_long.parquet"
    return pd.read_parquet(p) if p.exists() else pd.read_csv(S2 / "country_mass_long.csv.gz", dtype={"pid": str})

# ============================================================
# [weights_country] 핵심 C4 부호표: 4종 가중치 x 변형 x 표본 x 영토
# ============================================================
def yearly_stats(L, fixed, sample, terr):
    d = L if sample == "all" else L[L.venue_short.isin(fixed)]
    if terr: d = d.assign(country=d.country.replace({"HK": "CN", "MO": "CN"}))
    g = d.groupby(["year", "country"]).w.sum().unstack(fill_value=0); tot = g.sum(axis=1)
    kn = g.drop(columns=[c for c in ["UNK", "ZZ_former"] if c in g.columns]); kt = kn.sum(axis=1)
    top5 = kn.apply(lambda r: r.nlargest(5).sum(), axis=1)
    return pd.DataFrame(dict(top5_incl=top5 / tot, top5_known=top5 / kt, hhi_known=((kn.div(kt, axis=0)) ** 2).sum(axis=1),
                             cn_known=kn.get("CN", 0) / kt, us_known=kn.get("US", 0) / kt, unk=g.get("UNK", 0) / tot))

def st_weights_country():
    hdr("[weights_country] C4 부호표"); spec = check_prereg(); fixed = json.loads((S0 / "PREREG.json").read_text(encoding="utf-8"))["event"]["fixed_venues"]
    L = load_mass(); R = []; Y = []
    for (sc, var), d in L.groupby(["scheme", "variant"]):
        for sample in ["all", "fixed"]:
            for terr in [False, True]:
                S = yearly_stats(d, fixed, sample, terr); yrs = S.index.values.astype(float)
                for st in S.columns:
                    y = S[st]; R.append(dict(scheme=sc, variant=var, sample=sample, terr_merge=terr, stat=st, spearman=y.corr(pd.Series(y.index, index=y.index), method="spearman"), slope=np.polyfit(yrs, y.values, 1)[0], n_years=len(y)))
                Y.append(S.assign(scheme=sc, variant=var, sample=sample, terr_merge=terr).reset_index())
    R = pd.DataFrame(R); R["sign"] = np.sign(R.spearman).astype(int); pd.concat(Y).to_csv(S2 / "country_year_stats.csv", index=False)
    R.to_csv(S2 / "weights_signs_long.csv", index=False)
    key = ["variant", "sample", "terr_merge", "stat"]; A = R.groupby(key).agg(signs=("sign", lambda s: "".join("+" if v > 0 else "-" if v < 0 else "0" for v in s)), agree=("sign", lambda s: s.nunique() == 1),
                                                                         rho_min=("spearman", "min"), rho_max=("spearman", "max")).reset_index()
    A.to_csv(S2 / "weights_signs_summary.csv", index=False)
    core = A[A.stat.isin(["top5_incl", "top5_known", "hhi_known"])]
    print("핵심 C4(top5_incl, top5_known, hhi_known): 조합 중 4종 가중치 부호 일치 비율", f"{core.agree.mean():.2%}", f"({int(core.agree.sum())}/{len(core)})")
    print(core.pivot_table(index=["variant", "sample", "terr_merge"], columns="stat", values="signs", aggfunc="first").to_string())
    print("\n(scheme 순서는 equal, first_last, full_count, inv_n_authors의 알파벳 정렬 순서)")
    print("부호가 갈린 조합:\n", A[~A.agree][key + ["signs", "rho_min", "rho_max"]].round(2).to_string(index=False) if (~A.agree).any() else "없음")
    print("※ 연도 19개 Spearman이라 |rho|가 작으면 부호는 불안정합니다. rho_min/rho_max를 같이 보세요. 판정은 prereg의 pass_rule(부호 일치)이며 크기 기준은 없음")
    changelog(f"- {now()} | weights_country 실행: C4 부호 일치 {int(core.agree.sum())}/{len(core)} 조합 | S2.1 판정(핵심 목록 C4만)")

# ============================================================
# [unk_resid] 2024~2025 잔여 미상 표본
# ============================================================
def st_unk_resid():
    hdr("[unk_resid] 복구 후 잔여 미상(2024~2025) 표본"); check_prereg()
    F = load_F1X(); A = build_authorship(F).merge(F[["pid", "year", "venue_short"]], on="pid")
    U = A[(A.cs_rec == "") & (A.raw_aff.fillna("") != "") & A.year.isin([2024, 2025])]
    print(f"2024~2025 소속 문자열은 있으나 복구 후에도 미상: {len(U):,}건"); n = A[A.year.isin([2024, 2025])]; print(f"  전체 authorship 대비 {len(U) / len(n):.2%}")
    S = U.sample(min(30, len(U)), random_state=SEED)[["pid", "rank", "year", "venue_short", "raw_aff"]].assign(note=""); S.to_csv(S2 / "unk_resid_2024_25_sample.csv", index=False, encoding="utf-8-sig")
    print("저장: unk_resid_2024_25_sample.csv (30건을 읽고 note에 원인 유형 기입: 형식 문제, 기관명만, 빈 문자열 등)")
    V = A[(A.cs_rec == "") & A.year.isin([2024, 2025])]; print("소속 문자열 자체가 빈 비중(복구 후 미상 중):", f"{(V.raw_aff.fillna('') == '').mean():.1%}")

# ============================================================
# [missing] 토픽 결측 2.6
# ============================================================
def st_missing():
    hdr("[missing] 토픽 결측 분해 + 성향 가중"); check_prereg(); pp = check_prereg()["propensity"]
    from sklearn.linear_model import LogisticRegression
    from sklearn.model_selection import cross_val_predict, StratifiedKFold
    from sklearn.metrics import roc_auc_score
    F = load_has_topic(load_F1X()); F = F[F.year >= TOPIC_START].copy()
    F["nb"] = pd.cut(F.n_authors.fillna(1), [0, 1, 2, 3, 4, 5, 10, 1e9], labels=["1", "2", "3", "4", "5", "6-10", "11+"]).astype(str); F["solo"] = (F.n_authors == 1).astype(int)
    L = load_mass(); L = L[(L.scheme == "inv_n_authors") & (L.variant == "main")]; K = L[L.country != "UNK"].sort_values("w", ascending=False).drop_duplicates("pid").set_index("pid").country
    top = K.value_counts().head(8).index; F["cg"] = F.pid.map(K).map(lambda c: c if c in top else "other" if isinstance(c, str) else "UNK"); F.loc[F.year < WINDOW, "cg"] = "pre_window"
    T = []
    for dim, col in [("year", "year"), ("venue", "venue_short"), ("n_authors", "nb"), ("solo", "solo"), ("country_group", "cg")]:
        g = F.groupby(col).has_topic.agg(["size", "mean"]).reset_index().rename(columns={col: "level", "size": "n", "mean": "has_topic_rate"}); g["dim"] = dim; T.append(g)
    T = pd.concat(T); T.to_csv(S2 / "missingness_tables.csv", index=False, encoding="utf-8-sig")
    print("연도별 토픽 보유율:", F.groupby("year").has_topic.mean().round(3).to_dict()); print("저널별(하위 8):\n", T[T.dim == "venue"].sort_values("has_topic_rate").head(8).to_string(index=False))
    sp = SC / "coverage_vs_scopus.csv"
    if sp.exists():
        C = pd.read_csv(sp); cell = F.groupby(["venue_short", "year"]).has_topic.agg(["size", "mean"]).reset_index().rename(columns={"mean": "topic_rate"}); X = cell.merge(C[["venue_short", "year", "ratio", "flag"]], on=["venue_short", "year"], how="inner"); X = X[X["size"] >= 20]
        print(f"[Scopus 분모] 칸 {len(X)}개: 코퍼스/Scopus 비와 토픽 보유율의 Spearman {X.ratio.corr(X.topic_rate, method='spearman'):.2f}"); print(X.groupby("flag").topic_rate.agg(["size", "mean"]).round(3).to_string())
        X.to_csv(S2 / "missingness_vs_scopus.csv", index=False)
    out = []; W = {}
    for name, sub, cats in [("A", F, ["year", "venue_short", "nb"]), ("B", F[F.year >= WINDOW], ["year", "venue_short", "nb", "cg"])]:
        X = pd.get_dummies(sub[cats].astype(str), drop_first=True).astype(float); X["solo"] = sub.solo.values; y = sub.has_topic.values
        m = LogisticRegression(C=10, max_iter=2000); m.fit(X, y); p = m.predict_proba(X)[:, 1]
        pcv = cross_val_predict(LogisticRegression(C=10, max_iter=2000), X, y, cv=StratifiedKFold(pp["cv_folds"], shuffle=True, random_state=SEED), method="predict_proba")[:, 1]
        auc, aucv = roc_auc_score(y, p), roc_auc_score(y, pcv); pbar = y.mean(); w = np.where(y == 1, pbar / np.clip(p, 1e-6, 1), np.nan)
        lo, hi = np.nanpercentile(w, pp["clip_pct"]); wc = np.clip(w, lo, hi); ess = np.nansum(wc) ** 2 / np.nansum(wc ** 2); nt = int(y.sum()); low = p < pp["positivity_floor"]
        print(f"[모형 {name}] n={len(sub):,}, 토픽 보유 {nt:,} | AUC {auc:.3f} (CV {aucv:.3f}) | 유효 표본 {ess:,.0f} ({ess / nt:.1%}) | 절단 [{lo:.2f}, {hi:.2f}] | 양수성: 성향<{pp['positivity_floor']} 논문 {low.mean():.2%}")
        if low.any(): print("   양수성 위반 상위 저널:", sub[low].venue_short.value_counts().head(5).to_dict(), "→ 분리 보고와 경계(최악·최선 대입) 분석 대상")
        out.append(pd.DataFrame(dict(pid=sub.pid.values, model=name, p=p, w_ipw=wc, has_topic=y, low_p=low))); W[name] = dict(auc=auc, auc_cv=aucv, ess=ess, n_topic=nt, low_share=float(low.mean()))
    pd.concat(out).to_csv(S2 / "paper_ipw.csv.gz", index=False, compression="gzip"); json.dump(W, open(S2 / "propensity_diag.json", "w"), indent=1)
    print("※ 코호트(저자 층)와 경계 분석(최악·최선 대입)은 F4 구조 확인 후 추가. 가중치는 주제 논문 대상 분석에만 쓰고, 국가·연도 공급 몫은 토픽 유무와 무관하게 전체 논문으로 계산(2.6d)")
    changelog(f"- {now()} | missing 실행: 토픽 보유율 {F.has_topic.mean():.3f}, 모형 A/B 진단은 propensity_diag.json | S2.6")

# ============================================================
# [f2_check] F2/F1 정의 점검 (읽기 전용)
# ============================================================
def st_f2_check():
    hdr("[f2_check] F2 가중치 정의와 F1 열 점검")
    F1 = pd.read_csv(DATA / "F1_papers.csv", low_memory=False); F2 = pd.read_csv(DATA / "F2_author_paper.csv"); R = pd.read_csv(DATA / "F2_risk_names.csv")
    print("F1 in_sample:", F1.in_sample.value_counts().to_dict(), "| has_topic:", F1.has_topic.value_counts().to_dict(), "| (topic150 결측과 has_topic 일치:", bool((F1.topic150.notna() == F1.has_topic).all()), ")")
    print("in_sample x has_topic:\n", pd.crosstab(F1.in_sample, F1.has_topic).to_string())
    print("창(2007+) 안 in_sample False 논문:", int(((~F1.in_sample) & (F1.year >= WINDOW)).sum()), "(S1 국가 층은 in_sample을 거르지 않았음. 0이 아니면 2.6d 정의와 대조)")
    try:
        fv = set(json.loads((S0 / "PREREG.json").read_text(encoding="utf-8"))["event"]["fixed_venues"]); fx = set(F1[F1.fixed_venue].venue_short)
        print(f"F1.fixed_venue 저널 {len(fx)}개 vs PREREG 고정 저널 {len(fv)}개 | 같음: {fx == fv}")
        if fx != fv:
            print("   F1에만 있음:", sorted(fx - fv), "| PREREG에만 있음:", sorted(fv - fx))
            for v in sorted(fx ^ fv): q = F1[F1.venue_short == v]; print(f"   {v}: 논문 {len(q):,}편, 연도 {q.year.min()}~{q.year.max()}, F1.fixed_venue={bool(q.fixed_venue.any())}")
            print("   → S0 CHANGELOG에서 이 저널을 고정 목록에서 뺀(또는 넣은) 사유를 확인하세요. 분석은 PREREG 목록을 씁니다.")
    except Exception as e: print("PREREG 고정 저널 대조 실패:", e)
    g = F2.groupby("pid").agg(rows=("w", "size"), wsum=("w", "sum"), n=("n_authors", "first"), wmin=("w", "min"), wmax=("w", "max")).reset_index()
    print(f"\nF2 논문 {len(g):,}편 | F1 has_topic 논문 {int(F1.has_topic.sum()):,} | F2 pid가 F1 has_topic에 속함: {g.pid.isin(F1[F1.has_topic].pid).mean():.1%}")
    print(f"행 수 == n_authors(저자 전원 보존) 논문 비율: {(g.rows == g.n).mean():.1%} | 행 보존율(행/n_authors 합): {g.rows.sum() / g.n.sum():.1%}")
    print(f"w == 1/n_authors 인 행: {(np.abs(F2.w - 1 / F2.n_authors) < 1e-9).mean():.1%} | w == 1/(그 논문의 F2 행 수) 인 논문: {(np.abs(g.wmin - 1 / g.rows) < 1e-9).mean():.1%}")
    print("논문별 w 합 분위수:", g.wsum.quantile([0, .05, .25, .5, .95, 1]).round(3).to_dict(), f"| 합이 1(±1e-6)인 논문 {(np.abs(g.wsum - 1) < 1e-6).mean():.1%}")
    miss = F1[F1.has_topic & ~F1.pid.isin(g.pid)]
    print(f"has_topic인데 F2에 없는 논문 {len(miss):,}편(전원이 위험군 이름이어서 빠졌을 가능성): n_authors 분포 {miss.n_authors.value_counts().head(5).to_dict()}, 연도 중앙값 {miss.year.median():.0f}")
    print(f"위험군 이름 {len(R):,}개(ambiguous {int(R.ambiguous.sum())}, heavy {int(R.heavy.sum())}) | F2 행 중 위험군 이름 {F2.name_norm.isin(R.name_norm).mean():.2%}")
    m = F2.drop_duplicates("pid").merge(F1[["pid", "topic150"]], on="pid", suffixes=("", "_f1")); print(f"F2.topic150 == F1.topic150: {(m.topic150 == m.topic150_f1).mean():.1%}")
    print("\n해석: w가 1/(F2 행 수)이고 합이 모두 1이면 F2 = 위험군을 뺀 뒤 재정규화한 가중치(equal). inv_n_authors는 1/n_authors로 따로 만들어야 함.")

# ============================================================
# [risk] 위험군 재현율 2.2 (F2_risk_names 플래그 vs F7의 이름당 OpenAlex ID 수)
# ============================================================
def st_risk():
    hdr("[risk] 위험군 플래그 precision/recall (이름당 OpenAlex ID 2개 이상 = 분리된 사람의 근사 정답)"); check_prereg()
    R = pd.read_csv(DATA / RISK_FILE); D = pd.read_csv(DATA / F7_FILE, low_memory=False).dropna(subset=["name_norm"]); F1 = pd.read_csv(DATA / "F1_papers.csv", usecols=["pid", "year"])
    cov = D.merge(F1, on="pid").groupby("year").size(); print("F7 행의 연도 분포(10년 단위):", cov.groupby(cov.index // 10 * 10).sum().to_dict(), "| F7은 일부 authorship만 담고 있을 수 있음: 이 범위가 평가 범위")
    X = pd.DataFrame({"n_ids": D.groupby("name_norm").oa_id.nunique(), "n_rows": D.groupby("name_norm").size()}); X["multi"] = X.n_ids >= 2
    X = X.join(R.set_index("name_norm")[["ambiguous", "heavy"]]); X["ambiguous"] = X.ambiguous.fillna(False).astype(bool); X["heavy"] = X.heavy.fillna(False).astype(bool)
    print(f"F7 이름 {len(X):,}개 | 근사 정답(ID>=2) {int(X.multi.sum()):,} | 위험군 이름 중 F7에 없는 것 {int((~R.name_norm.isin(X.index)).sum())}/{len(R)}")
    out = []
    for scope, d in [("전체 이름", X), ("F7 행 2개 이상 이름", X[X.n_rows >= 2])]:
        for nm, fl in [("ambiguous", d.ambiguous), ("heavy", d.heavy), ("ambiguous|heavy", d.ambiguous | d.heavy)]:
            tp = int((d.multi & fl).sum()); prec = tp / max(int(fl.sum()), 1); rec = tp / max(int(d.multi.sum()), 1)
            out.append(dict(scope=scope, flag=nm, n_names=len(d), n_multi=int(d.multi.sum()), n_flag=int(fl.sum()), tp=tp, precision=round(prec, 3), recall=round(rec, 3)))
    O = pd.DataFrame(out); print(O.to_string(index=False)); O.to_csv(S2 / "risk_names_recall.csv", index=False)
    print("※ 정답이 근사입니다: 한 사람의 ID 분할도 ID>=2로 잡히고, 이름당 행이 1개뿐이면 ID>=2가 불가능합니다. 그래서 행 2개 이상 이름 기준을 병기했습니다. 동결된 위험군 변형 7종(PREREG variant_names)별 분해는 변형 정의 열이 이 파일에 없어 하지 않음")
    changelog(f"- {now()} | risk 실행(F2_risk_names 플래그 vs F7 이름당 ID 수): {out[-1]['flag']} / {out[-1]['scope']} precision {out[-1]['precision']}, recall {out[-1]['recall']} | S2.2")

# ============================================================
# [risk_selection] 위험군 이름 제외가 국가 구성에 주는 선택 효과 (F2에 없는 authorship = 제외된 저자)
# ============================================================
def st_risk_selection():
    hdr("[risk_selection] 위험군 이름 제외의 국가 선택 효과 (2007년 이후, 주제 논문)"); check_prereg()
    F = load_F1X(); A = build_authorship(F).merge(F[["pid", "year"]], on="pid")
    F2 = pd.read_csv(DATA / "F2_author_paper.csv", usecols=["pid", "name_norm", "author_pos"]); F2["pid"] = F2.pid.astype(str); F2["rank"] = F2.author_pos - 1
    F7 = pd.read_csv(DATA / F7_FILE, dtype={"pid": str}).dropna(subset=["name_norm"])
    j = F7.merge(F2[["pid", "rank", "name_norm"]], on=["pid", "rank"], suffixes=("_f7", "_f2"))
    print(f"[정렬 가정 검증] F7(pid, rank)와 F2(pid, author_pos-1) 매칭 {len(j):,}행, 이름 일치율 {(j.name_norm_f7 == j.name_norm_f2).mean():.1%} (author_pos = rank+1 가정)")
    T = A[A.pid.isin(set(F2.pid))].merge(F2[["pid", "rank"]].assign(kept=True), on=["pid", "rank"], how="left"); T["kept"] = T.kept.fillna(False).astype(bool)
    f2rows = int(F2.pid.isin(set(T.pid)).sum()); print(f"[정합] 창 안 주제 논문 {T.pid.nunique():,}편 | F2 행 {f2rows:,} vs authorship_level에서 F2와 매칭된 행 {int(T.kept.sum()):,}: {'PASS' if f2rows == int(T.kept.sum()) else 'FAIL(정렬 가정 또는 저자 순서 불일치 의심)'}")
    print(f"제외된 authorship 비중 {(~T.kept).mean():.1%}")
    T["c1"] = T.cs_main.str.split("|").str[0].replace("", "UNK"); top = T.c1.value_counts().head(8).index; T["cg"] = np.where(T.c1.isin(top), T.c1, "other")
    print("국가군별 제외 비중:\n", T.groupby("cg").kept.agg(n="size", dropped=lambda s: 1 - s.mean()).sort_values("n", ascending=False).round(3).to_string())
    Tk = T[T.kept].copy(); Tk_eq = Tk.copy(); Tk_eq["n"] = Tk_eq.groupby("pid")["rank"].transform("size")
    parts = {"전체 authorship(S1 방식)": country_mass(T, "cs_main", "inv_n_authors"), "equal(제외 후 재정규화)": country_mass(Tk_eq, "cs_main", "equal"), "inv_n(제외 질량 손실)": country_mass(Tk, "cs_main", "inv_n_authors")}
    yr = F.set_index("pid").year; rows = []
    for nm, M in parts.items():
        M = M.assign(year=M.pid.map(yr)); M["per"] = pd.cut(M.year, [2006, 2013, 2019, 2025], labels=["2007-13", "2014-19", "2020-25"])
        for per, g in M.groupby("per", observed=True):
            tot = g.w.sum(); kn = g[~g.country.isin(["UNK", "ZZ_former"])]; kt = kn.w.sum()
            for c in ["CN", "US", "HK"]: rows.append(dict(scheme=nm, period=str(per), country=c, share_incl_unk=g[g.country == c].w.sum() / tot, share_known=kn[kn.country == c].w.sum() / kt, mass=tot))
    R = pd.DataFrame(rows); R.to_csv(S2 / "risk_selection_country_shares.csv", index=False)
    print("\n국가 몫(국가 아는 질량 분모): 제외가 국가 구성을 바꾸는지\n", R.pivot_table(index=["country", "period"], columns="scheme", values="share_known").round(3).to_string())
    print("※ 위험군은 흔한 이름이라 동아시아 저자가 많이 빠질 수 있음: 저자 층 결과(S6, S7.2)의 지역 구성은 이 선택을 한계로 보고. 국가 층(S7)은 제외하지 않은 값이 주 분석")
    changelog(f"- {now()} | risk_selection 실행: 제외 authorship {(~T.kept).mean():.3f}, 국가별 제외 비중과 몫 차이는 risk_selection_country_shares.csv | S2.2, S6.6 범위 보고")

# ============================================================
# [commit]
# ============================================================
def st_commit():
    hdr("[commit] S2 산출물 해시 + 커밋 + 태그"); check_prereg()
    M = [dict(file=str(f.relative_to(ROOT)), sha256=sha_f(f)) for f in sorted(S2.glob("*")) if f.is_file() and f.name != "S2_MANIFEST.csv" and f.stat().st_size < 200e6]
    pd.DataFrame(M).assign(at=now()).to_csv(S2 / "S2_MANIFEST.csv", index=False); print(f"manifest {len(M)}개")
    if not COMMIT: print("COMMIT=False: 커밋 안 함"); return
    tag = TAG_NAME or f"S2-stageAB-{pd.Timestamp.now().strftime('%Y-%m-%d')}"
    if git("tag", "-l", tag).stdout.strip(): raise Halt(f"태그 {tag} 이미 있음")
    big = [f.name for f in S2.glob("*") if f.is_file() and f.stat().st_size > 50e6]
    if big: print("⚠ 50MB 초과 파일(커밋 용량 주의):", big)
    git("add", "--", "paper1/02_S2", check=True); c = git("commit", "-m", "S2 measurement stage A/B outputs"); print((c.stdout + c.stderr).strip()[:300]); git("tag", "-a", tag, "-m", "S2 stage A/B", check=True); print("태그:", tag)

FUNCS = {"schema": st_schema, "f2_check": st_f2_check, "prereg": st_prereg, "clean_flags": st_clean_flags, "country_rec": st_country_rec, "weights_country": st_weights_country,
         "unk_resid": st_unk_resid, "missing": st_missing, "risk": st_risk, "risk_selection": st_risk_selection, "commit": st_commit}

if __name__ == "__main__":
    assert S0.exists() and S1.exists(), "ROOT를 확인하세요"; S2.mkdir(parents=True, exist_ok=True); print(f"ROOT={ROOT} | STEPS={STEPS}")
    try:
        for s in STEPS: FUNCS[s]()
        print("\n✔ 요청한 단계 완료")
    except Halt as e: print(f"\n⛔ 중단: {e}")

ROOT=/home/yjlee/Research/Minnesota_Transport | STEPS=['f2_check']

[f2_check] F2 가중치 정의와 F1 열 점검
F1 in_sample: {True: 111685} | has_topic: {True: 88164, False: 23521} | (topic150 결측과 has_topic 일치: True )
in_sample x has_topic:
 has_topic  False  True 
in_sample              
True       23521  88164
창(2007+) 안 in_sample False 논문: 0 (S1 국가 층은 in_sample을 거르지 않았음. 0이 아니면 2.6d 정의와 대조)
F1.fixed_venue 저널 15개 vs PREREG 고정 저널 14개 | 같음: False
   F1에만 있음: ['JPT'] | PREREG에만 있음: []
   JPT: 논문 661편, 연도 1996~2025, F1.fixed_venue=True
   → S0 CHANGELOG에서 이 저널을 고정 목록에서 뺀(또는 넣은) 사유를 확인하세요. 분석은 PREREG 목록을 씁니다.

F2 논문 87,525편 | F1 has_topic 논문 88,164 | F2 pid가 F1 has_topic에 속함: 100.0%
행 수 == n_authors(저자 전원 보존) 논문 비율: 73.8% | 행 보존율(행/n_authors 합): 89.4%
w == 1/n_authors 인 행: 74.8% | w == 1/(그 논문의 F2 행 수) 인 논문: 100.0%
논문별 w 합 분위수: {0.0: 1.0, 0.05: 1.0, 0.25: 1.0, 0.5: 1.0, 0.95: 1.0, 1.0: 1.0} | 합이 1(±1e-6)인 논문 100.0%
has_topic인데 F2에 없는 논문 639편(전원이 위험군 이름이어서 빠졌을 가능성): n_authors 분포 {2: 237, 1: 196, 3: 145

In [ ]:
# `f2_check` 결과로 가중치 정의가 확정됐고, `prereg`를 실행해도 됩니다. 확인해 주실 것이 하나 남아 있습니다.

# ## 결과 판독
# - **고정 저널:** F1의 `fixed_venue` 열은 15개이고, PREREG에는 JPT가 없어 14개입니다. JPT는 661편, 1996~2025년입니다. 14개로 확정된 사유는 S0 CHANGELOG에서 찾아보세요. 저는 사유를 확인하지 못했습니다.
# - **분석에 쓸 저널 목록:** 동결된 PREREG의 14개를 쓰기로 `prereg` 문구에 적어 두었습니다. Scimago 쪽도 14개 기준이라 서로 맞습니다.
# - **F4, F5의 영향 가능성:** F4/F5가 `fixed_venue` 열로 만들어졌다면 `continuing_main`, `entrant_main`, 비고정 저널 비율에 JPT가 고정으로 들어가 있을 수 있습니다. 이 열들을 S6에서 쓰기 전에 어떻게 만들었는지 확인하세요.
# - **F2에서 빠진 639편:** 이 논문들은 저자 전원이 위험군 이름이라 빠진 것으로 보입니다. 저자 수 분포는 1인 196편, 2인 237편, 3인 145편, 4인 49편, 5인 11편입니다. 토픽 논문의 0.7%이고, 단독 저자 논문이 `equal` 가중에서 통째로 사라지는 규모입니다. 저자 층 결과에는 이 손실을 논문 단위로 보고하세요.
# - **저자 행 손실:** 10.6%가 위험군 제외로 빠졌습니다. `inv_n_authors`는 이 질량을 손실로 보고하는 정의입니다.
# - **나머지 점검 항목:** `in_sample`은 전부 True라 영향이 없고, F2의 토픽은 F1과 100% 같습니다.

# ## 지금 할 일
# 1. **`prereg` 실행 전에 아래 값을 최종 확인하세요.** 동결 뒤에는 사후 변경이 됩니다.
#    - 반복 제목 기준 `repeat_title_min=5`
#    - VT Mag 규칙의 정규식 `\[\s*mobile radio\s*\]`
#    - 성향 절단 1/99 백분위
#    - 양수성 기준 0.05
#    - 수축 상수 k=5
#    - 핵심 목록 C1~C5. C4만 이 파일에서 계산되고, 나머지는 후속 단계의 몫입니다.
#    - 위 숫자는 모두 제 임의 출발값이니 바꾸실 것이 있으면 지금 바꾸세요.
# 2. 설정에서 `CORE_CONFIRMED = True`, `STEPS = ["prereg"]`로 실행하세요. 출력에 S0 PREREG의 weights 항목이 나오니, 제가 쓴 정의(`equal` = 위험군 제외 후 재정규화, `inv_n_authors` = 1/n_authors, `first_last`, `full_count`)와 문언이 같은지 대조하세요. 다르면 알려 주세요.
# 3. 이어서 `["clean_flags","country_rec","weights_country","unk_resid"]`를 실행하고 출력을 붙여 주세요.
# `prereg` 출력과 JPT 사유(S0 CHANGELOG에 있다면)를 같이 붙여 주시면 다음 판독을 하겠습니다.
# 지금 할 일을 순서대로 실행하는 `s2_run_now.py`를 만들었습니다. 문법 검사와, 동결 파일을 만들고 해시를 확인한 뒤 재실행 시 동일하다고 통과하는지 가짜 폴더로 시험했습니다. 실제 자료로는 아직 돌려 보지 못했습니다.
# 이 파일은 `s2_measurement.py`(v3)를 불러서 쓰므로, 같은 폴더나 `~/Research/Minnesota_Transport`, `~/Downloads`에 두세요. 구버전이면 시작할 때 멈춥니다.

# ## 실행 순서 (`PHASE`를 바꿔 세 번 실행)
# 1. **`PHASE = "check"`** (읽기 전용)
#    - S0 PREREG의 weights 항목과 고정 저널 목록을 출력합니다.
#    - S0·S1 문서에서 JPT가 언급된 줄을 검색해, 고정 저널에서 빠진 사유를 찾습니다.
#    - 사유가 문서에 없으면 `prereg` 전에 CHANGELOG_S2에 직접 기록하세요.
#    - weights 문언이 제 정의와 다르면 `prereg`를 하지 말고 알려 주세요.


# 2. **`PHASE = "prereg"`**
#    - 파일 위쪽의 값 5개와 `CORE_CONFIRMED = True`를 설정하고 실행합니다. 동결되는 값은 다음과 같습니다.
#      - 반복 제목 기준: `REPEAT_TITLE_MIN = 5`
#      - VT Mag 규칙의 정규식
#      - 성향 가중 절단 백분위: 1/99
#      - 양수성 기준: 0.05
#      - 수축 상수: k=5
#    - 이 값들은 제 임의 출발값이니 바꾸실 것이 있으면 이 단계 전에 바꾸세요. 동결 뒤에 바꾸면 사후 변경이라 CHANGELOG에 기록해야 합니다.
#    - `CORE_CONFIRMED`를 켜기 전에 `s2_measurement.py`의 `CORE_LIST`(C1~C5)를 읽어 보세요. C4만 이 단계에서 계산되고 나머지는 후속 단계의 몫입니다.


# 3. **`PHASE = "analysis"`**
#    - `clean_flags`, `country_rec`, `weights_country`, `unk_resid`를 차례로 실행합니다.
#    - 한 단계에서 멈추거나 오류가 나면 거기서 중단하고 전체 오류를 출력합니다.


# ## 붙여 주실 것
# - `check`의 출력(JPT 사유와 weights 문언)
# - `prereg`가 끝난 뒤의 해시
# - `analysis` 전체 출력. 특히 `country_rec`의 불변식 PASS/FAIL과 S1 재현 최대 차이, `weights_country`의 부호 일치 비율을 봅니다.
# `country_rec`에서 불변식이 FAIL이면 `weights_country`는 실행되지 않고 멈추도록 되어 있습니다.

In [14]:
# -*- coding: utf-8 -*-
# =====================================================================================
# S2 지금 할 일 실행 셀 (s2_measurement.py v3 필요, OpenAlex 호출 없음)
# 사용법: PHASE를 순서대로 바꿔 실행
#   PHASE = "check"     JPT 사유 검색 + S0 PREREG weights 출력 (읽기 전용). 결과를 보고 아래 값을 확정
#   PHASE = "prereg"    S2 사전등록 부록 동결 (되돌리기 어려움: 값 확정 후에만)
#   PHASE = "analysis"  clean_flags, country_rec, weights_country, unk_resid 실행
# analysis 중 한 단계가 멈추면(Halt/오류) 거기서 중단하고 이유를 출력함
# =====================================================================================
import importlib.util, re, json, traceback
from pathlib import Path

PHASE = "check"

# ---- prereg에 동결될 값(결과를 보기 전에 확정. 기본값은 제 임의 출발값) ----
REPEAT_TITLE_MIN = 5            # 같은 저널 안에서 정규화 제목이 이 횟수 이상 반복되면 표시
VT_MOBILE_RADIO_REGEX = r"\[\s*mobile radio\s*\]"
CLIP_PCT = [1, 99]              # 성향 가중 절단 백분위
POSITIVITY_FLOOR = 0.05
SHRINK_K = 5
CORE_CONFIRMED = False          # C1~C5 핵심 목록을 확인했으면 True (prereg 단계에서 필요)
# ---------------------------------------------------------------------

ROOT = Path.home() / "Research/Minnesota_Transport"
cands = [Path.cwd() / "s2_measurement.py", ROOT / "s2_measurement.py", ROOT / "paper1/s2_measurement.py",
         ROOT / "paper1/02_S2/s2_measurement.py", Path.home() / "Downloads/s2_measurement.py"]
MOD = next((c for c in cands if c.exists()), None)
assert MOD is not None, f"s2_measurement.py를 찾지 못했습니다. 후보: {[str(c) for c in cands]}"
spec = importlib.util.spec_from_file_location("s2", MOD); m = importlib.util.module_from_spec(spec); spec.loader.exec_module(m)
assert "f2_check" in m.FUNCS and "risk_selection" in m.FUNCS, "구버전 s2_measurement.py입니다. v3(최신)를 사용하세요"
print(f"모듈: {MOD} | PHASE={PHASE}")
m.S2.mkdir(parents=True, exist_ok=True)

def phase_check():
    m.hdr("[check] S0 PREREG weights와 JPT 사유")
    try:
        pr = json.loads((m.S0 / "PREREG.json").read_text(encoding="utf-8"))
        print("S0 PREREG weights:", pr.get("weights"))
        print("S0 PREREG fixed_venues:", pr["event"]["fixed_venues"], f"({len(pr['event']['fixed_venues'])}개)")
    except Exception as e: print("PREREG 읽기 실패:", e)
    print("\nS0·S1 문서에서 JPT 언급 검색(고정 저널 제외 사유):")
    n = 0
    for base in [m.S0, m.S1]:
        for p in sorted(base.glob("*")):
            if p.is_file() and p.suffix in (".md", ".txt", ".json", ".csv") and p.stat().st_size < 5e6:
                try: txt = p.read_text(encoding="utf-8", errors="ignore")
                except Exception: continue
                for ln in txt.splitlines():
                    if re.search(r"\bJPT\b", ln): print(f"  [{p.relative_to(m.ROOT)}] {ln.strip()[:240]}"); n += 1
                    if n >= 25: break
            if n >= 25: break
    if n == 0: print("  언급 없음: 사유가 문서로 남아 있지 않다면 prereg 전에 CHANGELOG_S2에 직접 기록하세요")
    print("\n확인 후 아래 값을 필요하면 바꾸고 PHASE='prereg'로 실행:",
          dict(REPEAT_TITLE_MIN=REPEAT_TITLE_MIN, VT_REGEX=VT_MOBILE_RADIO_REGEX, CLIP_PCT=CLIP_PCT, POSITIVITY_FLOOR=POSITIVITY_FLOOR, SHRINK_K=SHRINK_K))

def phase_prereg():
    # 모듈 전역값을 덮어써서 make_spec이 이 값을 동결하게 함
    m.CORE_CONFIRMED = CORE_CONFIRMED
    ms = m.make_spec
    def patched():
        sp = ms(); cr = sp["clean_rules"]
        cr["repeat_title_min"] = REPEAT_TITLE_MIN; cr["vt_mobile_radio"]["regex"] = VT_MOBILE_RADIO_REGEX
        sp["propensity"]["clip_pct"] = CLIP_PCT; sp["propensity"]["positivity_floor"] = POSITIVITY_FLOOR; sp["portfolio"]["shrink_k"] = SHRINK_K
        return sp
    m.make_spec = patched
    try: m.st_prereg()
    except m.Halt as e: print("⛔ 중단:", e)

def phase_analysis():
    for s in ["clean_flags", "country_rec", "weights_country", "unk_resid"]:
        try: m.FUNCS[s]()
        except m.Halt as e: print(f"\n⛔ {s} 중단: {e}"); return
        except Exception: print(f"\n⛔ {s} 오류(아래 전체 출력을 붙여 주세요):"); traceback.print_exc(); return
    print("\n✔ analysis 4단계 완료")

{"check": phase_check, "prereg": phase_prereg, "analysis": phase_analysis}[PHASE]()

모듈: /home/yjlee/Research/Minnesota_Transport/paper1/s2_measurement.py | PHASE=check

[check] S0 PREREG weights와 JPT 사유
S0 PREREG weights: ['equal', 'inv_n_authors', 'first_last', 'full_count']
S0 PREREG fixed_venues: ['AAP', 'CACIE', 'JTG', 'T-ITS', 'TR-A', 'TR-B', 'TR-C', 'TR-D', 'TR-E', 'TR-F', 'TRR', 'TransSci', 'TranspPol', 'TranspRev'] (14개)

S0·S1 문서에서 JPT 언급 검색(고정 저널 제외 사유):
  [paper1/00_S0/DOI_PREFIX_REPORT.csv] JPT,10.5038,528,0.7987897125567323,1996,2021,0.0
  [paper1/00_S0/DOI_PREFIX_REPORT.csv] JPT,10.1016,132,0.19969742813918306,2022,2025,0.0
  [paper1/00_S0/DOI_PREFIX_REPORT.csv] JPT,10.18154,1,0.0015128593040847202,2025,2025,0.0
  [paper1/00_S0/EVENTS_CATALOG_FROZEN.csv] JPT,1996,founding_oa,high,1996 가을 창간; 창간부터 오픈액세스; USF CUTR,,
  [paper1/00_S0/EVENTS_CATALOG_FROZEN.csv] JPT,2022,publisher_change,high,2022-01 Elsevier 인수,,
  [paper1/00_S0/EVENTS_DRAFT.csv] JPT,661,0.0182648401826484,,,,,,1996,1996.0,,False,1996,False,False,False,False
  [paper1/00_S0/EVENTS_FROZEN.csv]

In [ ]:
# ## 총평

# 이 결과들은 코퍼스가 Scopus와 총량 수준에서 잘 맞는다는 쪽을 지지합니다. 칸 단위 이탈은 대부분 결측이 아니라 연도 귀속 차이로 읽힙니다. 실질적인 문제로 남는 것은 2025년, RailEng, T-IV 2025입니다. S2 쪽은 `prereg` 전에 정리할 것이 세 가지 있습니다.

# ## 1. Scimago 후속 진단

# **총량은 잘 맞습니다.** 표 [6]을 제가 합산하면 고정 14개의 19개 연도 합계는 코퍼스 64,779편, Scopus 64,088편으로 +1.1%입니다. 연도별 비는 0.90~1.15입니다.

# **칸 단위 이탈은 잡음이 아니라 구조적입니다.** 고정 저널 표시율은 20.4%(54/265)입니다. 비고정 저널에서는 문서 30편 이상 칸이 31.8%, 미만 칸이 35.1%로 규모에 거의 영향을 받지 않습니다. 같은 논문 집합을 세는 두 출처의 차이라면 잡음은 작아야 하므로, 정의나 연도 귀속 차이로 보는 것이 맞습니다.

# **연도 귀속 이동의 증거가 뚜렷합니다.** 인접 연도의 높음과 낮음이 짝으로 나타나고, 합치면 사라집니다.

# | 대상 | 코퍼스 | Scopus |
# |---|---|---|
# | T-ITS 2020~22 | 3,032 | 3,049 |
# | 고정 저널 전체 2021~22 | 9,510 | 9,407 |
# | AAP 2007~09 | 588 | 564 |
# | TranspPol 2010~11 | 166 | 158 |

# 가설은 OpenAlex가 온라인 선공개 연도를, Scopus가 호(issue) 연도를 쓴다는 것입니다. F1의 연도 정의를 확인하고 `publication_date`와 volume/issue 연도를 비교하면 검증할 수 있습니다. 이것은 S5의 사건 대조(±1년)에 직접 영향을 줍니다. T-ITS는 약 2년 이동처럼 보이므로 변화점 해석 때 연도 정의를 명시해야 합니다.

# **실질 문제로 남는 것:**
# - **2025년:** 합계 차가 -1,158이고 T-TE, T-IV, TRR, T-ITS가 모두 부족합니다. 고정 저널도 0.956이고, 앞서 본 2025년 미상 1.9%와 같은 방향입니다.
# - **T-IV 2025:** 비가 0.06이고, 2024년은 1.13이었습니다. 시차로는 설명되지 않는 급락이라 수집 누락을 먼저 의심해야 합니다.
# - **RailEng:** 비 중앙값이 0이라 15칸 중 적어도 8칸에 코퍼스 논문이 없습니다. 저널 ID나 DOI 매핑을 확인하세요.
# - **ITS Mag와 VT Mag:** 풀링 비가 0.80과 0.88입니다. 이는 Scimago 비인용 문서 비중 0.26과 0.15와 방향이 일치하므로, 문서 유형 정의 차이로 보는 설명이 자연스럽습니다. 다만 증명된 것은 아닙니다.

# **다국적 공저 대조:**
# - 풀링 Spearman 0.887은 저널 간 수준 차이가 섞여 부풀려진 값입니다. 저널 내 rho(0.41~0.77)가 더 엄격한 지표인데, 칸이 11~19개뿐이라 구간이 넓습니다. 예컨대 n=19에서 0.58이면 95% 구간이 대략 0.2~0.8입니다. 저널 내 rho를 주 지표로 보고하세요.
# - VT Mag의 수준 차이(14.2%p)는 다른 저널 최대치(7.1%p)의 약 2배이지만, 제외해도 풀링 값이 거의 안 변합니다(0.899).
# - T-ITS 2008은 두 출처 모두 62편으로 커버리지 문제가 아닌데 21.0% 대 61.3%로 벌어집니다. 이웃 연도(25~37%)와도 동떨어집니다. 원인은 단정할 수 없습니다. 논문 10편 정도를 직접 열어 보면 OpenAlex가 초기 연도 다중 소속을 놓치는지 판가름되고, S1 수작업 200건이 2008년 근처를 포함했는지도 확인할 수 있습니다.

# ## 2. f2_check

# **확정된 것:**
# - F2.w는 위험군 제외 후 1/(남은 저자 수)로 재정규화된 값입니다. 논문별 합이 모두 1이므로 `inv_n_authors`는 따로 만들어야 합니다.
# - `in_sample`은 전부 True라 영향이 없습니다.
# - 토픽 열은 F1과 100% 일치하고, 결측 21.06%는 `has_topic`과 일치합니다.

# **미해결이거나 주의할 것:**
# - **저자 행 손실 10.6%를 위험군 제외로 귀속한 것은 아직 검증되지 않았습니다.** 이름 결측 등 다른 경로도 가능합니다. 또 보존율 89.4%는 EDA의 86%와 대조되지 않았습니다. `risk_selection`의 정합 점검이 이를 확인합니다.
# - **639편 손실(토픽 논문의 0.72%)**은 `equal` 가중에서 단독 저자 196편이 통째로 사라지는 규모이므로 논문 단위로 보고해야 합니다.
# - **JPT는 사유가 문서로 남아 있지 않습니다.** 검색에서는 DOI 접두 변경과 2022년 Elsevier 인수 사건만 나왔고, 제외 사유는 없었습니다. 인수와 접두 변경이 겹친다는 점은 사실이지만, 그것이 제외 이유라는 것은 추측입니다. 사유를 모르면 CHANGELOG_S2에 "사유 미확인"으로 적고, F4/F5가 `fixed_venue`로 만들어졌는지 확인하세요.
# - **국가 층에서는 `equal`과 `inv_n_authors`가 같은 값입니다.** 실질 3종이므로 부호 일치 비율은 3종 기준으로 병기해야 합니다. 4종으로 세면 일치가 과장됩니다.

# ## 3. `prereg` 전에 정리할 것

# 1. **S0 PREREG의 weights에는 이름 4개만 있고 정의 문언이 없습니다.** "문언 대조"는 이름 일치까지만 확인됐고, 정의는 계획서 2.1에서 가져온 것입니다. 부록이 정의의 최초 동결본이 되므로 그 점을 CHANGELOG에 명시하세요.
# 2. **계획서 2.1의 "저자 순서 알파벳순 사전 점검"이 스크립트에 구현돼 있지 않습니다.** 알파벳순 논문은 `first_last`에서 빼고 `equal`로 처리하는 규칙인데, 지금은 전 논문에 `first_last`가 적용됩니다. 동결 전에 구현하거나, 제외하기로 하는 변경을 기록해야 합니다.
# 3. **연도 이동은 기술 열로만 추가하고 임계값(0.80/1.25)은 건드리지 마세요.** 2년 풀링 비를 보조 열로 만들고, 사후 추가임을 CHANGELOG에 적으면 됩니다.

# ## 권장 순서

# 1. 알파벳순 점검 구현, JPT 사유 기록
# 2. `prereg` 실행
# 3. `analysis` 실행
# 4. 병행: T-IV 2025와 RailEng 수집 점검, F1 연도 정의 확인
# 5. 2025년은 변화점 탐지에서 끝점 민감도(2025 제외)를 사전 명시

# 원자료를 직접 읽지 못한 부분(예: TranspPol 2008의 32편 대 40편)은 평가에서 제외했습니다.

In [ ]:
# 세 가지를 한 파일로 묶어 `s2_fix_and_run.py`를 만들겠습니다.

# `s2_fix_and_run.py`를 만들었습니다. 문법 검사와 가짜 자료 시험은 하지 못했고, 실제 자료로도 아직 돌려 보지 못했습니다. `s2_measurement.py`(v3)와 같은 폴더나 `~/Research/Minnesota_Transport`, `~/Downloads`에 두세요.

# ## 세 가지가 처리되는 방식

# 1. **알파벳순 점검**
#    - `alpha` 단계가 F2에서 남은 저자의 `author_pos` 순서를 보고 이름 키가 비감소인 논문을 알파벳순으로 판정합니다.
#    - 판정 대상은 남은 저자 3명 이상인 논문입니다. 2인 논문은 우연히 알파벳순일 확률이 50%이고, `first_last`도 2인은 0.5씩이라 `equal`과 같기 때문입니다.
#    - 관측 건수와 우연 기대(1/k!)를 전체, 저자 수별, 기간별로 출력합니다. 결과는 `alpha_flags.csv`에 저장되고 해시가 부록에 동결됩니다.
#    - `analysis` 단계에서는 알파벳순 논문에 `first_last` 대신 `equal`(1/n)이 적용됩니다.
# 2. **JPT 사유 기록**: 문서에서 사유를 못 찾았으므로 상단의 `JPT_REASON`에 "사유 미확인"과 검색된 사실(DOI 접두 변경, 2022년 Elsevier 인수)을 적어 두었습니다. 사유를 찾으면 이 문자열을 고친 뒤 `prereg`를 실행하세요.
# 3. **weights 정의의 출처**: S0 PREREG에는 이름 4개만 있었고 S2 부록이 정의의 최초 동결본이라는 점을 부록과 CHANGELOG_S2에 모두 적습니다.

# ## 실행 순서

# 1. `PHASE="alpha"`로 실행합니다. 출력에 `name_norm` 표본 10개가 나옵니다. 성이 앞이면 `first_token`, 뒤면 `last_token`, 판단이 어려우면 `full`로 `NAME_KEY`를 정합니다. `NAME_KEY`가 `None`이면 중단됩니다.
# 2. `NAME_KEY`를 넣고 `alpha`를 다시 실행해 표를 확인합니다. 표를 보고 정할 것은 이름 형식뿐이고, 기준(`ALPHA_MIN_N=3`)은 지금 확정해 두어야 사후 변경이 되지 않습니다.
# 3. `CORE_CONFIRMED=True`와 나머지 값 5개(`REPEAT_TITLE_MIN` 등)를 확인하고 `PHASE="prereg"`로 실행합니다.
# 4. `PHASE="analysis"`로 실행하고 전체 출력을 붙여 주세요.

# ## 알아 둘 점

# - F2는 토픽이 있는 논문만 담고 있어서, 토픽 없는 논문과 저자 전원이 위험군인 논문은 알파벳순을 판정할 수 없습니다. 이런 논문에는 `first_last`가 그대로 적용되고, 이 사실은 부록에 명시됩니다. 국가 층(전체 논문 대상)에서는 `first_last`가 일부만 보정된다는 뜻입니다.
# - 위험군으로 빠진 저자가 있는 논문은 남은 저자의 순서만 봅니다. 이 경우 알파벳순으로 오판할 가능성이 약간 올라가므로, `alpha`가 출력하는 "저자 전원 보존 논문만" 표와 비교해 보세요.
# - 이 스크립트는 `analysis`를 직접 실행하므로 `s2_run_now.py`의 `analysis`는 쓰지 마세요. 그쪽에는 알파벳순 규칙이 적용되지 않습니다.

In [8]:
# -*- coding: utf-8 -*-
# =====================================================================================
# S2 prereg 전 정리 3종 + 동결 + 분석 실행 (s2_measurement.py v3 필요, OpenAlex 호출 없음)
#   1) 저자 순서 알파벳순 점검 구현 (계획서 2.1): 알파벳순 논문은 first_last에서 equal(1/n)로 처리
#   2) JPT가 고정 저널에서 빠진 사유를 CHANGELOG_S2에 기록 (문서로 확인 안 되면 '사유 미확인'으로 기록)
#   3) weights 정의의 최초 동결본이 S2 부록이라는 점을 부록과 CHANGELOG_S2에 명시
# 사용법: PHASE를 순서대로 바꿔 실행
#   PHASE = "alpha"     알파벳순 점검 (읽기 전용 + alpha_flags.csv 저장). 이름 형식 확인 후 NAME_KEY 지정
#   PHASE = "prereg"    위 3종을 포함해 S2 부록 동결 (되돌리기 어려움: 값 확정 후에만)
#   PHASE = "analysis"  알파벳순 규칙을 적용해 clean_flags, country_rec, weights_country, unk_resid 실행
# =====================================================================================
import importlib.util, json, hashlib, traceback
from pathlib import Path
import numpy as np, pandas as pd
from scipy.special import gammaln

PHASE = "analysis"

# ---- 알파벳순 규칙 (결과를 보기 전에 확정: alpha 단계 출력의 이름 표본을 보고 NAME_KEY만 정함) ----
NAME_KEY = "last_token"        # "full": name_norm 전체 / "first_token": 첫 토큰이 성 / "last_token": 마지막 토큰이 성
ALPHA_MIN_N = 3        # 남은 저자 수가 이 값 이상인 논문만 판정(2인은 우연 일치 50%라 제외, first_last도 2인은 0.5씩이라 equal과 같음)
# ---- prereg에 동결될 값 (s2_run_now.py와 같은 값을 유지) ----
REPEAT_TITLE_MIN = 5
VT_MOBILE_RADIO_REGEX = r"\[\s*mobile radio\s*\]"
CLIP_PCT = [1, 99]
POSITIVITY_FLOOR = 0.05
SHRINK_K = 5
CORE_CONFIRMED = True
# ---- JPT 사유: S0/S1 문서에서 확인되지 않았음. 사유를 찾으면 이 문자열을 고치고 prereg 하세요 ----
JPT_REASON = "사유 미확인(S0/S1 문서에 제외 기록 없음; 검색된 것은 DOI 접두 10.5038->10.1016 변경과 2022-01 Elsevier 인수 사건뿐이며 이것이 제외 사유인지는 확인되지 않음). 분석은 PREREG.json의 14개 목록을 사용"
# -------------------------------------------------------------------------

ROOT = Path.home() / "Research/Minnesota_Transport"
cands = [Path.cwd() / "s2_measurement.py", ROOT / "s2_measurement.py", ROOT / "paper1/s2_measurement.py",
         ROOT / "paper1/02_S2/s2_measurement.py", Path.home() / "Downloads/s2_measurement.py"]
MOD = next((c for c in cands if c.exists()), None)
assert MOD is not None, f"s2_measurement.py를 찾지 못했습니다. 후보: {[str(c) for c in cands]}"
_sp = importlib.util.spec_from_file_location("s2", MOD); m = importlib.util.module_from_spec(_sp); _sp.loader.exec_module(m)
assert "f2_check" in m.FUNCS and "risk_selection" in m.FUNCS, "구버전 s2_measurement.py입니다. v3(최신)를 사용하세요"
print(f"모듈: {MOD} | PHASE={PHASE}")
m.S2.mkdir(parents=True, exist_ok=True)
ALPHA_CSV = m.S2 / "alpha_flags.csv"

# ---------------------------------------------------------------------------
def name_key(s):
    s = s.fillna("").astype(str).str.strip().str.lower()
    if NAME_KEY == "full": return s
    if NAME_KEY == "first_token": return s.str.split().str[0].fillna("")
    if NAME_KEY == "last_token": return s.str.split().str[-1].fillna("")
    raise m.Halt('NAME_KEY를 "full", "first_token", "last_token" 중 하나로 지정하세요(위에 출력된 이름 표본의 형식을 보고 정함)')

def phase_alpha():
    m.hdr("[alpha] 저자 순서 알파벳순 점검 (F2 기준: 위험군 제외 후 남은 저자의 순서)")
    F2 = pd.read_csv(m.DATA / "F2_author_paper.csv", usecols=["pid", "name_norm", "author_pos", "n_authors"], dtype={"pid": str})
    F1 = pd.read_csv(m.DATA / "F1_papers.csv", usecols=["pid", "year"], dtype={"pid": str})
    F2 = F2.sort_values(["pid", "author_pos"]).reset_index(drop=True)
    print("name_norm 표본(형식 확인용: 성이 앞인지 뒤인지):", F2.name_norm.dropna().sample(10, random_state=m.SEED).tolist())
    F2["key"] = name_key(F2.name_norm); F2["bad"] = F2.key == ""
    prev = F2.groupby("pid")["key"].shift(); F2["viol"] = prev.notna() & (F2.key < prev)
    g = F2.groupby("pid").agg(k=("pid", "size"), n=("n_authors", "first"), bad=("bad", "any"), viol=("viol", "any")).reset_index()
    g["complete"] = g.k == g.n; g["evaluable"] = (g.k >= ALPHA_MIN_N) & ~g.bad; g["is_alpha"] = g.evaluable & ~g.viol
    g["p_chance"] = np.where(g.evaluable, np.exp(-gammaln(g.k + 1)), np.nan)   # 무작위 순서에서 알파벳순일 확률 1/k!(동점 무시)
    g = g.merge(F1, on="pid", how="left"); g["period"] = pd.cut(g.year, [1899, 2006, 2013, 2019, 2025], labels=["<=2006", "2007-13", "2014-19", "2020-25"])
    g["kbin"] = pd.cut(g.k, [0, 2, 3, 4, 6, 10, 10 ** 6], labels=["<=2", "3", "4", "5-6", "7-10", "11+"])
    print(f"F2 논문 {len(g):,}편 | 판정 가능(남은 저자 >= {ALPHA_MIN_N}, 이름 결측 없음) {int(g.evaluable.sum()):,}편 ({g.evaluable.mean():.1%})")
    def rep(d, by):
        t = d[d.evaluable].groupby(by, observed=True).agg(papers=("pid", "size"), alpha=("is_alpha", "sum"), expected=("p_chance", "sum")).reset_index()
        t["obs_rate"] = t.alpha / t.papers; t["exp_rate"] = t.expected / t.papers; t["excess"] = t.alpha - t.expected; return t.round(4)
    print("\n[전체] 관측 vs 우연 기대:\n", rep(g.assign(all="all"), "all").to_string(index=False))
    print("\n[남은 저자 수별]\n", rep(g, "kbin").to_string(index=False))
    print("\n[기간별]\n", rep(g, "period").to_string(index=False))
    print("\n[저자 전원 보존 논문만(complete)]\n", rep(g[g.complete].assign(all="complete"), "all").to_string(index=False))
    print("※ 관측률이 우연 기대(1/k!)를 크게 넘는 구간이 '알파벳순 관행'의 증거입니다. 규칙은 기대를 넘는지와 무관하게 정의대로 적용되며, 이 표는 영향 규모를 보고하기 위한 것입니다.")
    g[["pid", "k", "n", "complete", "evaluable", "is_alpha"]].to_csv(ALPHA_CSV, index=False)
    print(f"\n저장: {ALPHA_CSV.name} sha256 {m.sha_f(ALPHA_CSV)[:12]} | 알파벳순 판정 {int(g.is_alpha.sum()):,}편")
    print("한계: F2는 토픽 논문(위험군 제외 후)만 담고 있어 토픽 없는 논문과 전원 위험군 논문은 판정 불가 -> 그 논문은 first_last를 그대로 적용(부록에 명시됨)")

def load_alpha_set(spec):
    if not ALPHA_CSV.exists(): raise m.Halt("먼저 PHASE='alpha'를 실행하세요")
    if m.sha_f(ALPHA_CSV) != spec["alpha_flags_sha256"]: raise m.Halt("alpha_flags.csv가 동결 후 변경되었습니다")
    A = pd.read_csv(ALPHA_CSV, dtype={"pid": str}); return set(A.loc[A.is_alpha, "pid"])

def phase_prereg():
    if not ALPHA_CSV.exists(): raise m.Halt("먼저 PHASE='alpha'를 실행하세요")
    if NAME_KEY is None: raise m.Halt("NAME_KEY가 지정되지 않았습니다(alpha 단계에서 쓴 값과 같아야 함)")
    sha = m.sha_f(ALPHA_CSV); m.CORE_CONFIRMED = CORE_CONFIRMED; ms = m.make_spec
    def patched():
        sp = ms(); cr = sp["clean_rules"]; cr["repeat_title_min"] = REPEAT_TITLE_MIN; cr["vt_mobile_radio"]["regex"] = VT_MOBILE_RADIO_REGEX
        sp["propensity"]["clip_pct"] = CLIP_PCT; sp["propensity"]["positivity_floor"] = POSITIVITY_FLOOR; sp["portfolio"]["shrink_k"] = SHRINK_K
        w = sp["weights"]
        w["first_last_alpha_rule"] = dict(name_key=NAME_KEY, min_n=ALPHA_MIN_N, source="F2_author_paper(위험군 제외 후 남은 저자의 author_pos 순서)",
                                          rule=f"남은 저자 {ALPHA_MIN_N}명 이상이고 name key가 author_pos 순으로 비감소(동점 허용)이면 알파벳순 논문. first_last에서만 equal(1/n)로 대체, 다른 가중치에는 영향 없음",
                                          not_applicable="F2에 없는 논문(토픽 없음, 전원 위험군)은 판정 불가이므로 first_last를 그대로 적용")
        w["definitions_origin"] = "S0 PREREG.json은 가중치 이름 4개만 담고 있어 정의 문언이 없었음. 이 부록의 정의(계획서 2.1에서 옮김)가 가중치 정의의 최초 동결본"
        w["jpt_note"] = JPT_REASON; sp["alpha_flags_sha256"] = sha
        return sp
    m.make_spec = patched; m.st_prereg()
    m.changelog(f"- {m.now()} | 알파벳순 점검 구현·동결(name_key={NAME_KEY}, min_n={ALPHA_MIN_N}, alpha_flags sha256 {sha[:12]}): 알파벳순 논문은 first_last에서 equal로 처리 | 계획서 2.1에 있었으나 v3 스크립트에 구현 누락 | first_last 결과에만 영향, 국가 층 first_last 질량 변화")
    m.changelog(f"- {m.now()} | weights 정의의 최초 동결본은 S2 부록 | S0 PREREG.json에는 이름 4개만 있고 정의 문언 없음(대조는 이름 일치까지만 가능했음) | 정의는 계획서 2.1 기준")
    m.changelog(f"- {m.now()} | JPT(F1.fixed_venue=True, PREREG 고정 14개에는 없음): {JPT_REASON} | f2_check와 S0/S1 문서 검색 | F4/F5가 fixed_venue 열로 만들어졌다면 JPT가 고정으로 들어갔을 수 있어 S6 전에 확인 필요")

def phase_analysis():
    spec = m.check_prereg()
    if "alpha_flags_sha256" not in spec: raise m.Halt("부록에 알파벳순 규칙이 없습니다. 이 스크립트의 prereg 단계로 동결했는지 확인하세요")
    AL = load_alpha_set(spec); orig = m.author_w
    def author_w(A, scheme):
        w = np.asarray(orig(A, scheme), dtype=float)
        if scheme == "first_last":
            mask = A["pid"].isin(AL).values; w[mask] = 1.0 / A["n"].values.astype(float)[mask]
        return w
    m.author_w = author_w; print(f"알파벳순 논문 {len(AL):,}편에 first_last 대신 equal 적용")
    for s in ["clean_flags", "country_rec", "weights_country", "unk_resid"]:
        try: m.FUNCS[s]()
        except m.Halt as e: print(f"\n⛔ {s} 중단: {e}"); return
        except Exception: print(f"\n⛔ {s} 오류(아래 전체 출력을 붙여 주세요):"); traceback.print_exc(); return
    print("\n✔ analysis 4단계 완료")

{"alpha": phase_alpha, "prereg": phase_prereg, "analysis": phase_analysis}[PHASE]()

모듈: /home/yjlee/Research/Minnesota_Transport/paper1/s2_measurement.py | PHASE=analysis
알파벳순 논문 6,011편에 first_last 대신 equal 적용

[clean_flags] 비연구 항목 규칙 표시
토픽 파일 F1_papers.csv, 열 topic150: 값 분포 상위 {nan: 23521, 137.0: 1246, 72.0: 1075, 147.0: 1028, 75.0: 1021, 4.0: 984, 52.0: 973, 22.0: 969}
F1X 111,416편 중 토픽 보유 79.1% (계획서 EDA: 결측 약 21%와 대조)
규칙별 표시 건수: {'f_bib': 0, 'f_vtmr': 68, 'f_repeat': 57, 'f_dup': 441, 'f_topic18': 1} | 하나라도: 546 (0.49%)
(참고: 계획서의 '중복 제목 459행'과 f_dup 건수를 대조. 다르면 정의가 다른 것이니 이유를 확인)
규칙별 30건 표본 저장: clean_review_samples.csv (correct 열에 1/0 수작업 확인)
반복 제목 상위:
 venue_short                                                                                t  n
        JTG                                                                        tgrg page 21
   TransSci                                                                books of interest 13
   TransSci                                                                 book of interest 11
        JTG tgsg the transport geog

In [9]:
#

In [ ]:
# 알파벳순 점검은 정상적으로 끝났고, 이 결과로 `prereg`에 넘어가도 됩니다. 다만 이 규칙이 하는 일이 무엇인지는 정확히 적어 두어야 합니다.

# ## 판독

# | 남은 저자 수 | 판정 논문 | 알파벳순 | 우연 기대 | 초과 건수 | 초과분이 차지하는 몫 |
# |---|---|---|---|---|---|
# | 3 | 24,039 | 4,769 | 4,007 | 763 | 16% |
# | 4 | 15,908 | 1,002 | 663 | 339 | 34% |
# | 5~6 | 12,111 | 219 | 75 | 144 | 66% |
# | 7~10 | 2,896 | 20 | 0.4 | 19.6 | 98% |

# - **알파벳순 관행의 증거는 있습니다.** 전체에서 관측 6,011편이 우연 기대 4,745편보다 1,266편 많고, 관측 대 기대 비는 약 1.27입니다.
# - **3~4인 논문에서는 알파벳순으로 판정된 논문 대부분이 우연 일치입니다.** 3인에서는 판정된 논문의 84%, 4인에서는 66%가 기대치 안에 있습니다. 반면 5인 이상에서는 판정이 대부분 실제 관행을 가리킵니다(7~10인은 사실상 전부).
# - **그래서 규칙은 "알파벳순 관행 논문"이 아니라 "저자 순서가 알파벳순과 일치하는 논문"을 가려내는 것입니다.** 부록과 논문에서 이렇게 표현하세요.
# - **실제 영향은 작습니다.** 판정된 6,011편 중 5,771편(96%)이 3~4인입니다. 이 구간에서 `first_last`와 `equal`의 가중치 차이는 저자당 0.067 이하(3인: 0.3/0.4/0.3 대 0.333, 4인: 0.3/0.2/0.2/0.3 대 0.25)입니다. 차이가 큰 5인 이상 논문은 약 240편뿐입니다.
# - **기간별 비로 보면 시간 추세는 약합니다.** 관측 대 기대 비가 1.34, 1.33, 1.30, 1.19(2020~25)입니다. 관측률이 15.4%에서 8.6%로 내려간 것은 대부분 팀 규모가 커진 효과입니다.
# - **위험군으로 저자가 빠진 논문이 판정을 부풀리지는 않았습니다.** 저자 전원 보존 논문의 비(1.32)가 전체 비(1.27)보다 높고, 그 외 논문만 보면 1.11로 더 낮습니다. 걱정했던 방향과 반대입니다.

# ## 주의할 점

# 우연 기대 1/k!는 동점을 무시한 값입니다. 판정은 동점을 허용(비감소)하므로, Zhang, Wang, Li처럼 흔한 성이 많은 코퍼스에서는 실제 우연 일치율이 더 높고, 표의 초과분은 과대 추정입니다. 따라서 "초과분 1,266편"은 상한으로 보고하세요.

# ## 결정

# `ALPHA_MIN_N=3`을 그대로 쓰시기를 권합니다. 4로 올리면 3인 논문의 우연 일치를 줄일 수 있지만, 이 표를 본 뒤 바꾸는 것이라 사후 변경이 되고 `first_last` 결과도 거의 달라지지 않습니다. 이미 정한 기준을 지키는 편이 방어하기 쉽습니다.

# ## 다음 할 일

# 1. `PHASE = "prereg"`로 바꾸세요. `NAME_KEY`는 방금 쓴 값(`"last_token"`)과 같아야 합니다. 출력에는 키가 표시되지 않으므로, 상단 설정이 그 값인지 한 번 확인하세요.
# 2. `CORE_CONFIRMED = True`로 바꾸기 전에 `CORE_LIST`(C1~C5)와 값 5개(`REPEAT_TITLE_MIN=5`, VT Mag 정규식, `CLIP_PCT=[1, 99]`, `POSITIVITY_FLOOR=0.05`, `SHRINK_K=5`)를 마지막으로 확인하세요.
# 3. `JPT_REASON`은 사유를 찾지 못했다면 지금 문구 그대로 두셔도 됩니다.
# 4. `prereg` 이후에는 `alpha`를 다시 실행하지 마세요. `alpha_flags.csv`(해시 `4002a150851b`)가 동결 기록과 어긋나면 `analysis`가 중단됩니다.
# 5. `prereg` 출력에서 동결 해시가 나오면 `PHASE = "analysis"`로 넘어가고, 그 전체 출력을 붙여 주세요.

In [ ]:
# 이 중단은 의도한 동작이고, 아무것도 동결되지 않았습니다. 핵심 목록을 확인했다는 표시(`CORE_CONFIRMED`)가 `False`라서 `st_prereg`가 파일을 쓰기 전에 멈췄습니다. 부록 파일도 CHANGELOG 줄도 만들어지지 않았으니 그대로 다시 실행하셔도 됩니다.

# ## 지금 할 일

# 1. **핵심 목록 C1~C5를 읽고 확정하세요.** 이 목록은 `s2_measurement.py`의 `CORE_LIST`에 있고, 결과를 보기 전에 정해야 하는 항목입니다. 지금 들어 있는 안은 다음과 같습니다.

#    | ID | 통계 | 출처 | 이번 단계에서 계산 |
#    |---|---|---|---|
#    | C1 | 주제 유효수·HHI 시간 추세 부호 | S5.1(a) | 안 함(주제 층 필요) |
#    | C2 | 연속 노출 Spearman(Δexposure, d_all−d_fixed) 부호 | S5 주 분석 | 안 함 |
#    | C3 | 갱신 4항 분해의 진입·재배분 몫 방향 | S6 | 안 함(저자 층 필요) |
#    | C4 | 상위 5개국 몫(`top5_incl`, `top5_known`)과 `hhi_known`의 연도 추세 부호 | S7 | `weights_country`에서 계산 |
#    | C5 | 신규 저자 지역 구성 변화 방향 | S7.2 | 안 함(저자 층 필요) |

#    빼거나 더할 항목이 있으면 `s2_measurement.py`의 `CORE_LIST`를 고치세요. 한 가지만 짚으면, C4는 19개 연도의 Spearman이라 |rho|가 작으면 부호가 불안정합니다. 그래도 판정 기준은 부호 일치뿐이라고 부록에 이미 적혀 있으니, 그 점을 알고 두시면 됩니다.

# 2. **나머지 값을 마지막으로 확인하세요.** `REPEAT_TITLE_MIN=5`, VT Mag 정규식, `CLIP_PCT=[1, 99]`, `POSITIVITY_FLOOR=0.05`, `SHRINK_K=5`, `ALPHA_MIN_N=3`, `NAME_KEY="last_token"`입니다. 동결 뒤에 바꾸면 사후 변경입니다.

# 3. **스크립트 상단에서 `CORE_CONFIRMED = True`로 바꾸고 `PHASE = "prereg"`로 다시 실행하세요.** 이 스크립트는 같은 변수를 모듈에 넘겨 주므로 `s2_measurement.py` 쪽 `CORE_CONFIRMED`는 건드리지 않으셔도 됩니다.

# ## 성공했을 때 보일 것

# - `S0 PREREG의 weights 항목: [...]` 줄과 `동결: S2_PREREG_ADDENDUM.json <해시 12자리>` 줄이 출력됩니다. 이 해시를 붙여 주세요.
# - 그 뒤 CHANGELOG_S2에 기록 줄 4개(부록 동결, 알파벳순 규칙, weights 정의 출처, JPT)가 생깁니다.

# 해시가 나오면 곧바로 `PHASE = "analysis"`로 넘어가시면 됩니다.

In [ ]:
# 동결이 완료됐습니다. 부록 해시는 `646135ccaa68`이고, 이 값을 기록해 두세요. 앞에서 안내한 대로 `alpha_flags.csv`(해시 `4002a150851b`)는 이제 바꾸면 안 되고, `alpha`도 다시 실행하지 마세요.

# ## 확인할 것 하나

# 출력에는 부록이 어떤 값으로 동결됐는지가 나오지 않습니다. CHANGELOG_S2에 4줄(부록 동결, 알파벳순 규칙, weights 정의 출처, JPT)이 들어갔는지 열어서 확인하세요. 4줄이 아니라면 `m.changelog` 호출이 중간에 실패한 것이므로 알려 주세요. 이 경우 동결 자체는 유효하고, 빠진 줄은 직접 추가하면 됩니다.

# ## 다음 단계

# `PHASE = "analysis"`로 바꿔 실행하세요. `clean_flags`, `country_rec`, `weights_country`, `unk_resid`가 차례로 돌고, 한 단계가 멈추면 거기서 중단됩니다. 출력 전체를 붙여 주시면 아래를 우선 판독하겠습니다.

# - **`country_rec`의 불변식 PASS/FAIL과 S1 재현 최대 차이.** FAIL이면 `weights_country`는 실행되지 않습니다. 알파벳순 규칙이 `first_last` 질량 합을 깨지 않는지도 이 단계에서 확인됩니다.
# - **`weights_country`의 C4 부호 일치 비율.** 국가 층에서는 `equal`과 `inv_n_authors`가 같은 값이라 실질 3종입니다. 부호 일치 비율은 3종 기준으로 읽겠습니다.
# - **`clean_flags`의 `f_dup` 건수.** 계획서의 "중복 제목 459행"과 비교합니다.
# - **`clean_flags`의 Scimago 참고 상관.** 정리 규칙 표시 비중과 Scimago 비인용 문서 비중의 Spearman입니다. 정의가 달라서 목표가 아니라 그럴듯한지 보는 점검입니다.

# `clean_flags`는 `clean_review_samples.csv`에 규칙별 30건 표본을 저장합니다. `correct` 열에 1/0을 수작업으로 채우는 단계는 분석이 끝난 뒤 병행하시면 됩니다.

In [ ]:
# ## 총평

# `analysis` 4단계가 모두 끝났고 불변식은 전부 통과했습니다. 핵심 목록 C4는 24개 조합 중 23개에서 부호가 일치했고, 갈린 한 조합은 아래에서 설명합니다. 다만 이 결과에서 읽어낼 수 있는 것이 세 가지 있고, 확인이 필요한 항목도 몇 개 있습니다.

# ## 1. country_rec

# - 불변식 8개(변형 2종 × 가중치 4종)가 모두 PASS이고, S1 `country_authorship`과의 최대 차이는 6.7e-16입니다. 국가 층 재구축은 S1을 그대로 재현합니다.
# - `first_last` 불변식은 논문당 합이 1인지만 봅니다. 알파벳순 논문을 `equal`로 바꿔도 합은 1이라, 이 PASS는 알파벳순 적용 여부를 검증하지 않습니다. 적용은 "6,011편에 적용"이라는 출력으로만 확인됩니다.
# - 창 안 논문이 81,645편이고 authorship이 있는 논문은 81,441편입니다. 앞서 정리된 S1의 "전수 수집 81,485편"과는 160편 차이가 납니다. 정의가 다른 숫자일 수 있어 단정은 못 하지만, 같은 대상을 센 숫자라면 원인을 한 번 확인하세요.
# - 미상 질량 몫은 주 변형에서 12.7%(2007)에서 3.5%(2023)로 떨어집니다. 복구판은 2.8%에서 0.5%로 떨어지고, 두 변형 모두 2024~2025년에 다시 오릅니다(복구판 1.1%, 2.0%).

# ## 2. weights_country: C4 부호표

# | 통계 | 결과 | 판독 |
# |---|---|---|
# | `top5_incl` | 8개 조합 모두 `++++` | 연도에 따라 증가 |
# | `top5_known` | 8개 조합 모두 `++++` | 연도에 따라 증가 |
# | `hhi_known` | 7개 조합 `----`, 1개 `--+-` | 연도에 따라 감소(1개 조합 제외) |

# **`top5_incl`의 증가는 일부 기계적일 수 있습니다.** 분모에 미상이 들어가는데 주 변형의 미상이 12.7%에서 3.5%로 줄었기 때문입니다. 다만 미상을 분모에서 뺀 `top5_known`과 복구판에서도 같은 부호이므로, 부호 자체가 이 때문만은 아닙니다. 주된 근거는 `top5_known`으로 쓰세요.

# **상위 5개국 몫은 늘고 HHI는 줄었습니다.** 이는 모순이 아닙니다. 한 나라의 지배가 약해지고 상위권이 여러 나라로 나뉘어도 일어나는 조합입니다. 다만 그 설명은 가설이고, `country_year_stats.csv`의 `us_known`, `cn_known` 열로 확인해야 합니다. 어느 쪽이든 S7 서술에서 "집중이 높아졌다/낮아졌다"로 한마디로 쓰면 안 되고, 통계별로 방향을 구분해 써야 합니다.

# **갈린 한 조합은 `main` 변형, 전체 표본, 영토 합산입니다.**
# - `hhi_known`의 부호가 `--+-`이고, `full_count`만 +입니다. rho 범위가 -0.21~0.01이라 사실상 방향이 없는 구간입니다.
# - 사전등록 규칙에 따르면 이 조합은 FAIL입니다. "사실상 0이니 통과"로 바꾸지 말고, "이 조합에서 `hhi_known`의 방향은 정의에 민감하다"고 보고하세요.
# - 연도가 19개면 |rho|가 약 0.46은 넘어야 5% 수준에서 유의합니다. 다른 조합의 rho 크기도 `weights_signs_summary.csv`에서 확인하세요. 부호가 같아도 |rho|가 작으면 불안정합니다.
# - 국가 층에서는 `equal`과 `inv_n_authors`가 같은 값이라 독립적인 가중치는 3종입니다. 23/24는 3종 기준으로 다시 세어도 같은 숫자입니다.

# ## 3. clean_flags

# - **`f_bib` 0건과 `f_topic18` 1건이 이상합니다.** 규칙이 틀렸거나, S0의 `EXCLUDE_PIDS_FROZEN`이 이미 이 항목들을 뺐을 가능성이 있습니다. F1 전체(제외 전)에서 같은 정규식과 `topic150 == 18` 건수를 세어 보면 가려집니다.
# - **`f_dup` 441건과 계획서의 459행 차이(18건)는 아직 설명되지 않았습니다.** F1X는 F1에서 269편이 빠져 있어, 그 안에 중복 18건이 있었다면 맞아떨어집니다. 이것도 F1 기준으로 다시 세면 확인됩니다. 가설일 뿐입니다.
# - **`f_repeat` 57건은 반복 제목 상위 5개의 합(21+13+11+7+5)과 같습니다.** JTG와 TransSci의 칼럼성 항목만 잡히고 있습니다.
# - **Scimago 참고 상관 0.19는 거의 정보가 없습니다.** 표시 비중이 전체의 0.49%뿐이라 변동이 작습니다. 대신 `clean_summary_venue_year.csv`에서 ITS Mag, CommTR, VT Mag, TranspRev의 `any_share`를 Scimago 비인용 문서 비중(0.26, 0.17, 0.15, 0.10)과 나란히 보세요. 규칙이 못 잡는 비연구 문서가 이 저널들에 있는지 가늠할 수 있습니다. 정의가 달라 증명은 못 되고 점검입니다.

# ## 4. unk_resid

# 복구 후에도 소속 문자열이 있는데 미상인 authorship이 2024~2025년에 34건(0.04%)뿐입니다. 이 해의 복구 후 미상 중 96.8%는 소속 문자열 자체가 비어 있습니다. 규칙 쪽의 문제가 아니라 원자료에 소속이 없는 저자입니다. 2025년이 높은 이유는 출처 쪽 시차일 수 있지만 확인된 것은 아닙니다. 2025년은 Scimago 대조(-1,158편)에서도 약한 해여서 같은 방향입니다. 표본 30건은 34건 중 거의 전부이니 읽어 보시면 됩니다.

# ## 5. 제가 빠뜨린 것

# 앞서 "2025년 끝점 민감도를 사전 명시"하라고 했는데 `prereg` 스크립트에는 넣지 않았습니다. 동결된 부록에는 없습니다. 결과를 이미 보셨으므로, 지금 추가하려면 CHANGELOG_S2에 "동결 후 추가한 탐색적 민감도(2025 제외 재계산, 판정 규칙은 불변)"로 기록해야 합니다.

# ## 다음 순서

# 1. 위 3번의 점검(`f_bib`, `topic18`, `f_dup`을 F1 기준으로 다시 세기)은 짧은 스크립트로 한 번에 돌릴 수 있습니다. 필요하시면 만들어 드리겠습니다.
# 2. `s2_measurement.py`에서 `STEPS = ["missing"]`, `["risk"]`, `["risk_selection"]`을 차례로 실행하세요. 이 단계들은 `first_last`를 쓰지 않으므로 알파벳순 보정이 필요 없습니다.
# 3. `clean_review_samples.csv`의 `correct` 열과 `unk_resid_2024_25_sample.csv`는 병행해서 채우시면 됩니다.
# 4. 마지막에 `commit`을 `COMMIT=False`로 먼저 돌려 매니페스트를 확인하고, 이후 `True`로 태그까지 남기세요.

In [10]:
# -*- coding: utf-8 -*-
# =====================================================================================
# clean_flags 0건·소수 건 점검 (읽기 전용, 파일 저장 없음, OpenAlex 호출 없음)
# 질문: (1) f_bib 0건, f_topic18 1건은 규칙 문제인가, S0 제외(EXCLUDE_PIDS_FROZEN)가 이미 뺀 결과인가?
#       (2) f_dup 441건과 계획서의 '중복 제목 459행' 차이는 F1 전체(제외 전) 기준이거나 정의 차이로 설명되는가?
# s2_measurement.py(v3)와 같은 폴더/후보 경로에 두세요. 부록(S2_PREREG_ADDENDUM.json)의 정규식과 주제 번호를 그대로 씁니다.
# =====================================================================================
import importlib.util, warnings
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings("ignore"); pd.set_option("display.width", 220); pd.set_option("display.max_columns", 30); pd.set_option("display.max_colwidth", 110)

PLAN_DUP_ROWS = 459           # 계획서 2.8의 '중복 제목 459행'
LOOSE_BIB = r"bibliograph"    # 느슨한 대조용(규칙 자체를 바꾸는 것이 아님)

ROOT = Path.home() / "Research/Minnesota_Transport"
cands = [Path.cwd() / "s2_measurement.py", ROOT / "s2_measurement.py", ROOT / "paper1/s2_measurement.py", ROOT / "paper1/02_S2/s2_measurement.py", Path.home() / "Downloads/s2_measurement.py"]
MOD = next((c for c in cands if c.exists()), None); assert MOD is not None, f"s2_measurement.py를 찾지 못했습니다: {[str(c) for c in cands]}"
sp = importlib.util.spec_from_file_location("s2", MOD); m = importlib.util.module_from_spec(sp); sp.loader.exec_module(m)

spec = m.check_prereg(); cr = spec["clean_rules"]; T18 = cr["topic_excluded"]; BIB = cr["bib_section"]
F1 = pd.read_csv(m.DATA / "F1_papers.csv", low_memory=False); F1["pid"] = F1.pid.astype(str)
FX = m.load_F1X()                                  # 동결 해시 검증 포함
F1["excluded"] = ~F1.pid.isin(set(FX.pid)); F1["t"] = F1.title.fillna("").map(m.normt)
parts = {"F1 전체(제외 전)": F1, "F1X(분석 대상)": F1[~F1.excluded], "S0 제외분": F1[F1.excluded]}
print(f"F1 {len(F1):,}편 | F1X {len(FX):,}편 | S0 제외 {int(F1.excluded.sum()):,}편 | 규칙(부록): bib={BIB!r}, 제외 토픽={T18}")

def has(d, pat): return d.title.fillna("").str.contains(pat, case=False, regex=True)

def dup_counts(d):
    d = d[d.t != ""]; k = ["venue_short", "year", "t"]
    return {"저널·연도·제목: 첫 건 뒤 중복(f_dup 정의)": int(d.duplicated(k).sum()), "저널·연도·제목: 중복군에 속한 전체 행": int(d.duplicated(k, keep=False).sum()),
            "저널·제목(연도 무관): 첫 건 뒤 중복": int(d.duplicated(["venue_short", "t"]).sum()), "제목만: 첫 건 뒤 중복": int(d.duplicated(["t"]).sum()),
            "제목만: 중복군에 속한 전체 행": int(d.duplicated(["t"], keep=False).sum())}

rows = []
for nm, d in parts.items():
    r = {"표본": nm, "편수": len(d), "f_bib(부록 규칙)": int(has(d, BIB).sum()), f"느슨한 '{LOOSE_BIB}'": int(has(d, LOOSE_BIB).sum()), f"topic150 in {T18}": int(d.topic150.isin(T18).sum())}
    r.update(dup_counts(d)); rows.append(r)
R = pd.DataFrame(rows).set_index("표본").T; R["계획서 459와 같은 값?"] = ""
for ix in R.index:
    if "중복" in ix: R.loc[ix, "계획서 459와 같은 값?"] = " / ".join(nm for nm in parts if R.loc[ix, nm] == PLAN_DUP_ROWS)
print("\n[1] 규칙별 건수(표본별)\n", R.to_string())

print("\n[2] 느슨한 패턴에 걸린 제목 표본 (부록 규칙이 못 잡는 표기가 있는지)")
lb = F1[has(F1, LOOSE_BIB)]
print(f"  F1 전체 {len(lb)}편 | 부록 규칙에도 걸림 {int(has(lb, BIB).sum())} | S0 제외분 {int(lb.excluded.sum())}")
print(lb[["venue_short", "year", "excluded", "title"]].head(12).to_string(index=False))

print("\n[3] 제외 토픽 번호 논문 (F1 전체)")
t18 = F1[F1.topic150.isin(T18)]; print(f"  {len(t18)}편 | 제외분 {int(t18.excluded.sum())}편 | 저널 분포 {t18.venue_short.value_counts().head(5).to_dict()}")
print(t18[["venue_short", "year", "excluded", "title"]].head(10).to_string(index=False))
print("  (계획서는 '쇠퇴 13개 주제', 'VT Mag 38번 칼럼' 등을 언급하지만 18번의 원래 규모는 이 출력으로만 확인됩니다)")

print("\n[4] S0 제외분이 어떤 항목인지 (규칙에 걸린 것 / 저널·연도 상위)")
ex = F1[F1.excluded]
print(f"  제외 {len(ex)}편 중 f_bib {int(has(ex, BIB).sum())}, 제외 토픽 {int(ex.topic150.isin(T18).sum())}, VT Mag [mobile radio] {int((has(ex, cr['vt_mobile_radio']['regex']) & (ex.venue_short == cr['vt_mobile_radio']['venue'])).sum())}")
print("  저널별:", ex.venue_short.value_counts().head(8).to_dict(), "| 연도 범위", int(ex.year.min()), "~", int(ex.year.max()))

print("\n[5] 같은 저널·연도·제목 중복 (F1X, 상위 저널)")
d = FX.assign(t=FX.title.fillna("").map(m.normt)); d = d[d.t != ""]; dd = d[d.duplicated(["venue_short", "year", "t"])]
print("  ", dd.venue_short.value_counts().head(8).to_dict())
print("해석 순서: [1]에서 F1 전체 기준 값이나 다른 정의 중 하나가 459와 같으면 차이는 정의/표본 차이. 어느 것도 같지 않으면 계획서 459는 다른 조건(예: 비논문 유형 포함)에서 나온 수치일 수 있어 출처를 확인하세요.")

F1 111,685편 | F1X 111,416편 | S0 제외 269편 | 규칙(부록): bib='bibliographic section', 제외 토픽=[18]

[1] 규칙별 건수(표본별)
 표본                            F1 전체(제외 전)  F1X(분석 대상)  S0 제외분 계획서 459와 같은 값?
편수                                 111685      111416     269               
f_bib(부록 규칙)                           35           0      35               
느슨한 'bibliograph'                      46          11      35               
topic150 in [18]                        1           1       0               
저널·연도·제목: 첫 건 뒤 중복(f_dup 정의)          463         441      20               
저널·연도·제목: 중복군에 속한 전체 행                909         872      33               
저널·제목(연도 무관): 첫 건 뒤 중복                562         512      48               
제목만: 첫 건 뒤 중복                         577         523      52               
제목만: 중복군에 속한 전체 행                    1057         990      63               

[2] 느슨한 패턴에 걸린 제목 표본 (부록 규칙이 못 잡는 표기가 있는지)
  F1 전체 46편 | 부록 규칙에도 걸림 35 | S0 제외분 35
venue_short  year  excluded           

In [11]:
# ## 결론

# 규칙은 의도대로 작동하고 있고, 동결된 규칙을 바꿀 필요는 없습니다. 다만 두 가지가 풀리지 않았습니다. 18번 토픽 규칙이 사실상 작동하지 않는다는 점과, 계획서의 459가 재현되지 않는다는 점입니다.

# ## 1. `f_bib` 0건: 설명됐습니다

# F1 전체에서 `f_bib` 35건은 모두 S0 제외분(TransSci의 "Bibliographic Section", 1997~2003년)이었고 F1X에는 하나도 없습니다. S0가 이미 뺀 것이라 규칙은 이 코퍼스에서는 할 일이 없습니다.

# 느슨한 패턴에서 F1X에 남은 11편은 "bibliographic analysis" 같은 제목의 실제 연구 논문이고, T-ITS의 서지 분석 논문 등이 그렇습니다. 부록의 엄격한 규칙(`bibliographic section`)이 이들을 잡지 않은 것이 올바른 동작이므로 정규식을 느슨하게 만들지 마세요.

# ## 2. `f_topic18` 1건: 규칙이 겨눈 대상이 아닌 것 같습니다

# F1 전체에서 `topic150 == 18`인 논문은 1편뿐입니다. 1974년 TRR의 도로 조명과 운전자 작업량 논문으로 실제 연구 논문이고, S0 제외분에는 18번이 하나도 없습니다.

# 계획서의 "주제 18번 제외"가 비연구 항목 군집을 겨눈 규칙이라면, 지금 데이터에서 18번은 그런 군집이 아닙니다. 가능한 설명은 두 가지로 생각됩니다.
# - 번호 체계가 다릅니다(0 기준과 1 기준 등).
# - 해당 군집이 이미 토픽 결측으로 처리됐습니다.

# 어느 쪽인지는 이 출력으로 확인되지 않습니다. 이 규칙이 연구 논문 1편을 표시하고 있으니 사실상 쓸모없는 규칙입니다. 영향 규모는 무시할 수 있지만, S3의 L5 검증("18번과 VT Mag 38번 칼럼을 규칙으로 잡는지")이 같은 번호를 쓰므로 번호 체계는 확인해 두세요. 아래 스니펫으로 확인할 수 있습니다.

# ```python
# # 어떤 토픽 번호에 비연구 항목이 몰려 있는지 확인 (읽기 전용)
# import pandas as pd
# F1 = pd.read_csv(m.DATA / "F1_papers.csv", low_memory=False)
# print("토픽 번호 범위:", F1.topic150.min(), F1.topic150.max(), "| 서로 다른 값", F1.topic150.nunique())
# t = F1.title.fillna("")
# print("TransSci 'Bibliographic Section':", F1[t.str.contains("bibliographic section", case=False)].topic150.value_counts(dropna=False).head(5).to_dict())
# print("VT Mag [Mobile Radio]:", F1[(F1.venue_short == "VT Mag") & t.str.contains(r"\[\s*mobile radio\s*\]", case=False)].topic150.value_counts(dropna=False).head(5).to_dict())
# print("TransSci 'books of interest':", F1[t.str.contains("books? of interest", case=False)].topic150.value_counts(dropna=False).head(5).to_dict())
# ```

# - VT Mag 칼럼 항목이 38번에 몰려 있으면 번호 체계는 맞고, 18번이 비연구 군집이 아니라는 뜻입니다.
# - 다른 번호에 몰려 있거나 대부분 결측이면 번호가 어긋났거나 비연구 항목이 토픽에서 빠진 것입니다.
# - `f_topic18`을 고치는 것은 동결된 `topic_excluded=[18]`을 바꾸는 일이므로 사후 변경입니다. 새 부록 파일로 분리하고 CHANGELOG_S2에 사유를 쓰세요.

# ## 3. `f_dup` 441건 대 459: 어느 정의로도 재현되지 않았습니다

# 정의 5종 중 459가 되는 것이 없습니다. 가장 가까운 값은 F1 전체에서 `f_dup` 정의로 센 463이고, 차이는 4입니다. 그래서 "F1X에서 제외분이 빠져서 줄었다"는 설명은 맞지 않습니다. 459는 다른 시점의 F1이나 다른 정규화로 나온 수치일 수 있지만, 이 출력으로 단정할 수는 없습니다. 계획서 459의 출처를 아시면 확인하시고, 모르시면 "재현 안 됨, 441 사용"으로 기록하세요.

# 참고로 441건 중 352건(80%)이 TRR입니다. "같은 저널·연도·제목"이 곧 중복 레코드는 아닙니다. TRR에는 같은 해에 같은 제목이 반복되는 항목(예: 서문, 색인류)이 있을 수 있는데, 어느 쪽인지는 이 출력으로 알 수 없습니다. `clean_review_samples.csv`의 `f_dup` 30건을 읽어 보시면 판가름됩니다. 중복 레코드가 아닌 정당한 항목이 많다면 `f_dup`은 "제거"가 아니라 "표시"로만 두어야 한다는 근거가 됩니다. 이미 표시만 하고 제거하지 않는 설계입니다.

# ## 4. 부수적으로 알게 된 것

# S0 제외 269편은 TRR 74, AAP 67, T-ITS 36, TransSci 35 순입니다. TransSci 35편은 `Bibliographic Section`으로 설명되지만, 나머지 234편의 제외 사유는 이 출력으로는 알 수 없습니다. S0의 REVIEW 파일에 사유가 있을 가능성이 있으니, `S2.8`의 "제거 전후 비교"를 쓸 때 사유를 확인하세요.

# ## 기록할 것

# 동결된 규칙은 바꾸지 않았으므로 CHANGELOG_S2에는 한 줄만 남기시면 됩니다.

# > `f_bib` 0건은 S0 제외(35편)로 설명됨, `f_topic18`은 F1 전체에서 1건(실제 연구 논문)으로 번호 체계 확인 필요, `f_dup` 441은 계획서 459와 어떤 정의로도 재현되지 않음(F1 전체 기준 463). 규칙 변경 없음.

# 다음은 `missing`, `risk`, `risk_selection` 실행입니다. 위 스니펫 결과도 같이 붙여 주세요.

In [12]:
# 어떤 토픽 번호에 비연구 항목이 몰려 있는지 확인 (읽기 전용)
import pandas as pd
F1 = pd.read_csv(m.DATA / "F1_papers.csv", low_memory=False)
print("토픽 번호 범위:", F1.topic150.min(), F1.topic150.max(), "| 서로 다른 값", F1.topic150.nunique())
t = F1.title.fillna("")
print("TransSci 'Bibliographic Section':", F1[t.str.contains("bibliographic section", case=False)].topic150.value_counts(dropna=False).head(5).to_dict())
print("VT Mag [Mobile Radio]:", F1[(F1.venue_short == "VT Mag") & t.str.contains(r"\[\s*mobile radio\s*\]", case=False)].topic150.value_counts(dropna=False).head(5).to_dict())
print("TransSci 'books of interest':", F1[t.str.contains("books? of interest", case=False)].topic150.value_counts(dropna=False).head(5).to_dict())

토픽 번호 범위: 0.0 149.0 | 서로 다른 값 150
TransSci 'Bibliographic Section': {49.0: 20, nan: 14, 70.0: 1}
VT Mag [Mobile Radio]: {38.0: 50, nan: 14, 59.0: 1, 125.0: 1, 44.0: 1}
TransSci 'books of interest': {49.0: 16, nan: 8}


In [ ]:
# ## 판독

# 번호 체계는 맞습니다. 토픽 번호는 0~149로 150개 값이고, VT Mag 칼럼이 38번에 몰려 있어서(50편, 결측 14편 제외하면 거의 전부) 계획서의 "VT Mag 38번 칼럼"과 일치합니다. 번호가 어긋났을 가능성은 낮아졌습니다. 출력이 상위 5개 값만 보여 주므로 합계는 일부 잘렸을 수 있습니다.

# 달라진 점은 두 가지입니다.

# - **서지·도서 안내 항목은 18번이 아니라 49번에 몰려 있습니다.** TransSci "Bibliographic Section"은 49번 20편, "Books of Interest"는 49번 16편입니다. 계획서의 "주제 18번 제외"가 이 군집을 겨눈 것이라면 번호가 49인 것으로 보이지만, 이것은 추정입니다. 18번을 49번으로 오기한 것인지, 다른 군집을 가리킨 것인지는 이 출력으로 알 수 없습니다.
# - **18번이 1편뿐인 것은 그 자체로 이상합니다.** 150개 군집이 평균 약 590편(F1 토픽 보유 88,164편 ÷ 150)인데 한 군집만 1편입니다. 18번 논문이 상류에서 이미 제거됐고 1편이 남았다는 설명이 가능하지만, 가설일 뿐입니다.

# 군집 수가 고정이라면, 규모가 이렇게 작은 군집이 하나 있는 것은 같은 파일의 토픽 번호가 한 번 재할당됐을 가능성도 시사합니다. 이것도 확인된 것이 아닙니다.

# ## 부수적으로 보이는 것

# 비연구 항목은 토픽 결측률이 높습니다. VT Mag 칼럼은 67편 중 14편(약 21%), "Bibliographic Section"은 35편 중 14편(40%), "Books of Interest"는 24편 중 8편(33%)이 결측입니다. 전체 결측률 21%와 비교하면 대체로 비슷하거나 높습니다. `missing` 단계의 성향 모형은 저널과 연도로 이를 일부 보정하겠지만, 이 항목들이 몰린 TransSci·VT Mag에서는 결측이 연구 논문의 결측과 다른 기제일 수 있습니다.

# ## 지금 할 일

# 규칙은 바꾸지 마세요. `f_topic18`은 연구 논문 1편을 표시할 뿐이라 해가 없고, 49번으로 바꾸는 것은 동결된 부록(`topic_excluded=[18]`)의 사후 변경이기 때문입니다.

# 49번을 규칙에 넣을지는 구성을 보고 정해야 합니다. 서지 안내 항목이 몰려 있어도 49번이 다른 연구 논문까지 섞인 큰 군집이라면, 군집 전체를 제외하는 것은 연구 논문을 잃는 일입니다. 아래 스니펫으로 18, 38, 49번의 규모와 구성을 확인하세요.

# ```python
# # 토픽 18, 38, 49의 규모와 구성 (읽기 전용)
# import pandas as pd
# F1 = pd.read_csv(m.DATA / "F1_papers.csv", low_memory=False)
# sz = F1.topic150.value_counts().sort_values()
# print("가장 작은 군집 5개:", sz.head(5).to_dict(), "| 중앙값", sz.median())
# for k in [18, 38, 49]:
#     d = F1[F1.topic150 == k]
#     print(f"\n[토픽 {k}] {len(d):,}편 | 저널 상위 {d.venue_short.value_counts().head(5).to_dict()} | 연도 {d.year.min():.0f}~{d.year.max():.0f}")
#     print(d.title.dropna().sample(min(8, len(d)), random_state=m.SEED).tolist())
# ```

# 판독은 이렇게 하세요.
# - **49번이 거의 전부 서지·도서 항목이면** 49번을 `topic_excluded`에 넣는 것이 타당합니다. 이때는 새 부록 파일로 분리하고, 결과를 본 뒤의 변경이라는 점을 CHANGELOG_S2에 명시하세요.
# - **49번이 연구 논문을 많이 포함하면** 49번은 제외하지 말고, 이 항목들은 S3의 비연구·복합 토픽 탐지(L5)에서 "복합 토픽"으로 표시하세요. 문서 단위 규칙(`f_bib`, `f_repeat`)이 이미 잡고 있는 부분은 그대로 둡니다.
# - **18번이 작은 이유**는 샘플을 봐도 알 수 없을 수 있습니다. 모르면 "원인 미확인, 1편"으로 기록하고 넘어가세요.

# ## 기록할 것

# 앞서 제안한 CHANGELOG_S2 한 줄에 아래를 덧붙이세요.

# > 토픽 번호는 0~149이고 VT Mag 칼럼은 38번, TransSci 서지·도서 안내 항목은 49번에 집중됨. 18번은 F1 전체에서 1편(규모 이상). `topic_excluded=[18]`은 변경하지 않음(사후 변경은 49번 구성 확인 후 판단).

# 다음은 계획대로 `missing`, `risk`, `risk_selection`을 실행하는 것이고, 위 스니펫 결과는 병행해서 붙여 주세요.

In [13]:
# 토픽 18, 38, 49의 규모와 구성 (읽기 전용)
import pandas as pd
F1 = pd.read_csv(m.DATA / "F1_papers.csv", low_memory=False)
sz = F1.topic150.value_counts().sort_values()
print("가장 작은 군집 5개:", sz.head(5).to_dict(), "| 중앙값", sz.median())
for k in [18, 38, 49]:
    d = F1[F1.topic150 == k]
    print(f"\n[토픽 {k}] {len(d):,}편 | 저널 상위 {d.venue_short.value_counts().head(5).to_dict()} | 연도 {d.year.min():.0f}~{d.year.max():.0f}")
    print(d.title.dropna().sample(min(8, len(d)), random_state=m.SEED).tolist())

가장 작은 군집 5개: {18.0: 1, 91.0: 169, 28.0: 194, 58.0: 221, 54.0: 233} | 중앙값 569.5

[토픽 18] 1편 | 저널 상위 {'TRR': 1} | 연도 1974~1974
['WARRANTING FIXED ROADWAY LIGHTING FROM A CONSIDERATION OF DRIVER WORK LOAD']

[토픽 38] 396편 | 저널 상위 {'VT Mag': 265, 'T-ITS': 81, 'TR-C': 7, 'ITS Mag': 6, 'TR-B': 5} | 연도 1990~2025
['Intelligent Reflective Surface Assisted Integrated Sensing and Wireless Power Transfer', 'The Challenges of Universal Broadband Connectivity [Mobile Radio]', 'CONICAL VOLUME-DELAY FUNCTIONS', 'Green Radio Technologies [Mobile Radio]', 'Looking at LTE in Practice: A Performance Analysis of the LTE System Based on Field Test Results', 'Exploring cellular network handover information for urban mobility analysis', 'Characterization of Indoor Reconfigurable Intelligent Surface-Assisted Channels at 304 GHz: Experimental Measurements, Challenges, and Future Directions', 'More Than the Eye Can See: Coherence Time and Coherence Bandwidth of Troposcatter Links for Mobile Receivers']

[토픽 49] 5

In [ ]:
# ## 판독

# 18번은 규칙 대상이 아니고, 49번도 통째로 제외할 군집이 아닙니다. 동결된 `topic_excluded=[18]`은 바꾸지 않는 쪽이 맞습니다.

# - **18번**: 1편뿐이고 1974년 TRR의 연구 논문입니다. 다음으로 작은 군집이 169편이니 18번만 규모가 비정상입니다. 원인은 이 출력으로는 알 수 없으니 "원인 미확인, 1편"으로 기록하세요.
# - **49번**: 582편 중 8편 표본이 서지 안내 하나, 과학계량 가이드, 연구실 소개, 체계적 문헌 고찰, 교통류 이론 논문이 섞인 구성입니다. 서지·도서 안내 항목(약 36편, 앞의 49번 집계 합)은 전체의 6% 수준이라 49번을 제외하면 연구 논문 500편 이상을 잃습니다. 표본이 8편이라 구성 비율은 대략적인 인상일 뿐이지만, "비연구 군집"이라는 가설은 이 표본과 맞지 않습니다. 오히려 리뷰, 방법론, 편집 문서가 모인 복합 군집으로 보입니다.
# - **38번**: 396편 중 VT Mag이 265편입니다. 무선통신 연구(RIS, LTE, 6G 계열)와 `[Mobile Radio]` 칼럼이 같은 군집에 있어, 38번도 통째로 제외할 수 없습니다. 이 칼럼들은 `f_vtmr` 규칙(68건)이 문서 단위로 이미 표시하고 있습니다. 샘플에 `CONICAL VOLUME-DELAY FUNCTIONS`(교통류 논문) 같은 이질적인 항목도 있는데, 이것이 38번의 잡음인지는 표본 하나로는 판단할 수 없습니다.

# ## 조치

# 1. **규칙은 바꾸지 마세요.** 동결된 `f_bib`, `f_vtmr`, `f_repeat`, `f_dup`, `f_topic18`이 그대로 문서 단위로 표시합니다.
# 2. **49번과 38번은 S3의 L5(복합·비연구 토픽 탐지)에서 "복합 토픽"으로 표시할 후보로 메모해 두세요.** 이 단계에서 군집 단위의 처리를 정하게 됩니다. 지금 정하면 결과를 본 뒤의 변경이 됩니다.
# 3. **CHANGELOG_S2에는 한 줄로 남기세요.**

# > 토픽 38(396편, VT Mag 265)은 무선통신 연구와 [Mobile Radio] 칼럼이 혼재, 토픽 49(582편)는 리뷰·방법론·편집 문서 등 복합 군집으로 보임(표본 8편 기준). 군집 단위 제외는 하지 않음. 18번은 F1 전체 1편(원인 미확인). `topic_excluded=[18]` 변경 없음, 49·38번은 S3 L5의 복합 토픽 후보.

# ## 다음 단계

# 이제 `s2_measurement.py`의 `STEPS`를 `["missing"]`으로 바꿔 실행하세요. 출력이 길 수 있으니 모형 A/B의 AUC, 유효 표본 비율, 양수성 위반(성향 0.05 미만 비중)과 Scimago 대조 부분을 특히 보겠습니다. 이어서 `["risk"]`, `["risk_selection"]` 순서입니다.

In [ ]:
# `s2_run_steps.py`를 만들었습니다. 문법 검사나 가짜 자료 시험은 하지 못했고, 실제 자료로도 아직 돌려 보지 못했습니다. `s2_measurement.py`(v3)와 같은 폴더나 `~/Research/Minnesota_Transport`, `~/Downloads`에 두세요.

# ## 실행 순서

# `STEPS`를 아래처럼 한 번에 하나씩 바꿔 실행하고, 매번 출력 전체를 붙여 주세요.

# 1. `STEPS = ["missing"]`
# 2. `STEPS = ["risk"]`
# 3. `STEPS = ["risk_selection"]`

# 세 단계를 한꺼번에 넣어도(`["missing", "risk", "risk_selection"]`) 순서대로 돌고, 한 단계에서 멈추거나 오류가 나면 거기서 중단합니다. 다만 단계마다 판독할 것이 달라서 하나씩 보는 것을 권합니다.

# ## 단계별로 볼 것

# - **`missing`**:
#   - 모형 A/B의 AUC(CV 포함)와 유효 표본 비율
#   - 성향 0.05 미만인 논문의 비중과 해당 저널(양수성 위반)
#   - Scimago 비와 토픽 보유율의 Spearman
#   - 저널·연도별 토픽 보유율의 낮은 곳
# - **`risk`**: F7의 연도 분포(평가 범위), 전체 이름 기준과 "F7 행 2개 이상 이름" 기준의 precision/recall. 근사 정답(이름당 OpenAlex ID 2개 이상)의 한계 때문에 두 기준을 같이 읽어야 합니다.
# - **`risk_selection`**: F7과 F2의 정렬 가정 검증(이름 일치율)과 F2 행 수 정합이 FAIL이면 그 뒤 국가별 제외 비중은 신뢰할 수 없습니다. PASS일 때만 제외가 국가 구성을 바꾸는지(국가별 제외 비중, 몫 차이)를 읽겠습니다.

# ## 알아 둘 점

# - `risk_selection`은 F2와 F7을 통째로 읽어서 메모리와 시간을 가장 많이 씁니다. 그 전에 `missing`과 `risk`가 끝났는지 먼저 확인하세요.
# - 각 단계는 동결된 부록의 해시를 확인하고 부록을 바꾸지 않습니다. 산출물은 `paper1/02_S2/`에 저장되고, CHANGELOG_S2에 실행 기록이 한 줄씩 추가됩니다.
# - `risk_selection`의 "F7의 `rank`와 F2의 `author_pos - 1`이 같다"는 정렬 가정은 이 단계 안에서 이름 일치율로 검증됩니다.

In [14]:
# -*- coding: utf-8 -*-
# =====================================================================================
# S2 단계 순차 실행 셀 (s2_measurement.py v3 필요, OpenAlex 호출 없음)
# 사용법: STEPS를 아래 순서대로 한 번에 하나씩 바꿔 실행하고, 매번 출력 전체를 붙여 주세요.
#   1) STEPS = ["missing"]         토픽 결측 분해 + 성향 가중 (Scimago 분모 대조 포함)
#   2) STEPS = ["risk"]            위험군 플래그 precision/recall
#   3) STEPS = ["risk_selection"]  위험군 제외의 국가 선택 효과
#   (["missing", "risk", "risk_selection"]처럼 여러 개를 넣으면 순서대로 돌고, 한 단계에서 멈추거나 오류가 나면 거기서 중단)
# 이 단계들은 first_last 가중치를 쓰지 않으므로 알파벳순 보정이 필요 없습니다.
# 동결 부록(S2_PREREG_ADDENDUM.json)의 해시를 각 단계가 확인하며, 부록을 바꾸지 않습니다.
# =====================================================================================
import importlib.util, traceback, time
from pathlib import Path

STEPS = ["missing", "risk", "risk_selection"]

ROOT = Path.home() / "Research/Minnesota_Transport"
cands = [Path.cwd() / "s2_measurement.py", ROOT / "s2_measurement.py", ROOT / "paper1/s2_measurement.py",
         ROOT / "paper1/02_S2/s2_measurement.py", Path.home() / "Downloads/s2_measurement.py"]
MOD = next((c for c in cands if c.exists()), None)
assert MOD is not None, f"s2_measurement.py를 찾지 못했습니다. 후보: {[str(c) for c in cands]}"
spec = importlib.util.spec_from_file_location("s2", MOD); m = importlib.util.module_from_spec(spec); spec.loader.exec_module(m)
ALLOWED = ["missing", "risk", "risk_selection"]
bad = [s for s in STEPS if s not in ALLOWED]; assert not bad, f"이 셀에서 실행하지 않는 단계: {bad} (허용: {ALLOWED})"
assert all(s in m.FUNCS for s in STEPS), "구버전 s2_measurement.py입니다. v3(최신)를 사용하세요"
print(f"모듈: {MOD} | STEPS={STEPS}")
m.S2.mkdir(parents=True, exist_ok=True)

for s in STEPS:
    t0 = time.time()
    try: m.FUNCS[s]()
    except m.Halt as e: print(f"\n⛔ {s} 중단: {e}"); break
    except Exception: print(f"\n⛔ {s} 오류(아래 전체 출력을 붙여 주세요):"); traceback.print_exc(); break
    print(f"\n✔ {s} 완료 ({time.time() - t0:.0f}초)")
else:
    print("\n✔ 요청한 단계 모두 완료")

모듈: /home/yjlee/Research/Minnesota_Transport/paper1/s2_measurement.py | STEPS=['missing', 'risk', 'risk_selection']

[missing] 토픽 결측 분해 + 성향 가중
토픽 파일 F1_papers.csv, 열 topic150: 값 분포 상위 {nan: 23521, 137.0: 1246, 72.0: 1075, 147.0: 1028, 75.0: 1021, 4.0: 984, 52.0: 973, 22.0: 969}
F1X 111,416편 중 토픽 보유 79.1% (계획서 EDA: 결측 약 21%와 대조)
연도별 토픽 보유율: {2000: 0.698, 2001: 0.722, 2002: 0.731, 2003: 0.741, 2004: 0.758, 2005: 0.8, 2006: 0.823, 2007: 0.757, 2008: 0.772, 2009: 0.792, 2010: 0.806, 2011: 0.831, 2012: 0.824, 2013: 0.84, 2014: 0.842, 2015: 0.863, 2016: 0.866, 2017: 0.868, 2018: 0.871, 2019: 0.882, 2020: 0.866, 2021: 0.867, 2022: 0.87, 2023: 0.87, 2024: 0.848, 2025: 0.811}
저널별(하위 8):
   level    n  has_topic_rate   dim
 VT Mag  960        0.528125 venue
   RTBM 1236        0.682848 venue
  CACIE 1960        0.699490 venue
   GEIT  223        0.704036 venue
    JPT  618        0.711974 venue
 OJ-ITS  362        0.712707 venue
   TRIP 1633        0.729333 venue
RailEng  204        0.754902 ve

In [15]:
#

In [ ]:
# ## 총평

# 세 단계 모두 오류 없이 끝났고, 가장 중요한 결과는 `risk_selection`에서 나왔습니다. 위험군 제외가 중국 소속 저자를 크게 더 빼고 있습니다. 반면 정합 점검이 1행 차이로 FAIL이라 먼저 원인을 찾아야 하고, `missing`에서는 2007년의 토픽 보유율 하락을 확인해야 합니다.

# ## 1. missing

# - **성향 가중은 약하게 작동합니다.** 모형 A와 B의 AUC가 0.726/0.731이고 교차검증 값이 0.724/0.729로 거의 같아 과적합은 없습니다. 유효 표본이 97~98%이고 절단 구간이 [0.87, 1.88]이며 성향 0.05 미만 논문이 0%입니다. 가중치가 작아서 가중과 무가중 결과의 차이도 작을 가능성이 높지만, 실제 차이는 다음 단계에서 비교해야 압니다.
# - **이 가중이 보정하는 것은 연도·저널·저자 수·국가로 설명되는 결측뿐입니다.** 결측이 초록이나 임베딩 유무처럼 관측되지 않은 요인에서 온다면 보정되지 않습니다. 이 출력으로는 그 기제를 알 수 없습니다.
# - **Scopus 대조에서 연관이 보이지 않았습니다.** 468개 칸에서 코퍼스 대 Scopus 비와 토픽 보유율의 Spearman이 -0.01이고, 표시 칸별 토픽 보유율도 high 0.814, low 0.797, ok 0.816으로 거의 같습니다. 논문 자체의 누락과 토픽만의 결측이 칸 수준에서는 서로 무관해 보입니다(문서 20편 미만 칸은 제외된 결과).
# - **2007년에 토픽 보유율이 갑자기 떨어집니다.** 2006년 0.823에서 2007년 0.757로 내려갔다가 다시 올라갑니다. 2007년은 국가 층의 창 시작 연도라서 저널 구성이 바뀌었거나 수집 방식이 달라졌을 가능성이 있습니다. 어느 쪽인지는 이 출력으로는 알 수 없고, 아래 스니펫으로 확인하세요.
# - **저널별로는 VT Mag이 0.53으로 가장 낮고** 다음이 RTBM 0.68, CACIE 0.70입니다. VT Mag은 앞서 본 `[Mobile Radio]` 칼럼이 몰린 저널입니다.
# - 2024년(0.848)과 2025년(0.811)도 하락이 이어집니다. 2025년이 약한 해라는 점은 다른 점검과 같은 방향입니다.

# ## 2. risk

# | 평가 대상 | 플래그 | 정밀도 | 재현율 |
# |---|---|---|---|
# | 전체 이름 | ambiguous | 0.866 | 0.290 |
# | 전체 이름 | heavy | 0.573 | 0.031 |
# | 전체 이름 | 합집합 | 0.814 | 0.298 |
# | 행 2개 이상 이름 | ambiguous | 0.907 | 0.290 |
# | 행 2개 이상 이름 | 합집합 | 0.849 | 0.298 |

# - **평가 범위는 2020~2025년 authorship뿐입니다.** F7 행이 모두 2020년대라서, 이 재현율은 그 기간에 대한 값입니다.
# - **재현율이 약 30%입니다.** 근사 정답(이름당 OpenAlex ID 2개 이상)인 3,749개 이름 중 2,633개가 플래그에 걸리지 않았습니다. 이 이름들은 F2에 남아 있어, 이름 기반 저자 식별에 동명이인이 섞였을 가능성이 있습니다. 다만 OpenAlex가 한 사람을 ID 둘로 쪼갠 경우도 정답으로 잡히므로 실제 동명이인 재현율은 이보다 높을 수도 있고, 반대로 OpenAlex가 동명이인을 하나의 ID로 합친 경우는 정답에서 빠져 낮을 수도 있습니다. 그래서 단일 숫자가 아니라 범위로 보고해야 합니다.
# - **정밀도는 부풀려졌을 수 있습니다.** 행이 1개뿐인 이름은 ID가 둘일 수 없고, 흔한 이름일수록 행이 많아 ID가 갈라질 기회도 많습니다. 위험군 플래그가 어떻게 만들어졌는지는 이 출력으로 알 수 없습니다. ID 수 같은 정보를 쓰고 만들었다면 이 평가는 순환적입니다.
# - **`heavy`는 ID 분리 탐지기로는 약합니다.** 합집합이 ambiguous만 쓸 때보다 이름 116개를 더 플래그하는데 정답은 29개 늘 뿐이라, 추가분의 정밀도가 약 25%입니다(제 계산). `heavy`가 원래 다른 목적(예: 다작 저자)으로 만들어졌는지는 모릅니다. 동결된 위험군 변형 7종별 분해는 변형 정의 열이 없어 하지 않았습니다.

# ## 3. risk_selection

# **정렬 가정은 통과했습니다.** 매칭 108,370행에서 이름 일치율이 100%입니다.

# **정합 점검은 FAIL입니다.** F2 행이 234,060개인데 `authorship_level`과 맞춘 행이 234,059개로 1행 차이입니다. 규모는 무시할 만하지만, 기준이 정확한 일치였으므로 원인을 찾아 기록해야 합니다. 아래 스니펫으로 그 한 행을 찾을 수 있습니다.

# **위험군 제외는 국가별로 크게 다릅니다.** 제외된 authorship 비중은 12.0%이고, 국가군별로는 다음과 같습니다.

# | 국가군 | 제외 비중 |
# |---|---|
# | CN | 23.4% |
# | AU | 11.8% |
# | NL | 10.0% |
# | CA | 9.5% |
# | US | 8.4% |
# | GB | 6.5% |
# | UNK, other | 6.1% |
# | DE | 3.5% |

# 중국 소속 저자가 미국의 약 2.8배 비율로 빠집니다. 흔한 성이 위험군에 많이 걸리기 때문일 수 있지만, 이 출력으로 확인된 것은 아닙니다.

# **논문 단위 국가 몫은 거의 변하지 않았습니다.** `equal`(제외 후 재정규화)은 전체 authorship(S1 방식)과 사실상 같습니다. CN 2020~25년이 0.352 대 0.352이고, 2014~19년이 0.133 대 0.135입니다. 한 논문 안에서 중국 저자 일부가 빠져도 남은 저자의 국가가 같으면 논문 질량이 유지되기 때문입니다. 따라서 국가 층(S7)이 제외 없는 값을 주 분석으로 쓰는 설계는 이 점검과 모순되지 않습니다.

# **저자 수 기준 구성은 달라집니다.** `inv_n`(제외 질량 손실)에서는 CN 2020~25년이 0.352에서 0.314로 3.8%p 낮아지고 US는 0.205에서 0.214로 높아집니다. 출력된 표로 제가 계산하면, 창 안 주제 논문의 authorship 중 중국 소속 비중이 약 28.4%에서 제외 후 약 24.7%로 줄어듭니다(첫 국가 기준, 표의 반올림 때문에 근사치). 즉 저자 층 결과(S6, S7.2)에서 "신규 저자의 지역 구성"을 저자 수로 세면 중국 비중이 낮게 나올 수 있습니다. 이것은 한계로 명시하고, 그 크기를 이 표에서 가져다 쓰세요.

# 제외된 authorship 12.0%가 앞서 본 행 보존율 89.4%(손실 10.6%)와 다른 것은 창이 2007년 이후이기 때문일 수 있지만, 확인하지는 않았습니다.

# ## 지금 할 일

# 아래 스니펫은 읽기 전용입니다. 앞의 두 확인을 한 번에 합니다.

# ```python
# import json, pandas as pd
# import importlib.util
# from pathlib import Path
# ROOT = Path.home() / "Research/Minnesota_Transport"
# sp = importlib.util.spec_from_file_location("s2", ROOT / "paper1/s2_measurement.py"); m = importlib.util.module_from_spec(sp); sp.loader.exec_module(m)

# # (1) 정합 FAIL 1행 찾기
# F = m.load_F1X(); A = m.build_authorship(F)
# F2 = pd.read_csv(m.DATA / "F2_author_paper.csv", usecols=["pid", "name_norm", "author_pos"]); F2["pid"] = F2.pid.astype(str); F2["rank"] = F2.author_pos - 1
# x = F2[F2.pid.isin(set(A.pid))].merge(A[["pid", "rank"]].assign(inA=True), on=["pid", "rank"], how="left")
# print("F2에 있으나 authorship에 없는 (pid, rank):\n", x[x.inA.isna()])
# print("중복 (pid, rank) | F2:", int(F2.duplicated(["pid", "rank"]).sum()), "| authorship:", int(A.duplicated(["pid", "rank"]).sum()))

# # (2) 2007년 토픽 보유율 하락: 고정 저널 대 그 외, 2007년 주요 저널
# G = m.load_has_topic(F); G = G[G.year.between(2004, 2010)]
# fixed = json.loads((m.S0 / "PREREG.json").read_text(encoding="utf-8"))["event"]["fixed_venues"]
# print(G.assign(fixed=G.venue_short.isin(fixed)).groupby(["year", "fixed"]).has_topic.agg(["size", "mean"]).round(3).unstack())
# c = G.groupby(["year", "venue_short"]).has_topic.agg(["size", "mean"]).round(3).reset_index()
# print(c[c.year.isin([2006, 2007])].pivot(index="venue_short", columns="year", values=["size", "mean"]).sort_values(("size", 2007), ascending=False).head(10))
# ```

# - 한 행이 나오면 그 pid가 어떤 논문인지(저자 수 불일치 등) 보면 됩니다. 중복이 보이면 그쪽이 원인입니다.
# - 고정 저널에서도 2007년에 떨어지면 수집 방식 쪽, 비고정 저널에서만 떨어지면 저널 구성 쪽(낮은 보유율의 저널이 2007년에 진입)일 가능성이 높습니다.

# ## 그다음 순서

# 1. 위 스니펫 출력을 붙여 주세요.
# 2. CHANGELOG_S2에는 정합 FAIL 1행의 원인(또는 "원인 미확인")과 `risk`의 평가 범위(2020~25년)를 한 줄씩 남기세요.
# 3. `commit` 단계를 `COMMIT=False`로 먼저 돌려 매니페스트를 확인한 뒤 `True`로 태그를 남깁니다. 다만 위 두 확인이 끝난 뒤에 하세요.
# 4. 병행 항목으로 `clean_review_samples.csv`의 `correct` 열, `unk_resid_2024_25_sample.csv`, F4/F5가 `fixed_venue`로 만들어졌는지 확인이 남아 있습니다.
# 5. 저자 층(포트폴리오, 잡음 대응)은 아직 코드가 없습니다. F3, F4, F5의 열 목록(이름, dtype, 결측률)을 붙여 주시면 그에 맞춰 만들겠습니다.

In [16]:
import json, pandas as pd
import importlib.util
from pathlib import Path
ROOT = Path.home() / "Research/Minnesota_Transport"
sp = importlib.util.spec_from_file_location("s2", ROOT / "paper1/s2_measurement.py"); m = importlib.util.module_from_spec(sp); sp.loader.exec_module(m)

# (1) 정합 FAIL 1행 찾기
F = m.load_F1X(); A = m.build_authorship(F)
F2 = pd.read_csv(m.DATA / "F2_author_paper.csv", usecols=["pid", "name_norm", "author_pos"]); F2["pid"] = F2.pid.astype(str); F2["rank"] = F2.author_pos - 1
x = F2[F2.pid.isin(set(A.pid))].merge(A[["pid", "rank"]].assign(inA=True), on=["pid", "rank"], how="left")
print("F2에 있으나 authorship에 없는 (pid, rank):\n", x[x.inA.isna()])
print("중복 (pid, rank) | F2:", int(F2.duplicated(["pid", "rank"]).sum()), "| authorship:", int(A.duplicated(["pid", "rank"]).sum()))

# (2) 2007년 토픽 보유율 하락: 고정 저널 대 그 외, 2007년 주요 저널
G = m.load_has_topic(F); G = G[G.year.between(2004, 2010)]
fixed = json.loads((m.S0 / "PREREG.json").read_text(encoding="utf-8"))["event"]["fixed_venues"]
print(G.assign(fixed=G.venue_short.isin(fixed)).groupby(["year", "fixed"]).has_topic.agg(["size", "mean"]).round(3).unstack())
c = G.groupby(["year", "venue_short"]).has_topic.agg(["size", "mean"]).round(3).reset_index()
print(c[c.year.isin([2006, 2007])].pivot(index="venue_short", columns="year", values=["size", "mean"]).sort_values(("size", 2007), ascending=False).head(10))

F2에 있으나 authorship에 없는 (pid, rank):
           pid   name_norm  author_pos  rank  inA
154139  57786  ann jopson           4     3  NaN
중복 (pid, rank) | F2: 0 | authorship: 0
토픽 파일 F1_papers.csv, 열 topic150: 값 분포 상위 {nan: 23521, 137.0: 1246, 72.0: 1075, 147.0: 1028, 75.0: 1021, 4.0: 984, 52.0: 973, 22.0: 969}
F1X 111,416편 중 토픽 보유 79.1% (계획서 EDA: 결측 약 21%와 대조)
       size         mean       
fixed False True   False  True 
year                           
2004     20  1208  0.850  0.757
2005     27  1867  0.593  0.803
2006     71  1900  0.423  0.838
2007     83  1597  0.506  0.770
2008     72  1596  0.472  0.786
2009    139  1736  0.568  0.810
2010    123  2174  0.561  0.820
               size          mean       
year           2006   2007   2006   2007
venue_short                             
TRR          1194.0  829.0  0.899  0.825
AAP           174.0  192.0  0.736  0.698
T-ITS          51.0   73.0  0.667  0.712
TR-D           36.0   59.0  0.833  0.780
TR-B           49.0   59.0  0.77

In [ ]:
# ## 판독

# 두 확인 모두 영향이 작거나 설명 가능한 범위입니다. 다만 2007년 하락은 원인이 아직 확정되지 않았고, 성향 모형이 그 하락을 못 잡는다는 점은 지금 기록해 두어야 합니다.

# ## 1. 정합 FAIL 1행

# F2에 있는 `pid 57786`의 4번째 저자(`ann jopson`, `rank` 3)가 `authorship_level`에 없습니다. 중복은 양쪽 모두 0이라, 중복이 아니라 그 논문의 `authorship_level`에 해당 `rank` 행이 빠져 있는 것입니다. 저자 수가 어긋난 논문일 가능성이 있지만, 이 출력으로는 확인되지 않습니다.

# 영향은 234,060행 중 1행이라 무시할 수 있습니다. `risk_selection`의 `kept` 계산에서 이 행만 빠지는 정도입니다. 정렬 가정은 이름 일치율 100%로 이미 지지됐으므로 FAIL은 "원인 일부 확인, 영향 무시 가능"으로 기록하면 됩니다.

# ## 2. 2007년 토픽 보유율 하락

# - **하락은 고정 저널에서 일어납니다.** 고정 저널은 2006년 0.838에서 2007년 0.770으로 떨어졌습니다. 비고정 저널은 0.423에서 0.506으로 오히려 올랐고 논문 수도 71~83편으로 적어서 하락을 설명하지 못합니다.
# - **하락의 대부분은 TRR입니다.** TRR이 0.899에서 0.825로 떨어졌고 논문 수도 1,194편에서 829편으로 줄었습니다.
# - **구성 효과와 저널 내 변화를 대략 나눠 보면** 고정 저널 하락 0.068 중 구성(TRR 비중 63%에서 52%)이 약 0.018, 저널 내 변화가 약 0.050입니다. 출력에 있는 반올림된 값으로 제가 손으로 계산한 근사치입니다.
# - **다른 저널은 엇갈립니다.** AAP, TR-B, TR-E는 내려가고 T-ITS, TR-C는 올라갑니다. TRR 외에는 한 방향의 패턴이 보이지 않습니다.
# - **원인은 모릅니다.** 토픽이 임베딩에서 나왔다면 초록이나 임베딩 유무와 관련이 있을 수 있지만, 이 출력으로는 알 수 없는 가설입니다.

# ## 3. 이 결과가 걸리는 곳

# 1. **성향 모형이 이 하락을 못 잡습니다.** 모형 A와 B의 공변량은 연도, 저널, 저자 수 구간의 주효과만 있고 저널×연도 상호작용이 없습니다. 그래서 "TRR 2007"처럼 특정 저널·연도 칸에서만 일어난 결측은 보정되지 않습니다. AUC 0.73이라는 값과도 맞습니다.
# 2. **S5의 변화점 탐지에 2007년 인공 신호가 생길 수 있습니다.** 주제 층은 2000년부터인데, 2007년에 결측 구조가 바뀌고 주제 구성이 그 영향을 받으면 변화점이 이 연도에 잡힐 수 있습니다. 계획서 5.5(3)의 "임베딩 보유율 변화 같은 측정 아티팩트 표시"에 2007을 후보로 올려 두세요. 이 단계에서 이미 알고 있는 사실이므로 사전에 표시하는 것은 결과 보기 전의 조치입니다.
# 3. **상호작용을 넣은 성향 모형은 민감도 변형으로만 추가하세요.** 동결된 부록의 주 모형은 바꾸지 않습니다. 결과를 본 뒤 추가하는 것이므로 CHANGELOG_S2에 "동결 후 추가한 탐색적 민감도"로 기록해야 합니다.

# ## 지금 할 일

# 아래 스니펫은 읽기 전용입니다. TRR의 2006~2007년 사이에 어떤 열이 달라졌는지 봅니다.

# ```python
# # (1) 정합 FAIL 1행의 논문
# F = m.load_F1X(); A = m.build_authorship(F)
# print(F[F.pid == "57786"][["pid", "year", "venue_short", "n_authors", "title"]].to_string(index=False))
# print(A[A.pid == "57786"][["pid", "rank", "n"]].to_string(index=False))

# # (2) TRR 2005~2008: 열별 결측률(초록·DOI 등 토픽 결측과 관련될 만한 열 찾기)
# T = pd.read_csv(m.DATA / "F1_papers.csv", low_memory=False); T = T[(T.venue_short == "TRR") & T.year.between(2005, 2008)]
# nr = T.groupby("year").apply(lambda d: d.isna().mean()).T.round(3)
# print(nr[(nr.max(axis=1) - nr.min(axis=1)) > 0.03].to_string())   # 연도 사이에 3%p 넘게 변한 열만
# print(T.groupby("year").size().to_dict())
# ```

# - 두 번째 출력에서 `topic150`의 결측률과 함께 크게 달라지는 열이 있으면 그 열이 결측 기제의 후보입니다.
# - 그런 열이 없으면 "원인 미확인"으로 기록하고 넘어가세요.

# ## 기록할 것

# CHANGELOG_S2에 남길 줄은 아래와 같습니다.

# > 정합 FAIL: F2 234,060행 중 1행(pid 57786, rank 3)이 authorship_level에 없음, 중복 0, 영향 무시 가능. risk 평가 범위는 2020~25년(F7). 토픽 보유율은 2006→2007년 고정 저널에서 0.838→0.770로 하락(TRR 0.899→0.825가 주도, 구성 효과는 일부), 성향 모형(저널×연도 상호작용 없음)이 이를 보정하지 못함. 2007년을 S5의 측정 아티팩트 표시 후보로 지정.

# ## 다음 순서

# 1. 위 스니펫 출력을 붙여 주세요.
# 2. 그 뒤 `commit`을 `COMMIT=False`로 먼저 돌려 매니페스트를 확인하고 `True`로 태그를 남기세요. 태그는 `S2-stageAB-<날짜>` 형식으로 자동 생성됩니다. 50MB 넘는 파일(`paper_ipw.csv.gz`, `country_mass_long`) 경고가 나오면 커밋 용량을 확인하세요.
# 3. 저자 층 단계(포트폴리오, 잡음 대응, 제목 분류기, K 민감도)는 F3, F4, F5의 열 목록(이름, dtype, 결측률)을 받은 뒤에 만들겠습니다.

In [17]:
# (1) 정합 FAIL 1행의 논문
F = m.load_F1X(); A = m.build_authorship(F)
print(F[F.pid == "57786"][["pid", "year", "venue_short", "n_authors", "title"]].to_string(index=False))
print(A[A.pid == "57786"][["pid", "rank", "n"]].to_string(index=False))

# (2) TRR 2005~2008: 열별 결측률(초록·DOI 등 토픽 결측과 관련될 만한 열 찾기)
T = pd.read_csv(m.DATA / "F1_papers.csv", low_memory=False); T = T[(T.venue_short == "TRR") & T.year.between(2005, 2008)]
nr = T.groupby("year").apply(lambda d: d.isna().mean()).T.round(3)
print(nr[(nr.max(axis=1) - nr.min(axis=1)) > 0.03].to_string())   # 연도 사이에 3%p 넘게 변한 열만
print(T.groupby("year").size().to_dict())

  pid  year venue_short  n_authors                                                                                                                                          title
57786  2018        TR-F          4 Exploring the influence of attitudes, social comparison and image and prestige among non-cyclists to predict intention to cycle in Mexico City
  pid  rank  n
57786     0  3
57786     1  3
57786     2  3
year       2005   2006   2007   2008
topic150  0.139  0.104  0.177  0.145
{2005: 1186, 2006: 1199, 2007: 831, 2008: 709}


In [18]:
#

In [ ]:
# ## 판독

# 두 확인 모두 영향은 작지만, 첫 번째에서 앞서 안내한 가정 하나가 틀렸다는 점이 드러났고 두 번째는 원인을 못 찾았습니다.

# ## 1. 정합 FAIL 1행: 저자 수 불일치입니다

# `pid 57786`은 F1에서 `n_authors = 4`인데 `authorship_level`에는 3행(`rank` 0~2)만 있습니다. F2는 F1 기준으로 4명을 가지고 있어서 4번째 저자만 맞출 상대가 없었던 것입니다. 중복이나 정렬 오류가 아니라, OpenAlex에서 가져온 저자 수가 F1과 다른 논문입니다. 계획서 1.2의 "저자 수가 일치하지 않는 논문"에 해당하는 경우입니다.

# 이 1행 자체는 무시할 만하지만, 같은 유형이 얼마나 있는지는 이 출력으로 알 수 없습니다. 앞서 정합 점검이 1행만 어긋난 것은 F2와 맞춘 논문 범위에서의 결과이고, 창 안 전체 논문의 불일치율은 별개입니다.

# 더 걸리는 점은 가중치 정의입니다. 국가 층의 `author_w`는 `authorship_level`의 저자 수(`A.n`, 이 논문에서는 3)로 1/n을 계산하는데, 부록의 `inv_n_authors` 정의는 "F1의 원래 `n_authors`"(이 논문에서는 4)입니다. 이런 논문이 있으면 국가 층에서 `equal`과 `inv_n_authors`가 같은 값이라는 부록의 가정이 그 논문에서는 성립하지 않습니다. 불변식(논문당 합이 1)은 `A.n`을 쓰므로 계속 PASS이고, 그래서 지금까지 드러나지 않았습니다. 규모를 아래 스니펫으로 확인하세요.

# ```python
# # authorship_level 저자 수 vs F1 n_authors 불일치 규모 (창 안, 읽기 전용)
# F = m.load_F1X(); A = m.build_authorship(F); D = F[F.year >= m.WINDOW].set_index("pid")
# x = D.n_authors.to_frame("n_f1").join(A.groupby("pid").size().rename("n_auth"), how="inner")
# x["diff"] = x.n_auth - x.n_f1
# print(f"창 안 논문 {len(x):,}편 | 저자 수 불일치 {int((x['diff'] != 0).sum()):,}편 ({(x['diff'] != 0).mean():.2%}) | authorship이 더 적음 {int((x['diff'] < 0).sum()):,} / 더 많음 {int((x['diff'] > 0).sum()):,}")
# print("연도별 불일치율:", x.join(D.year).groupby("year")["diff"].apply(lambda s: round((s != 0).mean(), 4)).to_dict())
# ```

# - 불일치율이 1% 미만이면 부록에 한계로 한 줄 적고 넘어가도 됩니다. `equal`과 `inv_n_authors`가 같다는 문구는 "저자 수가 일치하는 논문에서"로 좁혀 쓰세요.
# - 높거나 연도에 따라 몰려 있으면 정의를 어느 쪽으로 정할지 결정해야 합니다. 동결된 정의의 사후 변경이므로 새 부록 파일로 분리하고 CHANGELOG_S2에 사유를 쓰세요.

# ## 2. 2007년 하락: 원인을 찾지 못했습니다

# - TRR 2005~2008년에서 연도 사이에 3%p를 넘게 변한 열은 `topic150` 하나뿐입니다. 결측률은 0.139, 0.104, 0.177, 0.145입니다. 앞서 본 TRR 토픽 보유율(2006년 0.899, 2007년 0.825)과도 일치합니다.
# - 함께 움직이는 다른 열이 없으므로 결측 기제의 후보를 F1의 열에서는 찾지 못했습니다. 이 점검은 결측률만 보므로, 열의 값이 달라진 경우는 잡지 못합니다.
# - 2007년의 결측 상승은 한 해짜리 반등입니다. 2008년에 다시 내려오지만 2006년 수준까지는 아닙니다.
# - TRR의 논문 수가 1,199편(2006)에서 831편(2007), 709편(2008)로 줄었습니다. 앞서 본 Scimago 대조에서 TRR의 코퍼스 대 Scopus 비는 2007년 0.96, 2008년 1.01이므로, 수집 누락이 아니라 실제로 문서 수가 줄어든 것으로 보입니다. 이 해석은 그 비에 근거한 것이고, 줄어든 이유까지는 알 수 없습니다.

# 더 파고들려면 TRR 2006년과 2007년의 토픽 결측 논문이 어떻게 다른지(제목 길이, 저자 수, 인용 수) 보면 되지만, 제가 가진 정보로는 결론이 나올 가능성이 낮습니다. 열 목록을 모르는 상태에서 추측으로 스니펫을 늘리기보다, "원인 미확인"으로 기록하는 쪽을 권합니다.

# ## 기록할 것

# 앞서 제안한 CHANGELOG_S2 줄의 정합 FAIL 부분을 아래로 바꾸세요.

# > 정합 FAIL: F2 234,060행 중 1행(pid 57786, rank 3)이 authorship_level에 없음. 이 논문은 F1 n_authors=4, authorship_level 3행(저자 수 불일치). 중복 없음, 영향 무시 가능. 전체 불일치율은 별도 점검(결과는 이 줄 아래에 추가). TRR 2006→2007년 토픽 결측 상승은 F1의 다른 열(결측률 3%p 초과 변동)과 함께 움직이지 않아 원인 미확인.

# ## 다음 순서

# 1. 위 불일치율 스니펫의 출력을 붙여 주세요.
# 2. 결과에 따라 `inv_n_authors` 문구를 정리한 뒤 `commit`을 `COMMIT=False`로 먼저 돌려 매니페스트를 확인하고, `True`로 태그를 남기세요.
# 3. 저자 층 단계는 F3, F4, F5의 열 목록(이름, dtype, 결측률)을 받은 뒤에 만들겠습니다.

In [19]:
# authorship_level 저자 수 vs F1 n_authors 불일치 규모 (창 안, 읽기 전용)
F = m.load_F1X(); A = m.build_authorship(F); D = F[F.year >= m.WINDOW].set_index("pid")
x = D.n_authors.to_frame("n_f1").join(A.groupby("pid").size().rename("n_auth"), how="inner")
x["diff"] = x.n_auth - x.n_f1
print(f"창 안 논문 {len(x):,}편 | 저자 수 불일치 {int((x['diff'] != 0).sum()):,}편 ({(x['diff'] != 0).mean():.2%}) | authorship이 더 적음 {int((x['diff'] < 0).sum()):,} / 더 많음 {int((x['diff'] > 0).sum()):,}")
print("연도별 불일치율:", x.join(D.year).groupby("year")["diff"].apply(lambda s: round((s != 0).mean(), 4)).to_dict())

창 안 논문 81,441편 | 저자 수 불일치 2편 (0.00%) | authorship이 더 적음 1 / 더 많음 1
연도별 불일치율: {2007: 0.0, 2008: 0.0, 2009: 0.0, 2010: 0.0, 2011: 0.0, 2012: 0.0, 2013: 0.0, 2014: 0.0, 2015: 0.0, 2016: 0.0, 2017: 0.0, 2018: 0.0002, 2019: 0.0, 2020: 0.0, 2021: 0.0, 2022: 0.0002, 2023: 0.0, 2024: 0.0, 2025: 0.0}


In [ ]:
# ## 판독

# 저자 수 불일치는 창 안 81,441편 중 2편(약 0.002%)이고 방향도 반반(authorship이 더 적은 논문 1편, 더 많은 논문 1편)입니다. 연도별로도 2018년과 2022년에 각 1편뿐이라 어느 해에도 몰려 있지 않습니다. 앞서 걱정했던 "국가 층에서 `equal`과 `inv_n_authors`가 같다"는 가정은 사실상 성립하고, 정의를 바꿀 필요는 없습니다. 이 점검은 `authorship_level`이 있는 논문만 비교한 결과입니다. authorship이 없는 204편은 `n_authors`로 미상 질량이 계산되므로 이 점검 대상이 아니고, 같은 문제가 생기지 않습니다.

# ## 기록할 것

# CHANGELOG_S2의 정합 FAIL 줄 아래에 한 줄만 추가하세요.

# > 저자 수 불일치: 창 안 81,441편 중 2편(2018, 2022 각 1편). 국가 층에서 equal=inv_n_authors 가정은 이 2편을 제외하고 성립. 정의 변경 없음.

# 부록의 `equal`=`inv_n_authors` 문구는 그대로 두세요. 동결된 파일을 고치는 것은 사후 변경이고, 2편은 한계로 위 CHANGELOG 줄에 남기는 것으로 충분합니다.

# ## 이제 commit 단계

# `commit`은 `s2_run_steps.py`가 허용하지 않으니, 그 셀의 아래 두 줄만 바꿔 쓰세요.

# ```python
# ALLOWED = ["missing", "risk", "risk_selection", "commit"]   # 기존 줄을 교체
# ...
# m.S2.mkdir(parents=True, exist_ok=True)
# m.COMMIT = False        # 먼저 False로 매니페스트만 확인, 확인 후 True로 바꿔 한 번 더 실행
# ```

# 그리고 `STEPS = ["commit"]`로 실행합니다.

# - **`COMMIT=False`로 먼저**: `S2_MANIFEST.csv`가 몇 개 파일을 담는지, 50MB 초과 경고(`paper_ipw.csv.gz`, `country_mass_long` 등)가 나오는지 봅니다. 경고가 나오면 용량을 확인하고 필요하면 저장소 설정(`.gitignore`, 대용량 파일 처리)을 정한 뒤 `True`로 가세요.
# - **`COMMIT=True`**: `paper1/02_S2` 폴더를 커밋하고 태그 `S2-stageAB-<날짜>`를 만듭니다. 같은 날 두 번 하면 "태그 이미 있음"으로 멈추니, 다시 해야 하면 `m.TAG_NAME`을 따로 지정하세요.

# **커밋 전에 스크립트를 02_S2에 복사해 두세요.** `commit`은 `paper1/02_S2`만 추가합니다. 지금 `s2_measurement.py`는 `paper1/` 바로 아래에 있고, `s2_fix_and_run.py`, `s2_run_steps.py`, `s2_recount_flags.py`는 별도 위치일 수 있습니다. 이 스크립트들이 태그에 들어 있지 않으면 결과가 어떤 코드로 나왔는지 재현할 수 없습니다. 복사한 뒤 매니페스트를 다시 만들면 해시가 함께 기록됩니다. 해시를 지금 확인할 수 있는 것은 부록(`646135ccaa68`)과 `alpha_flags.csv`(`4002a150851b`) 두 가지입니다.

# 수작업 열(`clean_review_samples.csv`의 `correct`, `unk_resid_2024_25_sample.csv`의 `note`)은 비어 있는 채로 커밋돼도 됩니다. 채운 뒤 새 커밋을 따로 남기면 되고, 태그는 이 단계(A/B) 시점의 기록입니다.

# ## 남은 일

# 1. `commit` 실행(위 순서)
# 2. 수작업 병행: `clean_review_samples.csv`의 `correct` 1/0, `unk_resid_2024_25_sample.csv`의 `note`
# 3. 확인 병행: F1의 연도 정의(`publication_date`와 호 연도), T-IV 2025와 RailEng 수집 점검, JPT 사유, F4/F5가 `fixed_venue`로 만들어졌는지
# 4. 저자 층 단계(포트폴리오, 잡음 대응, 제목 분류기, K 민감도): F3, F4, F5의 열 목록(이름, dtype, 결측률)을 붙여 주세요. 이 열 목록이 있어야 코드를 맞출 수 있습니다. `F4`의 겹치는 기간 열(`m_P10_14` 대 `m_P12_14`, `m_P20_24` 대 `m_P22_24`) 중 어느 쪽이 주 기간인지도 같이 알려 주세요.

In [ ]:
# -*- coding: utf-8 -*-
# =====================================================================================
# 수작업 검토 도우미 (Jupyter 셀 또는 터미널에서 실행, 입력창에 답을 입력)
#   TARGET = "clean"  clean_review_samples.csv의 correct 열(1/0) 채우기
#   TARGET = "unk"    unk_resid_2024_25_sample.csv의 note 열(원인 유형) 채우기
# 이 도구는 라벨을 자동으로 채우지 않습니다. 한 건씩 보여 주고 입력을 받아 저장합니다.
# 한 건마다 저장하므로 중간에 멈춰도 이어서 할 수 있고(빈 칸만 다시 물음), 최초 실행 때 원본을 *.orig.csv로 백업합니다.
# 입력: 1/0 또는 유형 코드·자유 문장 | s 건너뜀 | b 직전 항목 다시 | q 저장 후 종료
# =====================================================================================
import re, shutil
from pathlib import Path
import numpy as np, pandas as pd

TARGET = "clean"
ROOT = Path.home() / "Research/Minnesota_Transport"; S2 = ROOT / "paper1/02_S2"; DATA = ROOT / "analysis_data/final"
CLEAN = S2 / "clean_review_samples.csv"; UNK = S2 / "unk_resid_2024_25_sample.csv"

RULE_HINT = {  # 규칙마다 '1'의 의미(규칙이 의도대로 맞게 표시했다)를 보여 주는 문구
    "f_bib": "제목이 서지 안내 섹션(연구 논문이 아님)이면 1, 실제 연구 논문이면 0",
    "f_vtmr": "VT Mag의 [Mobile Radio] 칼럼(연구 논문이 아님)이면 1, 연구 논문이면 0",
    "f_repeat": "같은 저널에서 반복되는 칼럼·안내 제목(연구 논문이 아님)이면 1, 정당한 연구 논문이면 0",
    "f_dup": "같은 저널·연도·제목의 중복 레코드(같은 논문이 두 번 들어감)면 1, 서로 다른 정당한 항목이면 0",
    "f_topic18": "비연구 항목이면 1, 연구 논문이면 0",
}
UNK_CODES = {"f": "형식 문제", "i": "기관명만(국가 단서 없음)", "o": "기타"}   # 자유 문장도 입력 가능. 코드는 직접 고쳐 쓰세요.

normt = lambda s: re.sub(r"[^a-z0-9]+", " ", str(s).lower()).strip()

def wilson(k, n, z=1.96):
    if n == 0: return (np.nan, np.nan)
    p = k / n; d = 1 + z * z / n; c = p + z * z / (2 * n); h = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)); return ((c - h) / d, (c + h) / d)

def load(path):
    if not path.exists(): raise SystemExit(f"파일 없음: {path} (clean_flags/unk_resid 단계를 먼저 실행하세요)")
    bak = path.with_name(path.stem + ".orig.csv")
    if not bak.exists(): shutil.copy2(path, bak); print("원본 백업:", bak.name)
    return pd.read_csv(path, dtype=str, keep_default_na=False, encoding="utf-8-sig")

def save(df, path):
    tmp = path.with_name(path.stem + ".tmp.csv"); df.to_csv(tmp, index=False, encoding="utf-8-sig"); tmp.replace(path)

def loop(df, col, path, show, parse):
    todo = [i for i in df.index if df.at[i, col].strip() == ""]; print(f"\n남은 항목 {len(todo)}건 / 전체 {len(df)}건")
    k = 0
    while k < len(todo):
        i = todo[k]; print("\n" + "-" * 90 + f"\n[{k + 1}/{len(todo)}]"); show(df.loc[i])
        a = input("입력 > ").strip()
        if a.lower() == "q": break
        if a.lower() == "s": k += 1; continue
        if a.lower() == "b":
            if k > 0: k -= 1; df.at[todo[k], col] = ""; save(df, path)
            continue
        v = parse(a)
        if v is None: print("인식하지 못한 입력입니다."); continue
        df.at[i, col] = v; save(df, path); k += 1
    save(df, path); print(f"\n저장: {path.name} | 아직 빈 칸 {int((df[col].str.strip() == '').sum())}건")

def run_clean():
    df = load(CLEAN); F1 = None
    if (df.rule == "f_dup").any():
        hdr = pd.read_csv(DATA / "F1_papers.csv", nrows=0).columns; use = [c for c in ["pid", "title", "venue_short", "year", "doi_norm"] if c in hdr]
        F1 = pd.read_csv(DATA / "F1_papers.csv", usecols=use, dtype={"pid": str}, low_memory=False); F1["t"] = F1.title.fillna("").map(normt)
    def show(r):
        print(f"규칙 {r.rule} | {r.venue_short} {r.year} | pid {r.pid}\n제목: {r.title}\n기준: {RULE_HINT.get(r.rule, '규칙 표시가 맞으면 1, 틀리면 0')}")
        if r.rule == "f_dup" and F1 is not None:
            s = F1[(F1.venue_short == r.venue_short) & (pd.to_numeric(F1.year, errors="coerce") == float(r.year)) & (F1.t == normt(r.title))]
            print(f"같은 저널·연도·제목 레코드 {len(s)}건:"); print(s.drop(columns=["t", "title"]).to_string(index=False))
    loop(df, "correct", CLEAN, show, lambda a: a if a in ("0", "1") else None)
    d = df[df.correct.isin(["0", "1"])].copy(); d["ok"] = d.correct == "1"
    if len(d):
        t = d.groupby("rule").ok.agg(n="size", correct="sum").reset_index(); t["rate"] = (t.correct / t.n).round(3)
        t[["ci_low", "ci_high"]] = [wilson(int(r.correct), int(r.n)) for r in t.itertuples()]; print("\n규칙별 검토 결과(Wilson 95% 구간, 판정 기준은 사전에 정해진 것이 없음):\n", t.round(3).to_string(index=False))

def run_unk():
    df = load(UNK)
    def show(r):
        print(f"{r.venue_short} {r.year} | pid {r.pid} rank {r['rank']}\n소속 문자열: {r.raw_aff}\n코드: {UNK_CODES} (그 외는 자유 문장)")
    loop(df, "note", UNK, show, lambda a: UNK_CODES.get(a, a) if a else None)
    d = df[df.note.str.strip() != ""]
    if len(d): print("\n원인 유형 분포:\n", d.note.value_counts().to_string())

{"clean": run_clean, "unk": run_unk}[TARGET]()

In [27]:
# -*- coding: utf-8 -*-
# =====================================================================================
# S2 A/B 단계 마무리: CHANGELOG 기록 -> 스크립트 복사 -> 커밋 사전점검 -> 커밋+태그
# s2_measurement.py(v3) 필요. OpenAlex 호출 없음. PHASE를 순서대로 바꿔 한 번에 하나씩 실행하세요.
#   PHASE = "changelog"   지금까지의 점검 결과를 CHANGELOG_S2.md에 기록(이미 기록했으면 건너뜀)
#   PHASE = "copy"        사용한 스크립트를 paper1/02_S2로 복사(태그에 코드가 함께 담기도록)
#   PHASE = "commit_dry"  COMMIT=False: 매니페스트 생성 + 파일 크기·git 상태 점검(커밋하지 않음)
#   PHASE = "commit"      CONFIRM_COMMIT=True일 때만: 커밋 + 태그 (push는 하지 않음)
# =====================================================================================
import importlib.util, shutil
from pathlib import Path

PHASE = "commit_dry"
TAG_NAME = None              # None이면 S2-stageAB-<오늘 날짜>
CONFIRM_COMMIT = False       # commit 단계에서 True로 바꿔야 실제 커밋
OVERWRITE = False            # copy 단계에서 02_S2에 내용이 다른 같은 이름 파일이 있을 때 덮어쓸지
BIG_WARN_MB = 50             # 이 크기 초과 파일은 경고
BIG_STOP_MB = 95             # 이 크기 초과 파일이 있으면 commit 중단(GitHub 단일 파일 100MB 제한 고려)
SCRIPTS = ["s2_measurement.py", "s2_run_now.py", "s2_fix_and_run.py", "s2_run_steps.py", "s2_recount_flags.py", "s2_finalize.py", "s2_review_samples.py"]
MARK = "[점검 묶음 A]"

ROOT = Path.home() / "Research/Minnesota_Transport"
cands = [Path.cwd() / "s2_measurement.py", ROOT / "s2_measurement.py", ROOT / "paper1/s2_measurement.py",
         ROOT / "paper1/02_S2/s2_measurement.py", Path.home() / "Downloads/s2_measurement.py"]
MOD = next((c for c in cands if c.exists()), None)
assert MOD is not None, f"s2_measurement.py를 찾지 못했습니다. 후보: {[str(c) for c in cands]}"
_sp = importlib.util.spec_from_file_location("s2", MOD); m = importlib.util.module_from_spec(_sp); _sp.loader.exec_module(m)
assert "commit" in m.FUNCS, "구버전 s2_measurement.py입니다. v3(최신)를 사용하세요"
print(f"모듈: {MOD} | PHASE={PHASE}")
m.S2.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------------------------
def phase_changelog():
    m.hdr("[changelog] 점검 결과 기록")
    cl = m.S2 / "CHANGELOG_S2.md"; txt = cl.read_text(encoding="utf-8") if cl.exists() else ""
    if MARK in txt: print("이미 기록됨(건너뜀):", MARK); return
    L = [
        f"- {m.now()} | {MARK} 규칙·정의 변경 없음(점검 기록) | 정합 점검 FAIL 1행: risk_selection에서 F2 234,060행 중 1행(pid 57786, rank 3)이 authorship_level에 없음. 이 논문은 F1 n_authors=4, authorship_level 3행(저자 수 불일치). 중복 없음, F7-F2 정렬 가정(이름 일치율 100%)은 통과 | risk_selection 정합 점검 | 영향 무시 가능",
        f"- {m.now()} | 저자 수 불일치 점검(변경 없음) | 창 안 81,441편 중 2편만 authorship_level 저자 수와 F1 n_authors가 다름(2018, 2022 각 1편, authorship이 적은 논문 1·많은 논문 1) | 국가 층의 equal=inv_n_authors 가정은 이 2편을 제외하고 성립 | 정의 변경 없음",
        f"- {m.now()} | clean_flags 점검(규칙 변경 없음) | f_bib 0건은 S0 제외로 설명됨(F1 전체 35건이 모두 S0 제외분, F1X 0건; 느슨한 'bibliograph' 패턴으로 F1X에 남은 11편은 실제 연구 논문). f_topic18은 F1 전체 1건(1974 TRR 연구 논문). f_dup 441건(F1X)은 계획서의 459와 5개 정의 어느 것으로도 재현되지 않음(F1 전체 기준 463), 441건 중 352건이 TRR | S2.8 | 규칙은 표시만 하므로 분석 영향 없음",
        f"- {m.now()} | 토픽 번호 메모(변경 없음) | 번호 범위 0~149. 38번(396편, VT Mag 265편)은 무선통신 연구와 [Mobile Radio] 칼럼이 혼재, 49번(582편)은 리뷰·방법론·편집 문서 등 혼재(표본 8편 기준)하고 TransSci 서지·도서 안내 항목이 49번에 집중. 군집 단위 제외는 하지 않음. 18번은 F1 전체 1편으로 규모가 비정상이나 원인 미확인 | S2.8, S3 L5 | topic_excluded=[18] 변경 없음, 38·49번은 S3 L5 복합 토픽 후보",
        f"- {m.now()} | 토픽 결측 2007년 하락(원인 미확인) | 고정 저널 토픽 보유율이 2006년 0.838에서 2007년 0.770으로 하락(TRR 0.899에서 0.825가 주도), 비고정 저널은 반대 방향. TRR 2005~2008에서 연도 사이 결측률이 3%p 넘게 변한 다른 열은 없었음. 성향 모형(저널x연도 상호작용 없음)은 이 하락을 보정하지 못함 | missing | S5 측정 아티팩트 표시와 상호작용 모형 민감도 추가 여부는 별도 결정(추가 시 동결 후 탐색적 민감도로 기록)",
        f"- {m.now()} | risk 평가 범위 | F7 141,778행이 모두 2020년대(2020~2025)라 위험군 플래그의 precision/recall(합집합 기준 약 0.81~0.85 / 약 0.30)은 이 기간에 대한 근사값(정답=이름당 OpenAlex ID 2개 이상). 위험군 변형 7종별 분해는 변형 정의 열이 없어 하지 않음 | S2.2 | 단일 값이 아니라 범위로 보고",
        f"- {m.now()} | risk_selection 결과 | 제외된 authorship 12.0%, 국가군별 제외 비중 CN 23.4%, US 8.4% 등. 논문 단위 국가 몫은 제외 유무로 거의 변하지 않으나(CN 2020~25 equal 0.352 vs 전체 0.352), inv_n(제외 질량 손실) 기준은 0.314 | S2.2, S6.6 | 저자 층 결과(S6, S7.2)의 지역 구성은 이 선택을 한계로 보고",
    ]
    for ln in L: m.changelog(ln)
    print(f"{len(L)}줄 기록: {cl}")
    print("※ prereg 단계의 기록 줄(부록 동결, 알파벳순, weights 정의 출처, JPT)도 이 파일에 있는지 열어서 확인하세요.")

def find(name):
    for b in [Path.cwd(), ROOT, ROOT / "paper1", m.S2, Path.home() / "Downloads"]:
        if (b / name).exists(): return b / name
    return None

def phase_copy():
    m.hdr("[copy] 스크립트를 02_S2로 복사")
    for n in SCRIPTS:
        src = find(n); dst = m.S2 / n
        if src is None: print(f"  {n}: 못 찾음(건너뜀)"); continue
        if src.resolve() == dst.resolve(): print(f"  {n}: 이미 02_S2에 있음  sha {m.sha_f(dst)[:12]}"); continue
        if dst.exists() and m.sha_f(dst) != m.sha_f(src) and not OVERWRITE:
            print(f"  {n}: 02_S2에 내용이 다른 파일이 있어 덮어쓰지 않음(OVERWRITE=True로 바꾸면 덮어씀) | 원본 {src}"); continue
        shutil.copy2(src, dst); print(f"  {n}: 복사 {src} -> 02_S2  sha {m.sha_f(dst)[:12]}")
    print("※ 복사본에는 각 스크립트가 마지막으로 저장될 때의 PHASE/STEPS 설정이 들어 있습니다(재실행 전에 확인).")

def preflight(stop):
    r = m.git("rev-parse", "--is-inside-work-tree")
    if r.returncode: raise m.Halt(f"{m.ROOT}가 git 저장소가 아닙니다: {r.stderr.strip()}")
    files = [f for f in sorted(m.S2.glob("*")) if f.is_file()]; mb = {f.name: f.stat().st_size / 1e6 for f in files}
    print(f"02_S2 파일 {len(files)}개, 합계 {sum(mb.values()):.1f}MB")
    big = {k: round(v, 1) for k, v in mb.items() if v > BIG_WARN_MB}; over = {k: v for k, v in big.items() if v > BIG_STOP_MB}
    print(f"{BIG_WARN_MB}MB 초과:", big or "없음")
    if over and stop: raise m.Halt(f"{BIG_STOP_MB}MB 초과 파일 {over}: 커밋 전에 압축하거나 .gitignore/대용량 파일 처리를 정하세요")
    print("git 상태(02_S2):\n", m.git("status", "--short", "--", "paper1/02_S2").stdout.strip() or "(변경 없음)")
    print("기존 S2 태그:", m.git("tag", "-l", "S2-*").stdout.split() or "없음")

def phase_commit_dry():
    m.hdr("[commit_dry] 매니페스트 + 사전점검 (커밋 안 함)"); preflight(stop=False)
    m.COMMIT = False; m.FUNCS["commit"]()
    print("\n확인 후 문제가 없으면 CONFIRM_COMMIT=True, PHASE='commit'으로 실행하세요.")

def phase_commit():
    m.hdr("[commit] 커밋 + 태그 (push 안 함)")
    if not CONFIRM_COMMIT: raise m.Halt("CONFIRM_COMMIT=True로 바꾼 뒤 실행하세요(commit_dry 점검 후)")
    preflight(stop=True); m.COMMIT = True; m.TAG_NAME = TAG_NAME; m.FUNCS["commit"]()
    print("\n최근 커밋:", m.git("log", "-1", "--oneline").stdout.strip()); print("S2 태그:", m.git("tag", "-l", "S2-*").stdout.split())
    print("※ 원격에 올리려면 직접 push하세요(git push, git push --tags). 이 스크립트는 push하지 않습니다.")

try:
    {"changelog": phase_changelog, "copy": phase_copy, "commit_dry": phase_commit_dry, "commit": phase_commit}[PHASE]()
    print("\n✔ 완료")
except m.Halt as e: print(f"\n⛔ 중단: {e}")

모듈: /home/yjlee/Research/Minnesota_Transport/paper1/s2_measurement.py | PHASE=commit_dry

[commit_dry] 매니페스트 + 사전점검 (커밋 안 함)
02_S2 파일 22개, 합계 14.1MB
50MB 초과: 없음
git 상태(02_S2):
 ?? paper1/02_S2/
기존 S2 태그: 없음

[commit] S2 산출물 해시 + 커밋 + 태그
manifest 21개
COMMIT=False: 커밋 안 함

확인 후 문제가 없으면 CONFIRM_COMMIT=True, PHASE='commit'으로 실행하세요.

✔ 완료


In [ ]:
# ## 한 줄 요약
# S2의 A(기반)와 B(가중·결측)는 국가 층까지 코드로 돌렸고, 남은 것은 C(저자 지표), D(민감도), E(판정)입니다. 앞으로 S2의 중심 작업은 **저자 층 코드**이고, 그 전에 F3/F4/F5 열 구조가 필요합니다.
# ## 1. 계획서 S2 항목별 현황
# | 항목 | 현황 | 남은 것 |
# |---|---|---|
# | **2.1 가중치 4종** | 국가 층은 완료(불변식 8개 PASS, 핵심 C4 부호 일치 23/24). 알파벳순 점검을 구현해 6,011편을 `first_last`에서 `equal`로 처리하고 부록에 동결했습니다 | 저자 층의 가중치 적용, 전원 위험군이어서 빠진 639편의 논문 단위 보고 |
# | **2.2 위험군 재현율** | `risk` 완료(2020년대 범위, 재현율 약 0.30, 정밀도 0.81~0.91). `risk_selection` 완료(중국 소속의 제외 비중 23.4%, 미국 8.4%) | 위험군 변형 7종별 분해와 ID 교체(2020+) 병행(변형 정의 열이 없어 못 했음), `risk_names_report.md` 작성 |
# | **2.3 변수 사전** | 열 구조 점검(`schema`)과 골격 파일은 생성됨 | `definition`, `base_sample`, `derived_from`, `topic_papers_only` 열 채우기. F4의 `n_papers`·`first_year`(주제 없는 논문 포함)와 `m_*`(주제 논문만)의 기준 차이 명시 |
# | **2.4 포트폴리오 지표** | 안 함 | F3 기반의 저자×기간별 1·2위 토픽 비율, 엔트로피, HHI, 수축 추정(k=5), 임베딩 중심 거리 |
# | **2.5 잡음 대응** | 안 함 | 이동 여부(0/1)와 이동 크기의 2부 모형, 논문 3편 이상 부분표본, split-half 신뢰도, 질량 가중 집계 |
# | **2.6 토픽 결측** | (a) 연도·저널·저자 수·단독·국가별 보고 완료, (b) 성향 모형(AUC 약 0.73)과 (c) 양수성 진단(0%) 완료, (d) 국가·연도 몫은 전체 논문으로 계산하는 설계 반영 | 코호트·팀 규모별 결측, 경계 분석(최악·최선 대입), 2007년 하락 대응 결정, `missingness_report.md` |
# | **2.7 제목 분류기** | 안 함 | TF-IDF 1~2gram + 선형 분류기, 저널×연도 그룹 교차검증, 토픽별 F1, 예측 토픽은 민감도에만 사용(tau와 F1 낮은 토픽 제외 규칙 사전등록) |
# | **2.8 정리 규칙** | 표시 완료(546건, 표시만 하고 제거 안 함). 점검 결과 규칙 변경 없음 | 규칙별 30건 수작업 검토, 제거 전후 결과 비교 |
# | **2.9 K 민감도** | 안 함 | K∈{75, 150, 300}×seed 5회 재클러스터링, 핵심 결과의 부호 유지 여부와 ARI. 임베딩 파일 위치를 아직 모릅니다 |
# 사전등록 부록(`646135ccaa68`)은 동결됐습니다. 커밋과 태그는 아직 확인되지 않았습니다.
# ## 2. 앞으로 할 일(순서)
# **지금 바로 (마무리)**
# 1. `s2_finalize.py`로 CHANGELOG 기록, 스크립트 복사, 커밋과 태그
# 2. `s2_review_samples.py`로 `correct`와 `note` 열 채우기
# **독립적으로 시작할 수 있는 것(저자 층 정보 불필요)**
# 3. **2.7 제목 분류기**: F1의 제목과 `topic150`만 있으면 됩니다.
# 4. **2.8 제거 전후 비교**: 수작업 검토가 끝난 뒤에 진행합니다.
# **F3/F4/F5 구조를 받은 뒤 (C 단계)**
# 5. 2.3 변수 사전을 채웁니다.
# 6. 2.4 포트폴리오 지표와 2.5 잡음 대응을 구현합니다.
# 7. 2.6의 코호트·팀 규모별 결측과 경계 분석을 추가합니다.
# 8. 2.1 가중치 4종을 저자 층 결과로 반복합니다.
# **D 단계**
# 9. 2.9 K 민감도(임베딩 파일 위치 필요)
# 10. 2.2의 위험군 변형 7종 분해와 보고서
# **E 단계 (판정)**
# 11. 부호표, 불변식, 음성 대조(주제 라벨을 저널×연도 안에서 섞으면 핵심 효과가 사라지는지) 정리
# E 단계의 핵심 목록 C1, C2, C3, C5는 주제·저자 층 통계가 나와야 계산됩니다. 계획서가 S2 종료 시점에 이 목록의 부호표를 언제 확정하라고 명시하지는 않아서, S2는 "측정 도구와 점검을 확정하는 단계"로 닫고 부호표는 S5~S7 결과가 나올 때 채우는 쪽으로 제가 읽고 있습니다. 이 해석이 맞는지 확인해 주세요.
# ## 3. 결정이 필요한 것
# - **2007년 토픽 결측 하락**: S5의 측정 아티팩트 표시 후보로 올릴지, 저널×연도 상호작용 성향 모형을 민감도로 추가할지 정해야 합니다. 추가하면 동결 후 탐색적 민감도로 기록해야 합니다.
# - **2025년 끝점 민감도**(2025 제외 재계산): 같은 방식으로 추가 여부를 정해야 합니다.
# - **F4의 주 기간 열**: `m_P10_14` 대 `m_P12_14`, `m_P20_24` 대 `m_P22_24` 중 어느 쪽인지 정해야 합니다.
# ## 4. 병행 확인(본 분석을 막지는 않음)
# - F1의 연도 정의(`publication_date` 대 호 연도): S5의 ±1년 사건 대조에 걸립니다.
# - T-IV 2025와 RailEng 수집 점검
# - JPT 제외 사유와 F4/F5가 `fixed_venue` 열로 만들어졌는지
# ## 5. 필요한 입력
# 저자 층 코드를 만들려면 아래를 붙여 주세요.
# - F3, F4, F5의 열 목록(이름, dtype, 결측률)
# - 토픽·임베딩 파일의 위치(2.9용)
# - 위 3번의 결정 사항
# 제가 다음에 만들 우선순위는 입력 없이 바로 가능한 2.7(제목 분류기)이고, 원하시면 그것부터 시작하겠습니다.

In [ ]:
# ## 총평
# 분석은 전반적으로 탄탄합니다. 제가 직접 계산해 본 숫자는 모두 맞았습니다. 고정 저널 합계 64,779 대 64,088, T-ITS 2020~22의 3,032 대 3,049, 알파벳순 표의 초과분 비율, 중국 저자 비중 28.4%에서 24.7%로의 변화, 저널 내 rho의 구간 0.2~0.8, n=19에서의 임계 |rho| 0.46이 그렇습니다. 가설과 확인된 사실을 구분하는 태도도 일관됩니다.
# 약한 곳은 계산이 아니라 **검정의 설계와 아직 닫히지 않은 항목**입니다.
# ## 1. 가장 큰 문제: 통과 기준의 검정력이 낮습니다
# C4의 "가중치 간 부호 일치 23/24"는 좋은 소식처럼 보이지만 정보량이 적습니다.
# - 국가 층에서는 `equal`과 `inv_n_authors`가 같은 값이라 독립 가중치는 3종뿐입니다.
# - 나머지 가중치도 국가 몫을 거의 같게 만듭니다. 한 논문의 질량은 어느 가중치에서나 1이고, 저자 간 배분만 달라집니다.
# - 대상은 19개 연도에 걸친 단조에 가까운 추세입니다. 이런 시계열의 Spearman 부호는 거의 확실히 같습니다.
# 그래서 "부호 일치"는 사실상 어떤 결과도 통과시키는 기준이 됩니다. 이 점은 S5~S6에서 중요합니다. 지금 기준은 "정의에 민감하지 않다"를 보여 주지 못하고, "부호는 같다"만 보여 줍니다. 부호와 함께 **크기(예: rho나 기울기의 범위)나 구간이 겹치는지**를 기준에 넣는 편이 방어에 유리합니다. 이미 동결된 pass_rule이므로 바꾸려면 보완 기준을 별도 부록으로 추가하고 CHANGELOG에 기록해야 합니다.
# ## 2. 위험군 검증의 정답 기준이 약합니다
# "이름당 OpenAlex ID 2개 이상"은 동명이인뿐 아니라 **OpenAlex가 한 사람을 둘로 쪼갠 경우**도 잡습니다. 2020년대 데이터에서는 후자가 상당할 가능성이 큽니다. 게다가 정밀도 0.87에 재현율 0.29라는 패턴은 위험군 플래그가 ID 수와 비슷한 정보로 만들어졌을 때 나오는 모양과 같습니다. 그렇다면 정밀도는 순환적입니다. 플래그를 어떻게 만들었는지 확인하는 것이 먼저이고, 그 전에는 재현율 0.30을 "놓친 동명이인 비율"로 읽으면 안 됩니다. 범위로 보고하는 방침은 맞습니다.
# ## 3. 재현성 위험이 하나 있습니다
# 알파벳순 규칙은 `s2_fix_and_run.py`가 `m.author_w`를 덮어쓰는 방식으로 적용됩니다. 나중에 `s2_measurement.py`의 `weights_country`만 단독으로 돌리면 이 규칙이 **조용히 빠집니다**. 불변식도 이를 못 잡습니다(질량 합이 1이라 PASS). 다음 두 가지를 권합니다.
# - 규칙을 `s2_measurement.py` 본체로 옮기거나, 결과 파일에 "알파벳순 적용 여부와 해시"를 기록하게 하세요.
# - 알파벳순 판정이 F2(토픽 논문)에서만 나오므로, 토픽 보유율이 70%에서 87%로 오르는 동안 `first_last`의 정의가 연도에 따라 미세하게 달라집니다. 영향은 작아 보이지만(3~4인, 가중 차이 0.067 이하) 논문에서는 한 줄 한계로 적어 두세요.
# ## 4. S5와 S6를 막을 수 있는 미해결 항목
# 중요도 순입니다.
# 1. **F1의 연도 정의**: Scimago 대조에서 나온 연도 귀속 이동 가설이 맞으면 S5의 ±1년 사건 대조와 변화점 해석이 직접 흔들립니다. T-ITS는 약 2년 이동처럼 보입니다.
# 2. **JPT와 F4/F5**: F1의 `fixed_venue`에는 JPT가 있고 PREREG에는 없습니다. F4/F5가 이 열로 만들어졌다면 S6의 `continuing_main`과 `entrant_main`이 오염됩니다. 사유는 문서로 남아 있지 않으니 기억하시는 분이 직접 확인하셔야 합니다.
# 3. **T-IV 2025(비 0.06)와 RailEng(비 중앙값 0)**: 수집 누락 가능성이 큽니다.
# 4. **81,485편(S1)과 81,645편(창 안 F1X)의 160편 차이**: 아직 설명이 없습니다.
# 5. **2007년 토픽 보유율 하락**: 성향 모형에 저널×연도 상호작용이 없어 보정되지 않고, S5에서 인공 변화점이 될 수 있습니다. 측정 아티팩트 후보로 지금 표시해 두는 것은 결과를 보기 전의 조치라 비용이 없습니다.
# ## 5. 잘 된 점
# - 가설을 "원인은 확인되지 않음"으로 일관되게 표시합니다. f_dup 441 대 459처럼 재현되지 않는 수치도 "재현 안 됨"으로 기록하는 방침이 좋습니다.
# - 동결 규칙을 결과를 본 뒤 바꾸려는 유혹(f_topic18을 49번으로 바꾸기, ALPHA_MIN_N 올리기)을 막았습니다.
# - 성향 가중이 약하게 작동한다는 점(ESS 97%)과 그 한계(관측되지 않은 요인은 보정 못 함)를 둘 다 짚었습니다.
# - 저자 수준 위험군 제외가 국가별로 불균등하다(CN 23.4%, US 8.4%)는 발견은 S6과 S7.2의 실질적 한계이고, 논문에 쓸 만한 결과입니다.
# ## 6. 전략적 제언
# S2에서 진단이 계속 가지를 치고 있습니다(f_dup 459, 토픽 18, 2007 하락 원인 등). 지금 상태는 지나치게 방어적인 편입니다. 논문의 기여는 S5~S6에 있으니, **S2는 위 4번의 1~3번을 닫고 마무리하는 쪽**을 권합니다. 원인 미확인으로 기록하고 넘어가도 되는 항목에는 시간 상한을 두세요. 한 가지 확인이 필요합니다. 이 속도로 S2 전체를 하면 S5 전에 몇 주가 더 걸리는데, 투고 일정에 비춰 감당 가능한 수준인지 점검해 보세요.
# 가장 먼저 할 일은 F1의 연도 정의와 JPT 문제를 닫는 것입니다. 원하시면 연도 정의(publication_date 대 호 연도) 비교 스크립트부터 만들어 드리겠습니다.

In [28]:
# -*- coding: utf-8 -*-
# =====================================================================================
# S2 마무리 점검 3종 (읽기 전용, OpenAlex 호출 없음, 동결 파일 수정 없음)
#   [A] F1 연도 정의: 날짜 열이 있으면 직접 비교, 없으면 Scopus 대비 이동(shift) 검정으로 간접 판별
#   [B] JPT: F1.fixed_venue(15개) vs PREREG(14개). F3/F4/F5 열 구조와 오염 규모(상한) 점검
#   [C] T-IV 2025, RailEng: 수집 누락 여부를 DOI 접두, 연도별 건수, Scimago 행과 대조
# 사용법: s2_measurement.py(v3)와 같은 폴더 또는 후보 경로에 두고 실행. PHASE로 하나씩 또는 "all".
#   기록: WRITE_CHANGELOG=True면 마지막에 제안 줄을 CHANGELOG_S2.md에 추가(기본 False, 출력만)
# 이 스크립트가 하지 않는 것: 규칙·임계값·동결 부록 변경. 결과를 본 뒤 규칙을 바꾸면 사후 변경입니다.
# =====================================================================================
import importlib.util, json, re, warnings
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings("ignore"); pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40); pd.set_option("display.max_rows", 200)

PHASE = "all"                 # "A", "B", "C", "all"
WRITE_CHANGELOG = False
SHIFTS = [-2, -1, 0, 1, 2]    # 코퍼스 연도 y를 Scopus 연도 y+s와 비교
MIN_YEARS = 8                 # 저널별 이동 검정에 필요한 최소 연도 수
FOCUS = {"T-IV": [2023, 2024, 2025], "RailEng": list(range(2007, 2026))}

ROOT = Path.home() / "Research/Minnesota_Transport"
cands = [Path.cwd() / "s2_measurement.py", ROOT / "s2_measurement.py", ROOT / "paper1/s2_measurement.py", ROOT / "paper1/02_S2/s2_measurement.py", Path.home() / "Downloads/s2_measurement.py"]
MOD = next((c for c in cands if c.exists()), None); assert MOD is not None, f"s2_measurement.py를 찾지 못했습니다: {[str(c) for c in cands]}"
_sp = importlib.util.spec_from_file_location("s2", MOD); m = importlib.util.module_from_spec(_sp); _sp.loader.exec_module(m)
S0, S1, S2, DATA, SC = m.S0, m.S1, m.S2, m.DATA, m.SC
hdr = m.hdr; NOTES = []
def note(s): NOTES.append(s); print("  >>", s)

def headers(p):
    try:
        if p.suffix == ".parquet":
            import pyarrow.parquet as pq; return list(pq.ParquetFile(p).schema.names)
        return list(pd.read_csv(p, nrows=0).columns)
    except Exception: return []

def fixed14():
    return json.loads((S0 / "PREREG.json").read_text(encoding="utf-8"))["event"]["fixed_venues"]

# ---------------------------------------------------------------------------------
def phase_A():
    hdr("[A] F1 연도 정의: OpenAlex 선공개(온라인) 연도 vs Scopus 호 연도")
    # A1. 날짜·호 관련 열 탐색 (F1과 S1·DATA의 모든 표 헤더)
    pat = re.compile(r"date|issue|volume|pub_?year|epub|online|first_?pub", re.I); found = []
    for base in [DATA, S1]:
        for p in sorted(base.glob("*")):
            if p.is_file() and (p.suffix in (".csv", ".parquet") or p.name.endswith(".csv.gz")):
                hit = [c for c in headers(p) if pat.search(c)]
                if hit: found.append((p.name, hit))
    print("날짜·호 관련 열 후보:", found or "없음")
    F1 = pd.read_csv(DATA / "F1_papers.csv", low_memory=False)
    dcols = [c for c in F1.columns if re.search(r"date", c, re.I)]
    if dcols:
        c = dcols[0]; d = pd.to_datetime(F1[c], errors="coerce"); ok = d.notna()
        diff = (d.dt.year - F1.year)[ok]
        print(f"F1.{c}: 파싱 가능 {ok.mean():.1%} | 날짜 연도 != F1.year 비율 {(diff != 0).mean():.2%}")
        print("차이 분포:", diff.value_counts().head(6).to_dict())
        tt = F1[ok].assign(dy=d[ok].dt.year).query("venue_short == 'T-ITS' and year in [2019,2020,2021,2022]")
        print("T-ITS 연도별 (F1.year vs 날짜 연도) 건수:\n", pd.crosstab(tt.year, tt.dy).to_string())
        note(f"F1.year와 {c}의 연도가 다른 비율 {(diff != 0).mean():.2%}. F1.year의 정의는 이 열과의 관계로 확정할 것")
    else:
        note("F1에 날짜 열이 없음: 연도 정의는 S1 수집 코드(OpenAlex publication_year 사용 여부)에서 확인해야 함")
    # A2. 이동 검정: 코퍼스 y vs Scopus y+s (Scimago 대조표만 사용)
    sp = SC / "coverage_vs_scopus.csv"
    if not sp.exists(): print("coverage_vs_scopus.csv 없음: A2 생략"); return
    C = pd.read_csv(sp); C = C[C.documents.fillna(0) > 0]; fx = set(fixed14()); rows = []
    for v, g in C.groupby("venue_short"):
        n = g.set_index("year").n_corpus; s = g.set_index("year").documents
        if len(g) < MIN_YEARS: continue
        for sh in SHIFTS:
            sc = s.copy(); sc.index = sc.index - sh            # y+sh의 Scopus 값을 y에 맞춤
            j = pd.concat([n, sc], axis=1, keys=["n", "s"]).dropna(); j = j[(j.n > 0) & (j.s > 0)]
            if len(j) < MIN_YEARS - 2: continue
            rows.append(dict(venue_short=v, fixed=v in fx, shift=sh, years=len(j), med_abs_logratio=float(np.median(np.abs(np.log(j.n / j.s)))), pooled_ratio=float(j.n.sum() / j.s.sum())))
    R = pd.DataFrame(rows)
    if R.empty: print("이동 검정 가능한 저널 없음"); return
    R.to_csv(S2 / "s2_close_shift_test.csv", index=False, encoding="utf-8-sig")
    best = R.loc[R.groupby("venue_short").med_abs_logratio.idxmin()][["venue_short", "fixed", "shift", "med_abs_logratio"]]
    zero = R[R["shift"] == 0].set_index("venue_short").med_abs_logratio; best["zero_shift"] = best.venue_short.map(zero); best["gain"] = best.zero_shift - best.med_abs_logratio
    print("저널별 최적 이동(중앙 |log 비|이 최소인 s). gain = 이동 0 대비 개선량\n", best.sort_values("gain", ascending=False).head(15).round(3).to_string(index=False))
    fb = best[best.fixed]; share = (fb["shift"] != 0).mean()
    print(f"\n고정 저널 {len(fb)}개 중 최적 이동이 0이 아닌 비율 {share:.0%} | 이동 +1: {(fb['shift'] == 1).sum()}, -1: {(fb['shift'] == -1).sum()}, 그 외: {((fb['shift'].abs() >= 2)).sum()}")
    print("해석 규칙: 이동 0이 최적인 저널이 대부분이고 gain이 작으면 연도 정의 차이는 소수 저널 문제. 한쪽 부호(+1 또는 -1)로 모이고 gain이 크면 정의 차이(OpenAlex 선공개 vs 호 연도) 가설과 일치.")
    print("주의: 이 검정은 집계 수준의 간접 증거이며 개별 논문의 연도 차이를 보여 주지는 않음. 원인 단정 금지.")
    note(f"Scopus 대비 이동 검정: 고정 저널 중 최적 이동 != 0 비율 {share:.0%}(s2_close_shift_test.csv). S5 사건 대조(±1년)에서 연도 정의를 한계로 명시할 것")

# ---------------------------------------------------------------------------------
def phase_B():
    hdr("[B] JPT와 F4/F5의 고정 저널 정의")
    F1 = pd.read_csv(DATA / "F1_papers.csv", low_memory=False); F1["pid"] = F1.pid.astype(str)
    fv = set(fixed14()); fx = set(F1[F1.fixed_venue].venue_short)
    print(f"F1.fixed_venue {len(fx)}개 vs PREREG {len(fv)}개 | F1에만: {sorted(fx - fv)} | PREREG에만: {sorted(fv - fx)}")
    w = F1[F1.year >= m.WINDOW]; n14 = int(w.venue_short.isin(fv).sum()); nj = int((w.venue_short == "JPT").sum())
    print(f"창(2007+) 논문: 고정 14개 {n14:,}편 | JPT {nj:,}편 | JPT가 15개 고정 집합에서 차지하는 비중 {nj / (n14 + nj):.2%}")
    # 구조 점검
    files = sorted(p for p in DATA.iterdir() if p.is_file() and re.match(r"F[345]", p.name) and (p.suffix in (".csv", ".parquet") or p.name.endswith(".csv.gz")))
    print("F3/F4/F5 파일:", [p.name for p in files])
    info = {}
    for p in files:
        cols = headers(p); hit = [c for c in cols if re.search(r"fixed|venue|jpt|continuing|entrant|main", c, re.I)]
        info[p.name] = cols; print(f"\n[{p.name}] {len(cols)}열 | 고정·저널·지속·진입 관련 열: {hit}")
    # F3에 저널 열이 있으면 JPT 오염 규모를 직접 센다
    for p in files:
        cols = info[p.name]
        vc = next((c for c in cols if c in ("venue_short", "venue", "journal")), None)
        ac = next((c for c in cols if c in ("name_norm", "author_id", "oa_id", "author")), None)
        if vc and ac and p.name.startswith("F3"):
            D = pd.read_csv(p, usecols=[vc, ac], low_memory=False)
            jp = D[D[vc] == "JPT"]; a_j = set(jp[ac]); a_14 = set(D[D[vc].isin(fv)][ac])
            only = a_j - a_14
            print(f"\n[{p.name}] JPT 행 {len(jp):,} | JPT에 나온 저자 {len(a_j):,} | 이 중 고정 14개에는 한 번도 안 나온 저자 {len(only):,} (JPT를 고정에 넣을 때만 '고정 저널 저자'가 되는 수)")
            note(f"F3 기준 JPT 영향 저자 {len(a_j):,}명, JPT 때문에만 고정 집합에 들어가는 저자 {len(only):,}명")
    print("\n판독: F4/F5에 'fixed' 계열 열이 있으면 그 열이 어떤 집합(14/15)으로 만들어졌는지 생성 코드를 확인해야 함. 이 스크립트는 F4/F5를 재생성하지 않음.")
    print("JPT 비중이 작으면(위 비중이 1% 안팎) 한계 문장으로 처리 가능. 크면 F4/F5의 지속·진입 정의를 PREREG 14개로 재생성하고 사후 변경으로 CHANGELOG_S2에 기록.")
    note(f"JPT는 창 안에서 15개 고정 집합 논문의 {nj / (n14 + nj):.2%}. F4/F5 생성 집합 미확인이면 S6 전에 확인 필요")

# ---------------------------------------------------------------------------------
def phase_C():
    hdr("[C] T-IV 2025와 RailEng: 수집 누락 점검")
    F1 = pd.read_csv(DATA / "F1_papers.csv", low_memory=False)
    C = pd.read_csv(SC / "coverage_vs_scopus.csv") if (SC / "coverage_vs_scopus.csv").exists() else None
    F1["doi_prefix"] = F1.doi_norm.fillna("").astype(str).str.split("/").str[0].replace("", "NO_DOI") if "doi_norm" in F1 else "NA"
    rows = []
    for v, yrs in FOCUS.items():
        d = F1[F1.venue_short == v]
        print(f"\n[{v}] F1 {len(d):,}편, 연도 {int(d.year.min()) if len(d) else '-'}~{int(d.year.max()) if len(d) else '-'}")
        t = d.groupby("year").agg(n=("pid", "size"), no_doi=("doi_prefix", lambda s: (s == "NO_DOI").mean()), n_prefix=("doi_prefix", "nunique"), topic=("has_topic", "mean") if "has_topic" in d else ("pid", "size")).reset_index()
        if C is not None:
            c = C[C.venue_short == v][["year", "n_corpus", "documents", "ratio", "flag"]]; t = t.merge(c, on="year", how="outer")
        t = t[t.year.isin(yrs)].round(3); print(t.to_string(index=False)); t.insert(0, "venue_short", v); rows.append(t)
        print("DOI 접두 상위:", d.doi_prefix.value_counts().head(4).to_dict())
        if v == "RailEng" and C is not None:
            z = C[(C.venue_short == v) & (C.n_corpus == 0) & (C.documents > 0)]
            print(f"RailEng: 코퍼스 0편인데 Scopus에는 있는 연도 {len(z)}개 -> {sorted(z.year.astype(int).tolist())}")
    pd.concat(rows).to_csv(S2 / "s2_close_venue_year_checks.csv", index=False, encoding="utf-8-sig")
    # 보조 근거: S0 DOI 접두 보고서, S1 audit, Scimago 메타
    for fn, base in [("DOI_PREFIX_REPORT.csv", S0), ("coverage_audit.csv", S1)]:
        p = base / fn
        if p.exists():
            D = pd.read_csv(p, header=0, low_memory=False); col = D.columns[0]
            vcol = next((c for c in D.columns if "venue" in c.lower()), None)
            if vcol is None:  # 헤더가 없는 형식이면 문자열 검색
                s = D[D.apply(lambda r: r.astype(str).str.contains("RailEng|T-IV").any(), axis=1)]
            else: s = D[D[vcol].isin(["RailEng", "T-IV"])]
            print(f"\n[{fn}] 관련 행 {len(s)}개(앞 12행)\n", s.head(12).to_string(index=False))
    mp = next(iter(SC.glob("**/scimago_journal_meta.csv")), None)
    if mp is not None:
        M = pd.read_csv(mp); mc = next((c for c in M.columns if "venue" in c.lower()), None)
        if mc: print("\n[Scimago 메타] RailEng, T-IV:\n", M[M[mc].isin(["RailEng", "T-IV"])].T.head(20).to_string())
    print("\n판독 규칙(미리 정함):")
    print("  RailEng: 코퍼스 0편 연도가 연속이고 DOI 접두가 단일하거나 DOI 결측이 높으면 매핑(저널 ID/DOI) 문제, 연도에 따라 들쭉날쭉이면 수집 누락. 어느 쪽이든 원인은 S1 수집 코드를 봐야 확정.")
    print("  T-IV 2025: 2024 대비 건수가 급감하고 DOI 접두가 같으면 선공개 지연이나 수집 시점 문제 가능성, 접두가 바뀌면 출판사 이동 가능성. 확정은 아님.")
    print("  조치: 두 저널은 해당 칸을 분석에서 제외하지 말고 '코퍼스 불완전 칸'으로 표시하고 S5 끝점 민감도(2025 제외)에 사전 명시.")
    note("T-IV 2025와 RailEng 칸은 s2_close_venue_year_checks.csv에 기록. 원인 미확인이면 '코퍼스 불완전 칸'으로 표시")

# ---------------------------------------------------------------------------------
PH = {"A": phase_A, "B": phase_B, "C": phase_C}
for k in (["A", "B", "C"] if PHASE == "all" else [PHASE]):
    try: PH[k]()
    except Exception as e:
        import traceback; print(f"\n⛔ 단계 {k} 오류(전체 출력을 붙여 주세요):"); traceback.print_exc()

hdr("[요약] CHANGELOG_S2에 남길 제안 줄 (규칙 변경 없음, 점검 기록)")
for s in NOTES: print("-", m.now(), "| S2 마무리 점검(변경 없음) |", s)
if WRITE_CHANGELOG:
    for s in NOTES: m.changelog(f"- {m.now()} | S2 마무리 점검(변경 없음) | {s}")
    print("CHANGELOG_S2.md에 기록함")
else: print("WRITE_CHANGELOG=False: 위 줄은 기록되지 않았습니다. 판독 후 True로 바꿔 다시 실행하거나 직접 붙여 넣으세요.")
print("\n다음: 이 스크립트를 02_S2로 복사한 뒤 s2_finalize.py의 SCRIPTS에 's2_close_checks.py'를 추가하고 commit_dry -> commit 순서로 마무리하세요.")


[A] F1 연도 정의: OpenAlex 선공개(온라인) 연도 vs Scopus 호 연도
날짜·호 관련 열 후보: [('credit_ledger.csv', ['date_utc'])]
  >> F1에 날짜 열이 없음: 연도 정의는 S1 수집 코드(OpenAlex publication_year 사용 여부)에서 확인해야 함
저널별 최적 이동(중앙 |log 비|이 최소인 s). gain = 이동 0 대비 개선량
 venue_short  fixed  shift  med_abs_logratio  zero_shift  gain
       IJST  False      2             0.186       0.288 0.102
  TranspRev   True     -2             0.098       0.152 0.054
    ITS Mag  False     -1             0.377       0.413 0.036
   TransSci   True      1             0.119       0.152 0.033
      T-ITS   True      1             0.158       0.175 0.018
       JTTE  False     -1             0.071       0.081 0.011
     VT Mag  False      1             0.150       0.158 0.007
      IJTST  False      2             0.136       0.136 0.000
       ETRR  False      0             0.041       0.041 0.000
      CACIE   True      0             0.136       0.136 0.000
       AMAR  False      0             0.053       0.053 0.000
        AAP   True      0 

In [29]:
# ## 총평

# 세 가지 모두 한 번에 닫히지는 않았습니다. 다만 RailEng은 수집 문제가 아닌 것으로 정리되고, T-IV 2025는 수집이 사실상 비어 있는 칸으로 확인됐으며, 연도 정의는 지난번 제 가설을 철회해야 합니다.

# ## 1. 연도 정의: 가설을 철회합니다

# - F1에는 날짜 열이 없어서 직접 비교는 불가능했습니다. 연도 정의는 S1 수집 코드에서 확인해야 합니다.
# - 고정 저널 14개 중 최적 이동이 0이 아닌 곳은 3개(21%)입니다. TranspRev는 −2, TransSci와 T-ITS는 +1입니다. 부호가 한쪽으로 모이지 않습니다.
# - gain도 작습니다. T-ITS는 중앙 |log 비|가 0.175에서 0.158로 줄었을 뿐입니다.
# - 앞에서 "T-ITS는 약 2년 이동처럼 보인다"고 했는데, 이 검정은 그것을 지지하지 않습니다. 이동 검정이 중앙값을 쓰기 때문에, 2020~22년처럼 인접 연도끼리 짝으로 상쇄되는 패턴은 약하게 잡힐 수 있습니다. 합산이 맞는다는 이전 관찰(3,032 대 3,049)은 그대로 유효합니다. 그러나 "OpenAlex 선공개 연도 대 Scopus 호 연도"라는 가설은 확인도 반증도 못 했습니다.
# - TranspRev는 이동보다 정의 차이일 가능성이 큽니다. 고정 저널 표시 칸 목록에서 7개 연도가 모두 high였습니다. 코퍼스가 Scopus보다 일관되게 크다는 뜻이라 문서 유형 정의 쪽 설명이 더 자연스럽지만, 이것도 확인된 것은 아닙니다.
# - 결론은 "고정 저널 대부분에서 연도 정의 차이는 보이지 않고, 일부 저널(T-ITS, TransSci, TranspRev)에서 ±1~2년 이동이 가능하다"입니다. S5 사건 대조(±1년)의 한계로 적으면 됩니다.

# 확정은 수집 코드로 하세요.

# ```bash
# grep -rn "publication_year\|from_publication_date\|publication_date" ~/Research/Minnesota_Transport --include=*.py --include=*.ipynb | head -20
# ```

# `publication_year`를 썼다면 OpenAlex의 연도 정의를 그대로 쓴 것입니다. 이 경우 "OpenAlex 연도 기준"이라고 명시하는 것이 방어 문장입니다.

# ## 2. JPT와 F4/F5: 위험이 반쯤 걸러졌습니다

# - **논문 단위**: 창 안에서 JPT는 15개 고정 집합 논문의 0.72%(469편)입니다. 이 수준이면 한계 문장으로 처리할 수 있습니다.
# - **저자 단위**: JPT에 나온 저자 947명 중 317명(33%)은 고정 14개에 한 번도 안 나옵니다. 논문 비중보다 저자 영향이 큽니다. 다만 F3에는 비고정 저널도 있어서, 이 317명 전부가 "JPT 때문에만 고정이 된 저자"는 아닙니다. 분모(전체 저자 수)도 이 출력에 없습니다. 상한으로만 읽으세요.
# - **F4/F5 열 이름에 `fixed` 계열이 없습니다.** F4는 `n_venues`, `continuing_main`, `entrant_main`이고 F5는 `main_venue0`뿐입니다. 이 출력만으로는 F4/F5가 `fixed_venue` 열로 만들어졌는지 알 수 없습니다. 만들어졌다는 증거도 없고, 아니라는 증거도 없습니다.
# - 아래 grep 한 번으로 닫힙니다.

# ```bash
# grep -rn "fixed_venue" ~/Research/Minnesota_Transport --include=*.py --include=*.ipynb | head -20
# ```

# - `fixed_venue`를 쓰는 곳이 F4/F5 생성 코드에 없으면 이 건은 닫힙니다.
# - 쓰는 곳이 있으면 `continuing_main`과 `entrant_main`의 정의를 읽어야 합니다. 그때는 PREREG 14개로 재생성하고 사후 변경으로 기록하세요. 이 경우 JPT 사유가 문서로 남아 있지 않은 것이 문제가 됩니다.

# ## 3. T-IV 2025와 RailEng

# **RailEng은 수집 문제가 아닙니다.** 앞서 "저널 ID나 DOI 매핑을 확인하라"고 했는데, 그 진단은 맞지 않았습니다.
# - 코퍼스는 2020년부터 시작하고(204편, DOI 접두 10.1007 하나), 2020~2025년 비는 1.00~1.21입니다. 합산하면 코퍼스 204 대 Scopus 193(약 1.06)입니다.
# - Scimago 메타의 `coverage` 필드는 2020~2026인데, Documents 값은 2011~2019년(25~40편)에도 있습니다. 같은 Scimago ID 아래 2020년 이전 이력이 붙어 있는 것입니다. 저널명 변경이나 이전 제호의 이력일 수 있다는 가설인데, 제가 확인한 것은 아닙니다.
# - 따라서 비 중앙값 0은 "코퍼스에 없는 해를 Scimago와 비교했기 때문"입니다. 9개 칸(2011~2019)은 수집 누락이 아니라 비교 불가 칸입니다.
# - 조치로는 동결된 플래그는 건드리지 말고, 부록 표에서 이 9개 칸을 "Scimago 전용 이력"으로 분리해 보고하세요. 규칙 변경이 아니라 보고 분리입니다.
# - 이는 E1(코퍼스 첫 논문 연도) 2020과 Scopus 색인 시작이 겹친다는 설명이 됩니다. 창간연도를 확인해 주는 것은 아닙니다.

# **T-IV 2025는 사실상 빈 칸입니다.**
# - 코퍼스는 2024년 806편에서 2025년 23편으로 줄었고, 토픽 보유율도 0.478로 낮습니다. DOI 접두는 10.1109 하나이고 DOI 결측은 0이라 출판사 이동은 아닙니다.
# - Scopus도 2024년 711편에서 2025년 383편으로 줄었습니다. 그래도 비 0.06은 대부분 코퍼스 쪽 문제입니다.
# - 코퍼스의 2025년 부족분 −1,158편 중 T-IV가 −360편(약 31%)입니다. T-IV는 비고정 저널이라 고정 14개 분석에는 영향이 없고, 전체 34개 표본의 2025년에만 영향이 갑니다.
# - 원인은 이 출력으로는 알 수 없습니다. 후보는 수집 시점이나 OpenAlex 쪽 지연입니다. OpenAlex 한 번 호출로 건수를 확인할 수 있습니다. 호출은 S1 자료를 바꾸지 않지만 크레딧을 쓰니 `credit_ledger`에 기록하세요.

# ```bash
# curl -s "https://api.openalex.org/works?filter=primary_location.source.issn:2379-8858,publication_year:2025&per-page=1" | python -c "import sys,json; print(json.load(sys.stdin)['meta']['count'])"
# ```

# OpenAlex 쪽 건수가 수백이면 수집 누락(S1 수집 코드 점검 필요)이고, 20편대면 OpenAlex 자체의 지연입니다.

# ## S2를 닫기 위한 결정

# 1. **2025 끝점 민감도(2025 제외 재계산)를 사전 명시로 추가하세요.** T-IV 2025가 사실상 비어 있고 Scopus 대조에서도 2025년이 약해서, 이제 근거가 충분합니다. 동결 부록 뒤에 추가하는 것이므로 CHANGELOG에 "동결 후 추가한 탐색적 민감도"로 기록하세요.
# 2. **T-IV 2025 칸을 "코퍼스 불완전 칸"으로 표시하세요.** 제외는 하지 않습니다.
# 3. **2007년 토픽 보유율 하락(TRR 주도)을 S5의 측정 아티팩트 표시 후보로 올리세요.** 이는 이미 제안한 사항입니다.
# 4. **grep 두 줄의 결과를 확인하세요.** 연도 정의와 F4/F5의 `fixed_venue` 사용 여부입니다.

# ## CHANGELOG_S2에 남길 줄

# 스크립트가 자동으로 쓰는 5줄은 일반 문구입니다. 아래로 대체하세요.

# > F1에 날짜 열 없음, 연도 정의 미확인(S1 수집 코드의 publication_year 사용 여부 확인 예정). Scopus 대비 이동 검정에서 고정 14개 중 3개(T-ITS +1, TransSci +1, TranspRev −2)만 최적 이동이 0이 아니고 부호가 일치하지 않으며 gain이 작음(0.018~0.054). 연도 정의 차이는 일부 저널의 한계로 보고. 변경 없음.

# > JPT: 창 안에서 15개 고정 집합 논문의 0.72%(469편). F3 기준 JPT 저자 947명 중 317명이 고정 14개에 미등장(상한, 분모 미확인). F4/F5의 `fixed_venue` 사용 여부는 열 이름으로 확인되지 않아 생성 코드 확인 예정. 사유 미확인 유지. 규칙 변경 없음.

# > RailEng: 코퍼스는 2020년부터(204편, 10.1007 단일), 2020~25년 비 1.00~1.21. 2011~2019년 9개 칸은 Scimago 전용 이력(Scimago coverage 필드는 2020~2026)이라 수집 누락이 아닌 비교 불가로 분리 보고. T-IV 2025: 코퍼스 23편 대 Scopus 383편(2024년 806편 대 711편), DOI 접두 불변, 2025년 합계 부족(−1,158)의 약 31%. 원인 미확인, 코퍼스 불완전 칸으로 표시, 2025 끝점 민감도를 동결 후 추가한 탐색적 항목으로 기록.

# grep 두 줄과 T-IV 건수를 붙여 주시면 마지막 판독을 하고, 그 뒤 커밋으로 S2를 마무리하겠습니다.

In [ ]:
# # S2 최종 결과와 S3 계획

# ## 0. 먼저 정리할 것

# 계획서의 S2는 2.1~2.9 아홉 항목인데, 지금까지 끝난 것은 **국가 층 측정, 토픽 결측, 위험군 점검, 정리 규칙 표시, 점검 3종**입니다. 저자 층(2.3 변수 사전 완성, 2.4, 2.5)과 2.7, 2.9는 아직 하지 않았습니다. 그래서 아래는 "S2 전체 완료"가 아니라 **S2-AB 단계의 최종 결과**로 읽어 주세요. 계획서가 S3을 S2와 병행하라고 하니 S3 착수는 문제없습니다. 앞서 확인을 부탁드린 해석, 곧 "S2는 측정 도구와 점검을 확정하는 단계로 닫고 C1~C5 부호표는 S5~S7에서 채운다"는 것을 이 문서의 전제로 둡니다. 다르게 보시면 알려 주세요.

# 커밋과 태그(`S2-stageAB-<날짜>`)가 실제로 만들어졌는지는 제가 확인하지 못했습니다.

# ## 1. S2-AB 최종 결과

# ### 1.1 계획서 항목별 상태

# | 항목 | 상태 | 핵심 수치 |
# |---|---|---|
# | 2.1 가중치 4종 | 국가 층 완료, 저자 층 미실시 | 불변식 8개 PASS, S1 재현 최대 차이 6.7e-16, C4 부호 일치 23/24 |
# | 알파벳순 규칙 | 구현·동결 | 판정 가능 55,257편 중 6,011편(우연 기대 4,745편), 96%가 3~4인 논문 |
# | 2.2 위험군 | 부분 완료 | 정밀도 0.81~0.91, 재현율 약 0.30(2020년대 한정), 변형 7종 분해 미실시 |
# | 2.3 변수 사전 | 골격만 | `variable_dictionary_skeleton.csv` |
# | 2.4, 2.5 | 미실시 | F3/F4/F5 열 목록 필요 |
# | 2.6 토픽 결측 | (a)~(d) 완료, 코호트·팀 규모별 결측과 경계 분석 미실시 | 보유율 79.1%, 성향 모형 AUC 0.73, 유효 표본 97% |
# | 2.7 제목 분류기 | 미실시 | F1의 제목과 `topic150`만 있으면 가능 |
# | 2.8 정리 규칙 | 표시 완료, 제거 전후 비교와 수작업 검토 미실시 | 표시 546건(0.49%) |
# | 2.9 K 민감도 | 미실시 | 임베딩 파일 위치 미확인 |

# ### 1.2 확정된 사실

# - **동결 기록**: S2 사전등록 부록 해시 `646135ccaa68`, `alpha_flags.csv` 해시 `4002a150851b`.
# - **고정 저널**: PREREG의 14개를 씁니다. JPT는 2000~2025년 중 2019년에 코퍼스와 Scopus 모두 0편이라 S0 v4.1-5의 "매년 1편 이상" 규칙에서 탈락했습니다. 창 안 논문의 0.72%라 영향은 작습니다.
# - **F4 주 기간 열**: `m_P10_14`와 `m_P20_24`. `continuing_main`과 `entrant_main`은 이 질량 조건으로만 정의되어 `fixed_venue`를 쓰지 않습니다.
# - **연도 정의**: `F1.year`와 OpenAlex `publication_year`가 비교 가능한 42,687편(F1의 38.2%)에서 99.99% 일치합니다. 전체 일반화는 보류합니다. Scopus와의 연도 차이는 고정 저널 14개 중 T-ITS, TransSci, TranspRev 3개에서만 이동이 보였고 방향이 일치하지 않습니다. 원인은 미검증입니다.
# - **Scopus 대조**: 고정 14개 합계는 코퍼스 64,779편 대 Scopus 64,088편(+1.1%)이고 연도별 비는 0.90~1.15입니다.
# - **T-IV 2025**: 코퍼스 23편, OpenAlex 75편, Scopus 383편입니다. 코퍼스 불완전 칸으로 표시하고 제외하지 않습니다. 원인은 미확인입니다.
# - **RailEng**: 코퍼스는 2020년부터이고 2020~2025년 비는 1.00~1.21입니다. 2011~2019년 9개 칸은 Scimago 쪽 이력이라 비교 불가 칸으로 분리 보고합니다.
# - **위험군 제외의 불균형**: 제외된 authorship은 12.0%이고 CN 23.4%, US 8.4%입니다. 논문 단위 국가 몫은 거의 변하지 않지만(CN 2020~25: 0.352 대 0.352), `inv_n`에서는 0.314로 낮아집니다. 저자 층 결과(S6, S7.2)의 지역 구성은 이 선택을 한계로 보고하세요.

# ### 1.3 보고 때 주의할 점

# - **C4 부호표의 검정력**: 23/24 일치는 좋은 소식이지만, 국가 층에서는 `equal`과 `inv_n_authors`가 같아 독립 가중치가 3종이고, 가중치가 한 논문의 질량(합 1)을 바꾸지 않아 19개 연도의 단조 추세는 거의 항상 같은 부호가 나옵니다. 그래서 이 기준만으로는 "정의에 민감하지 않다"를 보이지 못합니다. 크기(rho나 기울기 범위)를 병기하는 보완 기준을 별도 부록으로 추가하고 CHANGELOG에 기록할 것을 권합니다. 갈린 1개 조합은 `main`, 전체 표본, 영토 합산의 `hhi_known`(`full_count`만 +, rho −0.21~0.01)이고, 사전등록 규칙대로 FAIL로 보고합니다.
# - **위험군 정밀도의 순환 가능성**: 정답이 "이름당 OpenAlex ID 2개 이상"이라 ID를 쪼갠 경우도 포함되고, 플래그 생성 방식이 ID 수와 관련 있으면 정밀도가 부풀려집니다. 재현율 0.30은 "놓친 동명이인 비율"로 읽지 말고 범위로 보고하세요.
# - **알파벳순 규칙의 의미**: "알파벳순 관행 논문"이 아니라 "저자 순서가 알파벳순과 일치하는 논문"을 가려냅니다. 초과분 1,266편은 동점 무시 때문에 상한입니다.
# - **알파벳순 규칙의 재현성**: `m.author_w`를 덮어쓰는 방식이라 `weights_country`를 단독 실행하면 이 규칙이 조용히 빠지고 불변식도 못 잡습니다. 결과 파일에 적용 여부와 해시를 기록하게 하거나 본체로 옮기세요.
# - **2007년 토픽 보유율 하락**: 고정 저널에서 0.838→0.770(TRR 0.899→0.825 주도)입니다. 결측률이 3%p 넘게 같이 움직인 다른 열은 없었고, 성향 모형에 저널×연도 상호작용이 없어 보정되지 않습니다. 원인 미확인입니다.
# - **토픽 보유율의 시간 추세**: 2000년 0.698에서 2019년 0.882로 올랐다가 2025년 0.811로 내려갑니다. S5 변화점 탐지의 측정 아티팩트 후보입니다.

# ### 1.4 아직 열려 있는 항목

# | 항목 | 영향 |
# |---|---|
# | 창 안 논문 수 불일치(S1 전수 81,485편, F1X 81,645편, authorship 있는 논문 81,441편) | 정의 차이인지 확인 필요(S2를 막지 않음) |
# | `f_dup` 441건(F1 전체 463건)이 계획서의 459와 재현되지 않음 | "재현 안 됨, 441 사용"으로 기록 |
# | 토픽 18번이 F1 전체 1편(다음으로 작은 군집 169편) | 원인 미확인, 토픽 번호 재할당 가능성 |
# | 토픽 38번(396편, VT Mag 265편)과 49번(582편)의 혼재 | S3 L5의 복합 토픽 후보 |
# | T-ITS 2008 다국적 공저(우리 21.0% 대 Scimago 61.3%) | 원인 미확인 |
# | 규칙별 30건 수작업 검토(`clean_review_samples.csv`)와 `unk_resid` 표본 | 병행 |
# | `BASE` 정의(`(2) 데이터셋정의.ipynb`) | S6 전에 읽을 것 |
# | 위험군 변형 7종별 분해 | 변형 정의 열이 없어 못 함 |

# ### 1.5 CHANGELOG_S2에 남길 동결 후 추가 항목

# 두 항목은 결과를 본 뒤 추가하는 것이므로 "동결 후 추가한 탐색적 항목"으로 적으세요.

# - 2025 끝점 민감도(2025 제외 재계산): 판정 규칙은 불변
# - 2007년을 S5 측정 아티팩트 표시 후보로 지정

# ## 2. S3: 토픽 증거 카드와 신뢰 등급

# ### 2.1 목적과 산출
# 150개 토픽을 이름이 아니라 **증거 묶음(카드)**으로 만들고 A/B/C 등급을 매겨, 핵심 결과가 신뢰도 높은 토픽만으로도 유지되는지 확인할 수 있게 합니다. 산출은 `topic_evidence_cards.csv`, `supplement_label_grid.pdf`, `topic_grade.csv`입니다.

# ### 2.2 S2에서 넘어오는 제약
# 1. **고정 저널은 PREREG 14개**를 씁니다. EDA의 `F1.fixed_venue`(15개)를 쓰면 L4의 "비고정 저널 비율"에 JPT가 섞입니다.
# 2. **정리 규칙은 표시만 하고 제거하지 않았습니다.** 카드는 표시된 논문을 어떻게 다룰지 먼저 정해야 합니다(제외 판정은 결과를 보기 전에).
# 3. **토픽 18, 38, 49번**은 L5에서 처리합니다. 군집 단위 제외는 사후 변경이 되므로 L5 규칙을 정한 뒤 적용하세요.
# 4. **토픽 보유 논문만** 카드의 대상입니다(F1X의 79.1%). 토픽 결측은 카드에 나오지 않으므로, L4의 수치는 성향 가중(`paper_ipw.csv.gz`)을 적용한 값과 무가중 값을 나란히 보고하는 편이 안전합니다.
# 5. **창**은 주제 층 본분석 2000~2025입니다. 첫 기간(≤2004)은 2000~2004년만 본분석으로 보고 이전은 확장으로 표시합니다.
# 6. **Scimago 변수**(SJR 등)는 등급 계산과 대표 논문 선정에 쓰지 않습니다(3.0 원칙). 맥락 표에 병기만 합니다.

# ### 2.3 단계 순서

# | 순서 | 내용 | 비고 |
# |---|---|---|
# | S3-0 | 사전등록 부록 작성·동결(임계값 전부) | 결과를 보기 전에. S2와 같은 해시 방식 |
# | L1 | 기간별 구별 단어(정보적 사전분포 로그 오즈비, 단어와 bigram) | 기간 5개 × 토픽 150개. 기존 lift와 top-6 Jaccard 병기 |
# | L3 | 일관성(상위 10단어 NPMI)과 안정성(논문 부트스트랩 200회, top-10 Jaccard) | 시간 가장 많이 듦 |
# | L5 | 비연구·복합 토픽 탐지(동일 제목 반복, n<30, 동시출현 그래프 모듈러리티) | 18번과 VT Mag 38번 칼럼을 규칙이 잡는지 검증 |
# | L2 | 대표 논문 10편(구별 점수 상위 + 연도·저널 내 인용 백분위 상위) | 최근 연도 인용 미성숙 처리 필요 |
# | L4 | 구조 지문(저널 엔트로피, 1위 저널 비율, 정점 연도, 성장 기울기, 평균 팀 규모, 신규 저자 비율, 비고정 저널 비율, DOI 결측 비율) | 팀 규모와 신규 저자는 F4와 `BASE` 정의가 필요 |
# | L7 | 신뢰 등급 A/B/C | 실패 항목 0개 A, 1개 B, 2개 이상 C |
# | L6 | OpenAlex 토픽 대응표 | 선택 사항. 지금은 보류 |

# ### 2.4 S3-0에서 동결할 값(제 임의 출발값)

# | 값 | 제안 | 비고 |
# |---|---|---|
# | 최소 문서빈도(단어, bigram) | 10편, 5편 | 기간 전체 기준 |
# | 토픽×기간 최소 논문 수 | 20편 | 미만이면 그 칸은 비움 |
# | NPMI 실패 기준 | 0 미만 | 임계값은 표본 분포를 보기 전에 정할 것 |
# | 부트스트랩 Jaccard 실패 기준 | 중앙값 0.5 미만 | |
# | L5 실패 기준 | n<30 또는 반복 제목 비중이 큼 또는 복합 표시 | 반복 제목 비중의 값도 사전 지정 |
# | L4의 연도 인용 백분위 | 연도×저널 내 중간 순위, 2023년 이후는 제외 | 인용 창 정의는 4.3과 맞출 것 |

# 이 값들은 모두 임의입니다. 동결 후 바꾸면 사후 변경이니 지금 정하세요. L7의 "실패 항목"이 정확히 무엇인지도 이 부록에서 정해야 합니다(위 표의 네 항목이 후보).

# ### 2.5 S3의 통과 기준과 한계

# - **통과 기준**: A등급이 전체의 50% 이상이고, 핵심 결과의 부호와 크기(±30%)가 A등급 재추정에서 유지되는 것입니다.
# - **재추정은 S3에서 끝낼 수 없습니다.** 재추정할 핵심 결과(C1~C3, C5)가 S5~S7에서 나오기 때문입니다. S3은 등급표와 "A등급 비중 ≥50%"까지 확정하고, 재추정은 S5~S7 단계에서 닫는 것으로 계획하세요.
# - **사람이 읽어야 합니다.** 등급은 자동으로 매기지만, 구별 단어와 대표 논문을 직접 읽어 보지 않으면 "이 토픽이 무엇인가"를 논문에서 설명하기 어렵습니다. 최소한 등급 경계의 토픽과 S4에서 신생·쇠퇴로 분류될 토픽은 읽으시길 권합니다.
# - **단어 재료**: F1에 있는 텍스트는 제목뿐입니다. 초록이 있는지는 확인하지 못했습니다. 제목만 쓴다면 카드의 해상도가 제한되니 한계로 적으세요.

# ## 3. 필요한 입력

# 1. F3, F4, F5의 열 목록(이름, dtype, 결측률)과 `(2) 데이터셋정의.ipynb`의 `BASE =` 줄(L4의 팀 규모와 신규 저자 비율에 필요)
# 2. 초록 텍스트 보유 여부, 임베딩과 토픽 파일 위치(2.9와 L6 판단용)
# 3. 위 2.4의 임계값 확정 또는 수정

# 이것들이 오면 **S3-0 부록과 L1 코드**부터 만들겠습니다. 입력 없이 지금 바로 시작할 수 있는 것은 L1과 L3이고(F1의 제목과 `topic150`만 필요), 저자 층 정보가 필요한 L4의 일부만 뒤로 미룹니다.

In [39]:
# -*- coding: utf-8 -*-
# =====================================================================================
# S2 열린 항목 규명 (읽기 전용, OpenAlex 호출 없음, 동결 파일 수정 없음)
#   [1] 창 안 논문 수: S1 81,485 / F1X 81,645 / authorship 81,441 의 차이가 어떤 정의에서 나오는가
#   [2] f_dup 441 vs 계획서 459: 정규화·키·표본 조합 격자에서 459가 재현되는가
#   [3] 토픽 18번이 1편뿐인 이유: 다른 파일의 토픽 열, 라벨 배열, 노트북 코드 추적
#   [4] 토픽 38/49 구성: 규칙 지표 + 층화 표본 파일(수작업 라벨용) + 라벨 후 구성비
#   [5] T-ITS 2008 다국적 공저: Scimago 쪽 이웃 이탈 검정 + 우리 값 직접 재계산 + 논문 목록
#   [6] 수작업 검토 진행 현황과 규칙별 정확도(Wilson)
#   [7] BASE 정의 + F2/F3/F4/F5/F7 열 구조(변수 사전 재료) + 임베딩·토픽 파일 위치
#   [8] 위험군 변형 7종: 정의 위치 탐색, 변형별 precision/recall, 이름 빈도별 점검(순환성)
# 사용법: s2_measurement.py(v3)와 같은 폴더 또는 후보 경로에 두고 PHASE를 바꿔 실행("all" 또는 "1".."8").
# 규칙·임계값·동결 부록은 바꾸지 않습니다. 결과를 본 뒤 규칙을 바꾸면 사후 변경이므로 CHANGELOG_S2에 기록하세요.
# =====================================================================================
import importlib.util, json, re, unicodedata, warnings, traceback
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings("ignore"); pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40); pd.set_option("display.max_rows", 200); pd.set_option("display.max_colwidth", 90)

PHASE = "all"            # "all" 또는 "1".."8"
SEED = 2026
N_SAMPLE_PER_GROUP = 30  # [4] 층화 표본 크기(그룹당)

ROOT = Path.home() / "Research/Minnesota_Transport"
cands = [Path.cwd() / "s2_measurement.py", ROOT / "s2_measurement.py", ROOT / "paper1/s2_measurement.py", ROOT / "paper1/02_S2/s2_measurement.py", Path.home() / "Downloads/s2_measurement.py"]
MOD = next((c for c in cands if c.exists()), None); assert MOD is not None, f"s2_measurement.py를 찾지 못했습니다: {[str(c) for c in cands]}"
_sp = importlib.util.spec_from_file_location("s2", MOD); m = importlib.util.module_from_spec(_sp); _sp.loader.exec_module(m)
S0, S1, S2, DATA, SC = m.S0, m.S1, m.S2, m.DATA, m.SC
hdr = m.hdr

# ---------------------------------------------------------------- 공통 도구
def wilson(k, n, z=1.96):
    if n == 0: return (np.nan, np.nan)
    p = k / n; d = 1 + z * z / n; c = p + z * z / (2 * n); h = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)); return ((c - h) / d, (c + h) / d)

def nb_cells(path):
    try: return json.loads(Path(path).read_text(encoding="utf-8"))["cells"]
    except Exception: return []

def notebooks(bases):
    for b in bases:
        for p in sorted(Path(b).rglob("*.ipynb")):
            if ".ipynb_checkpoints" not in str(p): yield p

def grep_nb(pattern, bases, limit=20, flags=0):
    rx = re.compile(pattern, flags); n = 0
    for p in notebooks(bases):
        for ci, c in enumerate(nb_cells(p)):
            for li, ln in enumerate("".join(c.get("source", [])).splitlines()):
                if rx.search(ln):
                    print(f"  [{p.name} | cell {ci} | line {li + 1}] {ln.strip()[:200]}"); n += 1
                    if n >= limit: print("  ... (limit)"); return n
    if n == 0: print("  (일치 없음)")
    return n

def show_cell(path, pattern, max_lines=45, flags=re.M):
    rx = re.compile(pattern, flags)
    for ci, c in enumerate(nb_cells(path)):
        src = "".join(c.get("source", []))
        if rx.search(src):
            print(f"  --- {Path(path).name} cell {ci} ({c.get('cell_type')}) ---")
            for ln in src.splitlines()[:max_lines]: print("  |", ln[:200])
            return True
    print(f"  ({Path(path).name}에서 {pattern!r} 일치 셀 없음)"); return False

def f1_full():
    F1 = pd.read_csv(DATA / "F1_papers.csv", low_memory=False); F1["pid"] = F1.pid.astype(str); return F1

# ================================================================ [1]
def s1():
    hdr("[1] 창 안 논문 수 대조: S1 81,485 / F1X 81,645 / authorship 81,441")
    F1 = f1_full(); FX = m.load_F1X(); W1 = F1[F1.year >= m.WINDOW]; W = FX[FX.year >= m.WINDOW]
    A = m.read_tbl("authorship_level"); A["pid"] = A.pid.astype(str); P = m.read_tbl("country_authorship"); P["pid"] = P.pid.astype(str)
    sa, sp = set(A.pid), set(P.pid); res = {}
    res["F1 창 전체(제외 전)"] = len(W1); res["F1X 창"] = len(W); res["F1X 창 & DOI 있음"] = int(W.doi_norm.notna().sum())
    res["F1X 창 & pid가 authorship_level에 있음"] = int(W.pid.isin(sa).sum()); res["F1X 창 & pid가 country_authorship에 있음"] = int(W.pid.isin(sp).sum())
    res["authorship_level 고유 pid(전체 연도)"] = len(sa); res["country_authorship 고유 pid(전체 연도)"] = len(sp)
    raw = S1 / "oa_raw/works_by_doi.jsonl"; rawset = None
    if raw.exists():
        s = set()
        with raw.open(encoding="utf-8") as f:
            for ln in f:
                try: d = json.loads(ln).get("doi")
                except Exception: continue
                d = m.nd(d) if isinstance(d, str) else None
                if d: s.add(d)
        rawset = s; res["F1X 창 & DOI가 S1 raw JSONL에 있음"] = int(W.doi_norm.map(m.nd).isin(s).sum()); print(f"S1 raw JSONL 고유 DOI {len(s):,}")
    ca = S1 / "coverage_audit.csv"
    if ca.exists():
        C = pd.read_csv(ca); col = "n_papers" if "n_papers" in C.columns else None
        if col: res["coverage_audit n_papers 합(2007+)"] = int(C[C.year >= m.WINDOW][col].sum())
    R = pd.Series(res).to_frame("건수"); R["81,485와 같음"] = R.건수 == 81485; R["81,441과 같음"] = R.건수 == 81441; print(R.to_string())
    print("\n기준 수치와 일치하는 정의:", [k for k, v in res.items() if v in (81485, 81441)] or "없음(어느 정의도 재현 안 됨)")
    # 차이 분해
    W = W.assign(no_doi=W.doi_norm.isna(), no_auth=~W.pid.isin(sa), no_cty=~W.pid.isin(sp), no_topic=W.has_topic == 0 if "has_topic" in W else False)
    if rawset is not None: W["no_raw"] = ~W.doi_norm.map(m.nd).isin(rawset)
    flags = [c for c in ["no_doi", "no_auth", "no_cty", "no_raw", "no_topic"] if c in W]
    print("\n창 안 F1X 논문의 결측 유형별 건수(81,645 기준):", {c: int(W[c].sum()) for c in flags})
    print("목표 차이 160(= 81,645 - 81,485), 204(= 81,645 - 81,441), 44(= 81,485 - 81,441)와 같은 건수:", [c for c in flags if int(W[c].sum()) in (160, 204, 44)] or "없음")
    na = W[W.no_auth]; print(f"\nauthorship 없는 창 안 논문 {len(na):,}편 | 저널 상위 {na.venue_short.value_counts().head(6).to_dict()} | 연도 분포 {na.year.value_counts().sort_index().to_dict()}")
    print(f"  그 중 DOI 없음 {int(na.no_doi.sum())}, raw JSONL에도 없음 {int(na.no_raw.sum()) if 'no_raw' in na else 'NA'}")
    print(f"창 안 F1 전체(제외 전)와 F1X의 차이(S0 제외분) {len(W1) - len(W)}편 | authorship에는 있으나 F1X 창에 없는 pid {len(sa - set(W.pid))}개(전체 연도 포함)")
    print("\n노트북에서 81,485 출처 검색:"); grep_nb(r"81[,_]?485", [ROOT], limit=10)
    print("\n판독: 81,485가 위 정의 중 하나와 같으면 '정의 차이'로 기록. 어느 것과도 다르면 S1 노트북 검색 결과의 셀이 그 수치를 만든 정의입니다(그 셀을 읽으세요).")

# ================================================================ [2]
def s2():
    hdr("[2] f_dup 441 vs 계획서 459: 정규화·키·표본 격자")
    F1 = f1_full(); FX = m.load_F1X(); f2p = set(pd.read_csv(DATA / "F2_author_paper.csv", usecols=["pid"], dtype=str).pid)
    subsets = {"F1 전체": F1, "F1X": FX, "F1X 2000+": FX[FX.year >= 2000], "F1X 2007+": FX[FX.year >= m.WINDOW], "F1X 토픽 보유": FX[FX.has_topic == 1], "F1 토픽 보유": F1[F1.has_topic == 1], "F2 논문": F1[F1.pid.isin(f2p)]}
    norms = {"normt(현행)": m.normt, "소문자·공백": lambda s: re.sub(r"\s+", " ", str(s).lower()).strip(),
             "영숫자만(공백 제거)": lambda s: re.sub(r"[^a-z0-9]", "", str(s).lower()),
             "ASCII 접기": lambda s: re.sub(r"[^a-z0-9]+", " ", unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode().lower()).strip()}
    keys = {"저널·연도·제목": ["v", "y", "t"], "저널·제목": ["v", "t"], "연도·제목": ["y", "t"], "제목만": ["t"]}; rows = []
    for sn, d in subsets.items():
        d = d[d.title.notna()]
        for nn, fn in norms.items():
            t = d.title.map(fn).values
            for kn, cols in keys.items():
                X = pd.DataFrame({"v": d.venue_short.values, "y": d.year.values, "t": t}); X = X[X.t != ""]
                a = int(X.duplicated(cols).sum()); b = int(X.duplicated(cols, keep=False).sum())
                rows.append(dict(표본=sn, 정규화=nn, 키=kn, 첫건_뒤_중복=a, 중복군_전체행=b, 중복군_수=b - a))
    R = pd.DataFrame(rows); R.to_csv(S2 / "s2_open_fdup_grid.csv", index=False, encoding="utf-8-sig")
    hit = R[(R.첫건_뒤_중복 == 459) | (R.중복군_전체행 == 459) | (R.중복군_수 == 459)]
    print(f"격자 {len(R)}개 조합 중 어느 지표든 459인 조합: {len(hit)}개"); print(hit.to_string(index=False) if len(hit) else "  없음")
    R["최소거리"] = R[["첫건_뒤_중복", "중복군_전체행", "중복군_수"]].sub(459).abs().min(axis=1)
    print("\n459에 가장 가까운 12개:\n", R.sort_values("최소거리").head(12).to_string(index=False))
    print("\n현행 정의(F1X, 저널·연도·제목, normt)의 값:", int(R[(R.표본 == "F1X") & (R.정규화 == "normt(현행)") & (R.키 == "저널·연도·제목")].첫건_뒤_중복.iloc[0]))
    print("계획서 2.8 문구 출처 검색:"); grep_nb(r"459", [ROOT], limit=8)
    print("판독: 여러 조합이 459이면 가장 자연스러운 정의를 CHANGELOG에 '정의 불명, 이 조합과 일치'로 기록(규칙은 바꾸지 말 것). 하나도 없으면 '재현 안 됨, 441 사용'.")

# ================================================================ [3]
def s3():
    hdr("[3] 토픽 18번 1편: 다른 파일·노트북 추적")
    F1 = f1_full(); vc = F1.topic150.value_counts().sort_values()
    print("가장 작은 군집 6개:", vc.head(6).to_dict(), "| 존재하는 토픽 번호 수", F1.topic150.nunique(), "| 0~149 중 없는 번호:", sorted(set(range(150)) - set(F1.topic150.dropna().astype(int))))
    one = F1[F1.topic150 == 18]; print("18번 논문:\n", one[[c for c in ["pid", "year", "venue_short", "title", "has_topic", "in_sample"] if c in one]].to_string(index=False))
    print("\n[a] analysis_data 아래 모든 표의 토픽 열에서 18번 건수:")
    base = ROOT / "analysis_data"; rows = []
    for p in sorted(base.rglob("*")):
        if not p.is_file() or ".ipynb_checkpoints" in str(p) or p.stat().st_size > 800e6: continue
        try:
            if p.suffix == ".parquet":
                import pyarrow.parquet as pq; cols = pq.ParquetFile(p).schema.names
            elif p.suffix == ".csv" or p.name.endswith(".csv.gz"): cols = list(pd.read_csv(p, nrows=0).columns)
            else: continue
            for c in [c for c in cols if re.match(r"(?i)topic", c)]:
                s = pd.read_parquet(p, columns=[c])[c] if p.suffix == ".parquet" else pd.read_csv(p, usecols=[c])[c]
                if pd.api.types.is_numeric_dtype(s): rows.append(dict(file=str(p.relative_to(ROOT)), col=c, n_rows=len(s), n_unique=int(s.nunique()), n_18=int((s == 18).sum()), min_size=int(s.value_counts().min())))
        except Exception: continue
    print(pd.DataFrame(rows).to_string(index=False) if rows else "  토픽 열이 있는 표를 찾지 못함")
    print("\n[b] 라벨 배열 후보(.npy/.npz)에서 18번 건수(1차원 정수 배열만):")
    for p in sorted(base.rglob("*.np[yz]")):
        if ".ipynb_checkpoints" in str(p): continue
        try:
            a = np.load(p, mmap_mode="r", allow_pickle=False) if p.suffix == ".npy" else None
            if a is None: print(f"  {p.relative_to(ROOT)} (npz, 건너뜀)"); continue
            extra = f" | 18번 {int(np.count_nonzero(a == 18)):,}, 고유값 {len(np.unique(a)):,}" if a.ndim == 1 and np.issubdtype(a.dtype, np.integer) else ""
            print(f"  {p.relative_to(ROOT)} shape {a.shape} dtype {a.dtype}{extra}")
        except Exception as e: print(f"  {p.name}: 읽기 실패 {e}")
    print("\n[c] 노트북에서 토픽 18을 다루는 코드:")
    grep_nb(r"(topic\w*\s*(==|!=)\s*18\b|isin\(\[?[^)]*\b18\b[^)]*\)|\[18\]|\b18\s*번|topic.?18)", [ROOT], limit=25)
    print("\n판독: (a)에서 다른 표(예: 임베딩 클러스터링 원본)에 18번이 많으면 F1을 만드는 과정에서 18번 논문이 빠졌거나 번호가 재할당된 것입니다. 모든 표에서 1편이면 클러스터링 단계에서 이미 1편짜리 군집이었던 것입니다.")
    print("(c)의 셀이 18번을 제외하거나 재매핑하는 코드이면 그 셀이 원인입니다. 어느 쪽도 없으면 '원인 미확인'으로 두세요.")

# ================================================================ [4]
NONRES = re.compile(r"(?i)\b(editorial|editor.s note|preface|foreword|special issue|call for papers|book reviews?|books? of interest|bibliograph\w*|corrigendum|erratum|errata|in memoriam|obituary|table of contents|announcement|guide for authors|reviewers?\b.*(thank|list)|index to|front cover|ieee transactions on)")
REVIEW = re.compile(r"(?i)\b(review|survey|state[- ]of[- ]the[- ]art|meta-analysis|scientometric|bibliometric|tutorial|overview|roadmap|guide)\b")
WIRELESS = re.compile(r"(?i)(wireless|radio|\b5g\b|\b6g\b|\blte\b|mimo|antenna|channel|spectrum|cellular|reconfigurable intelligent|\bris\b|v2x|handover|beamform|propagation|signal|broadband)")

def s4():
    hdr("[4] 토픽 38 / 49 구성")
    from sklearn.feature_extraction.text import TfidfVectorizer
    F1 = f1_full(); FX = F1[F1.pid.isin(set(m.load_F1X().pid))]; out = []
    for k in (38, 49):
        d = FX[FX.topic150 == k].copy(); d["title"] = d.title.fillna(""); d["bracket"] = d.title.str.contains(r"\[[^\]]+\]"); d["nonres"] = d.title.str.contains(NONRES); d["review"] = d.title.str.contains(REVIEW); d["wireless"] = d.title.str.contains(WIRELESS)
        d["period"] = pd.cut(d.year, [1899, 2004, 2009, 2014, 2019, 2025], labels=["<=2004", "2005-09", "2010-14", "2015-19", "2020-25"])
        print(f"\n[토픽 {k}] {len(d):,}편 | 저널 상위 {d.venue_short.value_counts().head(6).to_dict()} | 기간 {d.period.value_counts().sort_index().to_dict()}")
        print(f"  제목 규칙 지표(대략적 대용치, 정밀도 미검증): 비연구 표지어 {d.nonres.mean():.1%}, 대괄호 칼럼 {d.bracket.mean():.1%}, 리뷰·서베이 어휘 {d.review.mean():.1%}, 무선통신 어휘 {d.wireless.mean():.1%}")
        top = d.venue_short.value_counts().index[0]; d["group"] = np.where(d.venue_short == top, f"{top}", "그 외 저널")
        print(d.groupby("group")[["nonres", "bracket", "review", "wireless"]].mean().round(3).assign(n=d.groupby("group").size()).to_string())
        for gname, g in d.groupby("group"):
            if len(g) < 20: continue
            vec = TfidfVectorizer(stop_words="english", ngram_range=(1, 2), min_df=3); X = vec.fit_transform(g.title); mu = np.asarray(X.mean(axis=0)).ravel(); terms = np.array(vec.get_feature_names_out())
            print(f"  [{gname}] 상위 용어:", ", ".join(terms[np.argsort(-mu)[:15]]))
        rng = np.random.default_rng(SEED)
        for gname, g in d.groupby("group"):
            idx = rng.choice(len(g), size=min(N_SAMPLE_PER_GROUP, len(g)), replace=False); s = g.iloc[idx][["pid", "venue_short", "year", "title"]].copy()
            s["topic"] = k; s["group"] = gname; s["group_size"] = len(g); s["sample_size"] = len(s); s["label"] = ""; out.append(s)
    path = S2 / "s2_open_topic38_49_manual.csv"
    if path.exists():
        L = pd.read_csv(path, dtype=str, keep_default_na=False, encoding="utf-8-sig"); done = L[L.label.str.strip() != ""]
        if len(done):
            print(f"\n기존 라벨 파일 읽음: 라벨 {len(done)}건. 라벨 코드: R 연구, N 비연구(칼럼·편집·서지), V 리뷰·서베이, O 기타")
            done = done.assign(w=done.group_size.astype(float) / done.sample_size.astype(float))
            for (k, lab), g in done.groupby(["topic", "label"]): pass
            for k, g in done.groupby("topic"):
                n = len(g); t = []
                for lab, gg in g.groupby("label"): t.append((lab, len(gg), round(gg.w.sum() / g.w.sum(), 3), tuple(round(x, 3) for x in wilson(len(gg), n))))
                print(f"  토픽 {k}: n={n} | (라벨, 건수, 가중 비율, 비가중 Wilson 95%)", t)
            print("  이 비율이 '군집 단위 제외 여부'를 정하는 근거입니다. 결정은 S3 사전등록 부록에 결과를 보기 전 규칙으로 적으세요."); return
    pd.concat(out).to_csv(path, index=False, encoding="utf-8-sig")
    print(f"\n표본 저장: {path.name} ({sum(len(x) for x in out)}건). label 열에 R/N/V/O를 채운 뒤 이 항목을 다시 실행하면 구성비가 나옵니다.")

# ================================================================ [5]
def s5():
    hdr("[5] T-ITS 2008 다국적 공저: Scimago 이탈 검정 + 직접 재계산")
    I = pd.read_csv(SC / "intl_collab_check.csv"); cols = [c for c in ["year", "n_known", "documents", "ours_multi_pct", "intl_collab_pct", "usable"] if c in I.columns]
    print(I[(I.venue_short == "T-ITS") & I.year.between(2005, 2012)][cols].round(1).to_string(index=False))
    rows = []
    for v, g in I[I.usable].groupby("venue_short"):
        g = g.set_index("year").sort_index()
        for y in g.index:
            if (y - 1 in g.index) and (y + 1 in g.index):
                for nm, col, ncol in [("ours", "ours_multi_pct", "n_known" if "n_known" in g else "documents"), ("scimago", "intl_collab_pct", "documents")]:
                    nb = (g.at[y - 1, col] + g.at[y + 1, col]) / 2; x = g.at[y, col]; n = g.at[y, ncol]
                    if pd.isna(nb) or pd.isna(x) or pd.isna(n) or n < 1: continue
                    p = min(max(nb / 100, 0.02), 0.98); se = 100 * np.sqrt(p * (1 - p) / n); rows.append(dict(series=nm, venue_short=v, year=y, value=x, neighbor_mean=nb, dev=x - nb, n=n, z=(x - nb) / se))
    D = pd.DataFrame(rows)
    if len(D):
        print("\n이웃 연도 평균 대비 이탈(이항 표준오차 기준 z). |z|>3 비율:", D.groupby("series").z.apply(lambda s: round((s.abs() > 3).mean(), 3)).to_dict(), "| 칸 수", D.groupby("series").size().to_dict())
        print("T-ITS 2008:\n", D[(D.venue_short == "T-ITS") & (D.year == 2008)].round(2).to_string(index=False))
        print("series별 |z| 상위 6:\n", D.reindex(D.z.abs().sort_values(ascending=False).index).groupby("series").head(6).sort_values(["series", "z"], key=lambda s: s if s.dtype == object else -s.abs()).round(2).to_string(index=False))
    F = m.load_F1X(); A = m.build_authorship(F); T = F[(F.venue_short == "T-ITS") & F.year.between(2006, 2011)]; At = A[A.pid.isin(set(T.pid))]
    def cset(s): return {c for x in s for c in str(x).split("|") if c}
    per = At.groupby("pid").agg(cm=("cs_main", cset), cr=("cs_rec", cset), n_known=("cs_main", lambda s: int((s != "").sum())), n_auth=("rank", "size")).reset_index().merge(T[["pid", "year", "title"]], on="pid")
    per["multi_main"] = per.cm.map(len) >= 2; per["multi_rec"] = per.cr.map(len) >= 2
    def rep(d): k = d[d.cm.map(len) >= 1]; return pd.Series(dict(papers=len(d), known_ge1=len(k), multi=int(d.multi_main.sum()), pct_of_known=round(100 * d.multi_main.sum() / max(len(k), 1), 1), pct_all_authors_known=round(100 * d[d.n_known == d.n_auth].multi_main.mean(), 1) if (d.n_known == d.n_auth).any() else np.nan, pct_rec=round(100 * d.multi_rec.sum() / max(len(k), 1), 1)))
    print("\nT-ITS 연도별 직접 재계산(우리 코퍼스, 소속 국가 2개 이상 논문 비율):\n", per.groupby("year").apply(rep).to_string())
    p8 = per[per.year == 2008].copy(); p8["countries"] = p8.cm.map(lambda s: "|".join(sorted(s))); p8["countries_rec"] = p8.cr.map(lambda s: "|".join(sorted(s)))
    aff = At.groupby("pid").raw_aff.apply(lambda s: " ;; ".join(str(x)[:90] for x in s if isinstance(x, str) and x)).rename("raw_aff") if "raw_aff" in At else None
    if aff is not None: p8 = p8.merge(aff, on="pid", how="left")
    p8[["pid", "title", "n_auth", "n_known", "countries", "countries_rec"] + (["raw_aff"] if aff is not None else [])].to_csv(S2 / "s2_open_tits2008_papers.csv", index=False, encoding="utf-8-sig")
    print(f"\nT-ITS 2008 논문 {len(p8)}편 목록 저장: s2_open_tits2008_papers.csv (다국적으로 판정된 {int(p8.multi_main.sum())}편의 raw_aff를 직접 읽어 보세요)")
    print("판독: (1) Scimago 시계열에서 2008이 이웃 대비 |z|가 크고 우리 시계열은 매끄러우면 Scimago 쪽 이상치 가능성이 큽니다. (2) 우리 값이 Scimago와 다른 방향으로 일관되게 낮으면 소속 누락 가능성. 어느 쪽이든 원인 단정은 금지, '2008 한 칸 불일치, 인접 연도는 일치'로 기록.")

# ================================================================ [6]
def s6():
    hdr("[6] 수작업 검토 진행 현황")
    p = S2 / "clean_review_samples.csv"
    if p.exists():
        d = pd.read_csv(p, dtype=str, keep_default_na=False, encoding="utf-8-sig"); done = d[d.correct.isin(["0", "1"])]
        print(f"clean_review_samples: 전체 {len(d)}건, 라벨 완료 {len(done)}건 ({len(done) / max(len(d), 1):.0%}) | 규칙별 대기 {d[d.correct.str.strip() == ''].rule.value_counts().to_dict()}")
        if len(done):
            t = done.assign(ok=done.correct == "1").groupby("rule").ok.agg(n="size", correct="sum").reset_index(); t["rate"] = (t.correct / t.n).round(3); t[["lo", "hi"]] = [wilson(int(r.correct), int(r.n)) for r in t.itertuples()]
            print(t.round(3).to_string(index=False)); print("판독: f_dup의 정확도가 낮으면 '표시=제거 아님'을 근거로 제거 전후 비교에서 f_dup을 분리하세요(규칙 변경 아님).")
    else: print("clean_review_samples.csv 없음")
    q = S2 / "unk_resid_2024_25_sample.csv"
    if q.exists():
        u = pd.read_csv(q, dtype=str, keep_default_na=False, encoding="utf-8-sig"); print(f"\nunk_resid 표본: 전체 {len(u)}건, note 완료 {int((u.note.str.strip() != '').sum())}건 | 분포 {u[u.note.str.strip() != ''].note.value_counts().to_dict()}")
    print("수작업 도구: s2_review_samples.py (TARGET='clean' 또는 'unk')")

# ================================================================ [7]
def s7():
    hdr("[7] BASE 정의와 F2/F3/F4/F5/F7 구조, 임베딩·토픽 파일")
    nb = ROOT / "(2) 데이터셋정의.ipynb"
    if nb.exists():
        for pat in [r"^\s*BASE\s*=", r"^\s*PERIODS\s*=", r"^\s*MAIN0\s*=", r"^\s*FIX\s*="]: show_cell(nb, pat, max_lines=40)
    else: print("(2) 데이터셋정의.ipynb 없음: grep_nb로 BASE 정의 검색"); grep_nb(r"\bBASE\s*=", [ROOT], limit=10)
    rows = []
    for fn in ["F2_author_paper.csv", "F3_author_period_venue_topic.csv", "F4_authors.csv", "F5_continuing_authors.csv", "F7_oa_author_ids.csv", "F2_risk_names.csv"]:
        p = DATA / fn
        if not p.exists(): print(f"  {fn}: 없음"); continue
        d = pd.read_csv(p, low_memory=False); print(f"\n[{fn}] {len(d):,}행 {d.shape[1]}열")
        for c in d.columns:
            s = d[c]; ex = s.dropna().iloc[0] if s.notna().any() else ""; rows.append(dict(file=fn, column=c, dtype=str(s.dtype), null_rate=round(float(s.isna().mean()), 4), n_unique=int(s.nunique()), example=str(ex)[:50]))
        print(pd.DataFrame(rows[-d.shape[1]:])[["column", "dtype", "null_rate", "n_unique", "example"]].to_string(index=False))
        if fn.startswith("F3"): print("  period 분포:", d["period"].value_counts().to_dict() if "period" in d else "열 없음")
    pd.DataFrame(rows).to_csv(S2 / "s2_open_f_columns.csv", index=False, encoding="utf-8-sig")
    try:
        F4 = pd.read_csv(DATA / "F4_authors.csv", usecols=["name_norm", "n_papers", "first_year"]); F3 = pd.read_csv(DATA / "F3_author_period_venue_topic.csv", usecols=["name_norm"]); F2 = pd.read_csv(DATA / "F2_author_paper.csv", usecols=["name_norm"])
        a4, a3, a2 = set(F4.name_norm), set(F3.name_norm), set(F2.name_norm)
        print(f"\n저자 수: F4 {len(a4):,} | F3 {len(a3):,} | F2 {len(a2):,} | F4∩F2 {len(a4 & a2):,} | F4-F2 {len(a4 - a2):,} | F2-F4 {len(a2 - a4):,} | F3-F4 {len(a3 - a4):,}")
        print("해석: F4가 F2 저자와 거의 같으면 BASE는 F2(주제 논문, 위험군 제외) 기반이고, F4 - F2가 크면 주제 없는 논문 저자도 포함합니다.")
    except Exception as e: print("저자 집합 비교 실패:", e)
    print("\n임베딩·토픽 관련 파일:")
    for p in sorted((ROOT / "analysis_data").rglob("*")):
        if p.is_file() and re.search(r"(?i)embed|specter|emb_|topic|cluster|label|kmeans", p.name) and ".ipynb_checkpoints" not in str(p):
            sh = ""
            if p.suffix == ".npy":
                try: a = np.load(p, mmap_mode="r", allow_pickle=False); sh = f" shape {a.shape} dtype {a.dtype}"
                except Exception: pass
            print(f"  {p.relative_to(ROOT)} ({p.stat().st_size / 1e6:.1f}MB){sh}")

# ================================================================ [8]
def find_keys(o, pat, path=""):
    rx = re.compile(pat, re.I)
    if isinstance(o, dict):
        for k, v in o.items():
            if rx.search(str(k)): print(f"  PREREG {path}/{k}: {str(v)[:400]}")
            find_keys(v, pat, f"{path}/{k}")
    elif isinstance(o, list):
        for i, v in enumerate(o[:50]): find_keys(v, pat, f"{path}[{i}]")

def s8():
    hdr("[8] 위험군 변형 7종: 정의 위치, 변형별 점검, 이름 빈도별 점검")
    try: find_keys(json.loads((S0 / "PREREG.json").read_text(encoding="utf-8")), r"risk|variant|ambig|heavy")
    except Exception as e: print("PREREG 읽기 실패:", e)
    print("\n노트북에서 변형 정의 검색:"); grep_nb(r"(variant_names|risk_variant|RISK_VARIANT|VARIANTS\s*=|ambiguous\s*=|heavy\s*=|\"ambiguous\"|\"heavy\")", [ROOT], limit=25)
    print("\n위험군·변형 관련 파일(analysis_data, paper1):")
    cand = []
    for b in [ROOT / "analysis_data", ROOT / "paper1"]:
        for p in sorted(b.rglob("*")):
            if p.is_file() and re.search(r"(?i)risk|variant|ambig", p.name) and ".ipynb_checkpoints" not in str(p) and p.suffix in (".csv", ".gz", ".parquet", ".json"):
                cols = m_headers(p); cand.append(p); print(f"  {p.relative_to(ROOT)} | 열 {cols[:12]}")
    D = pd.read_csv(DATA / m.F7_FILE, low_memory=False).dropna(subset=["name_norm"]); X = pd.DataFrame({"n_ids": D.groupby("name_norm").oa_id.nunique(), "n_rows": D.groupby("name_norm").size()}); X["multi"] = X.n_ids >= 2
    R = pd.read_csv(DATA / m.RISK_FILE); flagcols = [c for c in R.columns if c != "name_norm" and (R[c].dtype == bool or set(R[c].dropna().unique()) <= {0, 1, True, False})]
    print("\n위험군 파일의 플래그 열:", flagcols, "| 그 외 열:", [c for c in R.columns if c not in flagcols and c != "name_norm"])
    def ev(flag):
        tp = int((X.multi & flag).sum()); return dict(n_flag=int(flag.sum()), tp=tp, precision=round(tp / max(int(flag.sum()), 1), 3), recall=round(tp / max(int(X.multi.sum()), 1), 3))
    rows = []
    for c in flagcols: rows.append(dict(flag=c, **ev(X.join(R.set_index("name_norm")[c]).get(c).fillna(False).astype(bool))))
    for c in [c for c in R.columns if c not in flagcols and c != "name_norm" and R[c].nunique() <= 12]:
        for v in R[c].dropna().unique(): rows.append(dict(flag=f"{c}={v}", **ev(X.index.isin(R.loc[R[c] == v, "name_norm"]))))
    print("\n플래그/변형별 정밀도·재현율(근사 정답 ID>=2, 2020년대 F7):\n", pd.DataFrame(rows).to_string(index=False))
    fl = X.join(R.set_index("name_norm")[flagcols].astype(bool).any(axis=1).rename("flag")).fillna({"flag": False}); fl["flag"] = fl.flag.astype(bool)
    fl["bin"] = pd.cut(fl.n_rows, [0, 1, 2, 4, 9, 19, 10 ** 9], labels=["1", "2", "3-4", "5-9", "10-19", "20+"])
    T = fl.groupby("bin", observed=True).apply(lambda g: pd.Series(dict(names=len(g), flagged_share=round(g.flag.mean(), 3), multi_share=round(g.multi.mean(), 3), precision=round((g.flag & g.multi).sum() / max(g.flag.sum(), 1), 3), recall=round((g.flag & g.multi).sum() / max(g.multi.sum(), 1), 3))))
    print("\nF7 이름 빈도(행 수)별 점검(순환성·빈도 의존성 확인):\n", T.to_string())
    print("판독: flagged_share와 multi_share가 같은 빈도 구간에서 함께 오르면 플래그가 빈도 휴리스틱일 가능성이 높고, 정밀도는 '이름이 흔하다'로도 설명됩니다. 변형 7종의 정의 열이 위 검색에서 안 나오면 변형별 분해는 불가이므로 '변형 정의 미확보'로 기록하세요.")

def m_headers(p):
    try:
        if p.suffix == ".parquet":
            import pyarrow.parquet as pq; return pq.ParquetFile(p).schema.names
        if p.suffix == ".json": return list(json.loads(p.read_text(encoding="utf-8")).keys())[:12]
        return list(pd.read_csv(p, nrows=0).columns)
    except Exception: return []

# ================================================================ 실행
FN = {"1": s1, "2": s2, "3": s3, "4": s4, "5": s5, "6": s6, "7": s7, "8": s8}
for k in (list(FN) if PHASE == "all" else [PHASE]):
    try: FN[k]()
    except Exception:
        print(f"\n⛔ 항목 {k} 오류(아래 전체 출력을 붙여 주세요):"); traceback.print_exc()
print("\n완료. 결과를 본 뒤 규칙을 바꾸면 사후 변경입니다. 규명 결과는 CHANGELOG_S2에 '원인 확인' 또는 '원인 미확인'으로 새 줄에 기록하세요(기존 줄은 수정하지 않음).")


[1] 창 안 논문 수 대조: S1 81,485 / F1X 81,645 / authorship 81,441
S1 raw JSONL 고유 DOI 83,096
                                        건수  81,485와 같음  81,441과 같음
F1 창 전체(제외 전)                        81826       False       False
F1X 창                                81645       False       False
F1X 창 & DOI 있음                       81485        True       False
F1X 창 & pid가 authorship_level에 있음    81441       False        True
F1X 창 & pid가 country_authorship에 있음  81645       False       False
authorship_level 고유 pid(전체 연도)       81441       False        True
country_authorship 고유 pid(전체 연도)     81645       False       False
F1X 창 & DOI가 S1 raw JSONL에 있음        81431       False       False
coverage_audit n_papers 합(2007+)     81645       False       False

기준 수치와 일치하는 정의: ['F1X 창 & DOI 있음', 'F1X 창 & pid가 authorship_level에 있음', 'authorship_level 고유 pid(전체 연도)']

창 안 F1X 논문의 결측 유형별 건수(81,645 기준): {'no_doi': 160, 'no_auth': 204, 'no_cty': 0, 'no_raw': 214, 'no_topic': 12391}
목표 차이 160(= 81,645 - 

In [40]:
#

In [ ]:
# ## 총평

# 이번 규명 스크립트로 열린 항목 8개 중 4개가 닫히거나 크게 정리됐습니다. 그중 세 건(459, 토픽 18, 위험군 정밀도)은 이전 판독 일부를 고쳐야 합니다. 일부는 제가 앞에서 "원인 미확인"이나 "재현 안 됨"으로 둔 것인데, 노트북 안에 이미 답이 있었습니다. 실질적으로 남는 문제는 위험군 검증이 순환적이라는 점 하나입니다.

# ## 항목별 판독

# **[1] 창 안 논문 수: 닫혔습니다.**
# - 81,485는 창 안 F1X 중 DOI가 있는 논문 수이고, 81,645에서 DOI 없는 160편을 뺀 값입니다.
# - 81,441은 `authorship_level`에 있는 논문 수입니다. 창 안 F1X에서 authorship이 없는 204편(DOI 없음 160, DOI 있으나 authorship 없음 44)을 뺀 값입니다. 제 산술이고, 출력의 두 건수(160, 204)와 맞습니다.
# - 세 숫자는 서로 다른 정의라서 모순이 아닙니다. `country_authorship`은 81,645편 전부를 담고, authorship이 없는 논문은 미상 질량으로 들어갑니다.
# - 덧붙일 점이 하나 있습니다. DOI가 있는데 S1 raw JSONL에서 매칭되지 않는 논문이 54편입니다. DOI 정규화 차이일 수도 있지만 확인된 것은 아닙니다. 영향은 0.07% 수준이라 기록만 하세요.

# **[2] f_dup 441 대 459: 재현 안 됨이 아니라 정의 차이일 가능성이 큽니다.**
# - 격자에는 원문 그대로의 제목(정규화 없음) 조합이 없었습니다.
# - EDA 셀 16의 검증 문구는 "제목이 같은 논문이 459행(고유 제목 기준 중복 266건)"입니다. 266은 `F1.title.duplicated().sum()`, 곧 첫 건 뒤 중복입니다.
# - 중복군이 193개라면 중복군 전체 행은 266+193=459가 됩니다. 이는 제 산술에서 나온 가설입니다.
# - 이 가설이 맞으면 459는 저널·연도와 무관한 원문 제목 중복이라서, 현행 `f_dup`(저널·연도·정규화 제목)과 정의가 다릅니다. 다음 한 줄로 확인됩니다.

# ```python
# t = f1_full().title.dropna(); print(int(t.duplicated().sum()), int(t.duplicated(keep=False).sum()))  # 266, 459 기대
# ```

# - 일치하면 CHANGELOG에 "정의 차이로 확인, 규칙 변경 없음"이라고 적고 닫으세요.

# **[3] 토픽 18번: 원인이 이미 EDA에 기록돼 있습니다.**
# - EDA 셀 18에 "18번은 논문 1편(1974년)짜리 빈 주제이고 실질 주제는 149개"라고 적혀 있습니다.
# - 1편은 `paper_ml_features`, `paper_ml_scores`, F1, F6에서 모두 같습니다. 번호 재할당 가설은 지지되지 않습니다. 클러스터링 단계에서 이미 1편짜리 군집이었다고 읽는 것이 단순합니다.
# - 그러면 계획서 2.8의 "주제 18번 제외"는 사실상 공규칙입니다. 동결된 `topic_excluded=[18]`은 그대로 두고, 이 사실만 기록하세요.
# - S3 L5에 "18번을 규칙이 잡는지 검증"이 있는데 18번은 비어 있습니다. 38번(VT Mag 칼럼)과 49번(서지·도서 안내)을 검증 대상으로 삼는 편이 의미가 있습니다. 이 변경은 S3 사전등록 부록(결과 열람 전)에서 하세요.
# - 유효 주제는 149개입니다. K 민감도와 엔트로피 정규화에서 150과 149를 구분해 쓰세요.

# **[4] 토픽 38, 49: 규칙 지표로는 판정되지 않았고, 수작업이 필요합니다.**
# - 38번은 무선통신 어휘가 70.9%(VT Mag 76.9%, 그 외 58.8%)이고 대괄호 칼럼은 13.4%입니다. 대부분 무선통신 연구이고 칼럼이 일부 섞인 것으로 보입니다. 군집 제외는 부적절합니다.
# - 49번은 비연구 표지어 4.0%, 리뷰 어휘 3.1%입니다. 정규식 대용치라서 하한으로만 읽으세요. 그 외 저널 상위 용어에 `books`, `page`, `tgrg`가 있어 서지·안내 잔여 항목이 있습니다. CACIE 쪽은 구조·건설 공학 연구입니다. 한 군집에 잡다한 것이 모인 구성이라 "복합 토픽" 후보가 맞습니다.
# - 49번은 F1에서 582편, F1X에서 524편이라 S0 제외가 58편을 이미 걷어 냈습니다.
# - 수작업 표본 120편(`s2_open_topic38_49_manual.csv`, 그룹당 30편)의 `label` 열을 R/N/V/O로 채우고 다시 실행하면 구성비가 나옵니다. 표본이 작으니 구간을 넓게 읽으세요.

# **[5] T-ITS 2008: Scimago 쪽 이상치로 읽는 것이 자연스럽습니다.**
# - 우리 시계열의 2008년 z는 -0.81로 매끄럽습니다. Scimago는 z=4.59이고 61.3% 대 이웃 평균 33.8%입니다.
# - 소속을 모르는 저자만 보는 `pct_all_authors_known`은 21.1%, 복구판은 24.2%입니다. 소속 누락으로 40%p 격차를 설명하기는 어렵습니다.
# - 다만 Scimago의 T-ITS 이탈은 2021~2023년에도 큽니다(z 4.45~-6.50). 이항 표준오차는 실제 변동을 과소평가해서 z가 부풀려지므로 대략적인 선별용으로만 쓰세요. Scimago 시계열이 우리 것보다 거칠다(|z|>3 비율 3.7% 대 0.9%)는 정도가 안전한 결론입니다.
# - "2008 한 칸 불일치, 인접 연도는 일치, 원인 미확인"으로 기록하고 닫으세요. 논문 13편의 `raw_aff`를 읽는 것은 선택입니다.

# **[6] 수작업 검토: 0/91, 0/30입니다.**
# - 병목은 코드가 아니라 사람의 시간입니다. 91건이면 한 시간 안팎이고, `f_topic18` 1건은 연구 논문이라 `correct=0`입니다.
# - `s2_review_samples.py`로 f_dup 30건부터 하세요. 같은 저널·연도·제목 레코드가 함께 보입니다.

# **[7] 구조 점검: BASE 정의는 아직 못 봤습니다.**
# - 스크립트가 셀의 앞 45줄만 출력해서, 매칭된 셀 1에서 `BASE =`가 나오지 않았습니다. 이 노트북에서 직접 확인하세요.

# ```bash
# grep -n "BASE" "(2) 데이터셋정의.ipynb" | head
# ```

# - 주 기간은 `MAIN0, MAIN1 = "P10_14", "P20_24"`로 확인됩니다. 이전에 결정이 필요하다고 한 F4의 `m_P10_14`/`m_P20_24`가 맞습니다.
# - 저자 집합은 F3 ⊂ F2 ⊂ F4(F2−F4=0, F3−F4=0)입니다. F4−F2 38,711명은 주제 없는 논문의 저자와 위험군 이름으로 보이는데, 이 구성은 확인하지 않았습니다.
# - F3의 기간은 겹칩니다. `P12_14`는 `P10_14`에, `P22_24`는 `P20_24`에 포함됩니다. 저자 층 코드에서 기간별 합을 낼 때 이중 계산을 조심하세요.
# - F3에는 149개 토픽만 있어 18번이 없습니다.
# - 임베딩은 `analysis_data` 아래에는 없습니다. 노트북 주석상 `transport-atlas-data/paper_emb.bin`(int8 양자화)과 `paper_index.json`이 원본이고, `RECOMPUTE_TOPICS=True` 경로에 k-means 코드가 있습니다. 2.9 K 민감도는 이 파일로 가능해 보입니다.

# **[8] 위험군 검증: 순환성이 확인됐습니다. 이번 결과에서 가장 중요합니다.**
# - 노트북 `Atlas_데이터수집` 셀 4에서 `ambiguous`는 이름에 매칭된 OpenAlex ID 수가 2개 이상일 때로 정의됩니다. 제가 쓴 근사 정답(F7에서 이름당 ID 2개 이상)과 같은 종류의 정보입니다. 정밀도 0.87은 독립 검증이 아니라 두 OpenAlex 측정의 일치도입니다.
# - 앞에서 가설로만 말씀드린 순환성이 코드로 확인된 셈입니다. 이 방식은 계획서 2.2가 정한 것이니, 계획서 자체의 약점이기도 합니다.
# - 빈도별 표가 보여 주는 것은 다음과 같습니다.
#   - 재현율은 행 수가 많은 이름에서 높습니다(10~19행 0.68, 20행 이상 0.84).
#   - 행 2~4개 이름에서는 낮습니다(0.04~0.19).
#   - 전체 재현율 0.30은 희귀 이름이 대다수라서 생긴 평균입니다.
# - 따라서 "동명이인 재현율 0.30"으로 보고하면 안 됩니다. "OpenAlex ID 분할 기반 일치도(이름 빈도별)"로 정의해서 보고하세요. 독립 정답이 필요하면 ORCID가 있는 저자의 이름-ORCID 분할이나 수작업 표본이 대안입니다.
# - 변형 7종은 PREREG `variant_defs`에 정의가 있습니다. `F2_risk_names`에는 base만 들어 있지만 재구성은 가능합니다. 변형 7종에는 `heavy_8`, `heavy_20`이 포함됩니다. 데이터셋정의 노트북 63행의 `mx`(이름×연도 최대 논문 수)를 다시 만들어 `ambiguous ∪ {mx≥8}` 같은 집합을 만들면 됩니다. 이미 있는 `profile6/name_variants.csv`(열: 변형, 기간, 제외이름, conv, gap)는 이전 분석의 변형 결과라 대조에 쓸 수 있습니다.

# ## CHANGELOG_S2에 남길 줄 (규칙 변경 없음)

# > 창 안 논문 수: 81,485=창 안 F1X 중 DOI 보유, 81,441=authorship 있는 논문(authorship 없는 204편 = DOI 없음 160 + DOI 있으나 없음 44). 정의 차이로 확인. f_dup 459는 EDA 기준(원문 제목 중복 266건, 중복군 전체 행 459 추정)과 정의가 달라 f_dup 441과 비교 대상이 아님(가설, 위 한 줄로 확인 예정). 토픽 18번은 EDA에 기록된 1편짜리 빈 군집(유효 주제 149). 38번은 무선통신 연구 중심(VT Mag 칼럼 일부), 49번은 복합 군집으로 수작업 표본 대기. T-ITS 2008은 Scimago 쪽 이탈로 보이며 원인 미확인. 위험군 `ambiguous`는 OpenAlex ID 2개 이상으로 정의되어 F7 기반 정밀도·재현율은 독립 검증이 아님.

# ## 다음에 할 일

# 1. **한 줄 확인 두 개**: `t.duplicated` 수치(266, 459)와 `grep BASE`입니다.
# 2. **수작업 병행**: `clean_review_samples.csv`와 토픽 38/49 표본 120건입니다.
# 3. **S2 마무리**: 위험군 변형 7종 분해와 빈도별 재현율을 보고서(`risk_names_report.md`)로 쓰면 2.2가 닫힙니다. 그다음 `s2_finalize.py`로 commit_dry를 거쳐 커밋합니다. 커밋이 실제로 만들어졌는지는 아직 확인되지 않았습니다.
# 4. **S3 사전등록 부록(S3-0)**: 위 3번(L5 검증 대상 변경, 49번과 38번의 복합 토픽 처리 규칙)을 반영하고 동결하세요. 코드를 짜기 전에 `BASE`와 팀 규모·신규 저자 정의를 확인하면 L4도 막히지 않습니다.

# 시간 상한을 두자는 앞선 제안은 유효합니다. 이번에 닫은 항목들은 대부분 기존 노트북에 답이 있었으니, 앞으로는 열린 항목을 만들기 전에 EDA와 데이터셋정의 노트북부터 검색하는 것이 효율적입니다.

In [ ]:
# 변수 사전 CSV까지 만들었고, 파일 전달만 남았습니다.네 가지 산출물 중 두 가지는 완성했고, 두 가지는 아직 실행하지 않은 스크립트입니다.

# **완성한 것**
# - **`missingness_report.md`**: 이미 나온 출력만으로 썼고, 새로 계산한 값은 없습니다. 코호트·팀 규모별 결측, 경계 분석, 상호작용 모형은 미실시로 8절에 적었습니다.
# - **`variable_dictionary.csv`**: 66행이고, 확인 28행, 부분 확인 24행, 미확인 11행입니다. 정의는 점검 출력과 노트북에서 확인된 것만 넣었습니다. F5의 `em0`, `dc0`, `d_em`, `d_dc` 같은 열은 정의를 추측하지 않고 "미확인"으로 두었고, F4의 `continuing_main`·`entrant_main` 임계값도 미확인입니다. F1은 전체 열 목록을 본 적이 없어 코드에서 쓰인 열만 넣었습니다. 생성 노트북에서 해당 열의 정의를 채워 넣으세요.

# **실행이 필요한 것 (결과 파일은 아직 없음)**
# - **`s2_title_classifier.py`**: 실행하면 `title_classifier_report.md`, `title_topic_f1.csv`, `title_pred_missing_topic.csv`가 `02_S2`에 생깁니다. 저널×연도 그룹 교차검증을 쓰고, 결측 논문 예측은 민감도 전용입니다. 문법 검사만 했습니다.
# - **`s2_k_sensitivity.py`**: 실행하면 `K_sensitivity.csv`가 생깁니다. K ∈ {75, 150, 300} × seed 5회 재클러스터링에서 ARI, NMI와 C1 후보(주제 유효수, HHI)의 추세 부호를 계산합니다. 문법 검사만 했습니다.

# **결과를 읽을 때 알아 둘 점**
# - 두 스크립트의 임계값(최소 토픽 논문 수 20, tau 0.5, F1 하한 0.30)은 제 임의 값이고 동결 부록에 없었습니다. 실행 전에 CHANGELOG_S2에 사전 명시로 적으세요.
# - K 민감도는 핵심 결과 C2, C3, C5의 부호 유지를 판정하지 못합니다. 그 통계는 S5~S7에서 나오므로, 같은 재클러스터링 라벨로 그때 다시 계산해야 합니다. 이 점은 2.9의 통과 기준을 "이월"로 닫아야 한다는 뜻입니다.
# - 임베딩 위치와 행 구조(앞 4바이트 scale + int8)는 노트북 주석에서 가져온 가정이라 확인하지 못했습니다. 파일을 못 찾으면 스크립트가 후보 경로를 출력하고 멈춥니다.
# - 기존 `topic150`이 같은 전처리의 k-means에서 나왔는지 모르므로, 기준 대비 ARI는 "재현"이 아니라 "일치도"로 읽으세요.

# 두 스크립트의 출력을 붙여 주시면 판독하겠습니다. 그 전까지는 S2의 2.7과 2.9가 열려 있습니다.

In [41]:
# -*- coding: utf-8 -*-
# =====================================================================================
# S2 항목 2.7 제목 기반 토픽 예측 (분류기, LLM 아님). OpenAlex 호출 없음, 동결 부록 수정 없음.
# 산출: title_classifier_report.md, title_topic_f1.csv, title_pred_missing_topic.csv (paper1/02_S2)
#  - 학습: 토픽 보유 논문(2000+)의 제목 TF-IDF(1~2gram) + 선형 분류기(SGD, modified_huber)
#  - 검증: 저널x연도 그룹 5겹 교차검증(한 칸 전체가 검증 쪽으로 빠짐) -> 새 저널·연도로의 일반화를 잰다
#  - 결측 논문 예측은 민감도 분석 전용. 예측확률 tau와 토픽 F1 하한은 아래 값으로 '결과를 보기 전에' 정한다.
# 아래 임계값(MIN_TOPIC_PAPERS, TAU_MAIN, F1_FLOOR)은 제 임의 출발값이며 부록에는 없던 값입니다.
# 실행 전에 CHANGELOG_S2에 '2.7 임계값 사전 명시(동결 부록에 없던 항목)'로 적어 두세요.
# 이 스크립트는 문법 검사만 했고 실제 자료로는 실행하지 않았습니다. 예상 소요: 수 분~수십 분.
# s2_measurement.py(v3)와 같은 폴더 또는 후보 경로에 두고 실행하세요.
# =====================================================================================
import importlib.util, warnings
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")

SEED = 2026
TOPIC_START = 2000
MIN_TOPIC_PAPERS = 20      # 학습 가능한 토픽의 최소 논문 수(미만이면 분류 대상에서 제외하고 보고)
N_FOLDS = 5
TAU_MAIN = 0.50            # 예측확률(modified_huber 점수 정규화, 보정되지 않음) 하한
F1_FLOOR = 0.30            # 교차검증 토픽 F1이 이 값 미만인 토픽의 예측은 민감도에서 제외
TAU_GRID = [0.3, 0.5, 0.7, 0.9]

ROOT = Path.home() / "Research/Minnesota_Transport"
cands = [Path.cwd() / "s2_measurement.py", ROOT / "s2_measurement.py", ROOT / "paper1/s2_measurement.py",
         ROOT / "paper1/02_S2/s2_measurement.py", Path.home() / "Downloads/s2_measurement.py"]
MOD = next((c for c in cands if c.exists()), None); assert MOD is not None, f"s2_measurement.py를 찾지 못했습니다: {[str(c) for c in cands]}"
_sp = importlib.util.spec_from_file_location("s2", MOD); m = importlib.util.module_from_spec(_sp); _sp.loader.exec_module(m)
S2 = m.S2; S2.mkdir(parents=True, exist_ok=True)

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import SGDClassifier
from sklearn.model_selection import GroupKFold
from sklearn.metrics import f1_score, accuracy_score

def fit_predict(tr_text, tr_y, te_text):
    vec = TfidfVectorizer(ngram_range=(1, 2), min_df=5, sublinear_tf=True, strip_accents="unicode", lowercase=True)
    Xtr = vec.fit_transform(tr_text); Xte = vec.transform(te_text)
    clf = SGDClassifier(loss="modified_huber", alpha=1e-5, max_iter=30, tol=None, random_state=SEED, n_jobs=-1)
    clf.fit(Xtr, tr_y); P = clf.predict_proba(Xte)
    return clf.classes_[P.argmax(axis=1)], P.max(axis=1)

F = m.load_F1X(); F = F[F.year >= TOPIC_START].copy(); F["title"] = F.title.fillna("").astype(str).str.strip()
F = F[F.title != ""]
L = F[F.topic150.notna()].copy(); L["topic"] = L.topic150.astype(int)
sz = L.topic.value_counts(); keep = sz[sz >= MIN_TOPIC_PAPERS].index; dropped = sz[sz < MIN_TOPIC_PAPERS]
L = L[L.topic.isin(keep)].reset_index(drop=True); L["grp"] = L.venue_short.astype(str) + "_" + L.year.astype(str)
print(f"학습 대상 {len(L):,}편, 토픽 {len(keep)}개 | 제외 토픽(논문 {MIN_TOPIC_PAPERS}편 미만) {dict(dropped)}")

pred = np.empty(len(L), dtype=int); pmax = np.zeros(len(L))
for k, (tr, te) in enumerate(GroupKFold(N_FOLDS).split(L, L.topic, L.grp)):
    p, q = fit_predict(L.title.values[tr], L.topic.values[tr], L.title.values[te]); pred[te] = p; pmax[te] = q
    print(f"  fold {k + 1}/{N_FOLDS} 완료 (검증 {len(te):,}편)")
L["pred"] = pred; L["pmax"] = pmax; L["ok"] = L.pred == L.topic

labels = sorted(keep); f1s = f1_score(L.topic, L.pred, labels=labels, average=None, zero_division=0)
T = pd.DataFrame(dict(topic=labels, n=[int(sz[t]) for t in labels], f1=f1s)); T["above_floor"] = T.f1 >= F1_FLOOR
T.to_csv(S2 / "title_topic_f1.csv", index=False)
acc = accuracy_score(L.topic, L.pred); mac = float(np.mean(f1s)); wf = f1_score(L.topic, L.pred, labels=labels, average="weighted", zero_division=0)
L["period"] = pd.cut(L.year, [1999, 2004, 2009, 2014, 2019, 2025], labels=["2000-04", "2005-09", "2010-14", "2015-19", "2020-25"])
by_period = L.groupby("period", observed=True).ok.agg(["size", "mean"]).round(3)
okt = set(T[T.above_floor].topic)
rows = []
for tau in TAU_GRID:
    c = L.pmax >= tau; c2 = c & L.pred.isin(okt)
    rows.append(dict(tau=tau, 적용률_tau만=round(float(c.mean()), 3), 정확도_tau만=round(float(L.ok[c].mean()), 3) if c.any() else np.nan,
                     적용률_tau_F1하한=round(float(c2.mean()), 3), 정확도_tau_F1하한=round(float(L.ok[c2].mean()), 3) if c2.any() else np.nan))
TT = pd.DataFrame(rows)

# 결측 논문 예측(민감도 전용)
M_ = F[F.topic150.isna()].copy(); n_miss = len(M_)
p, q = fit_predict(L.title.values, L.topic.values, M_.title.values)
M_["pred_topic"] = p; M_["pmax"] = q; M_["use_in_sensitivity"] = (M_.pmax >= TAU_MAIN) & M_.pred_topic.isin(okt)
M_[["pid", "year", "venue_short", "pred_topic", "pmax", "use_in_sensitivity"]].to_csv(S2 / "title_pred_missing_topic.csv", index=False)

def tbl(d, index=True):
    d = d.reset_index() if index else d
    h = "| " + " | ".join(map(str, d.columns)) + " |\n|" + "---|" * len(d.columns) + "\n"
    return h + "\n".join("| " + " | ".join(str(v) for v in r) + " |" for r in d.itertuples(index=False))

low = T.sort_values("f1").head(10)
md = f"""# title_classifier_report: 제목 기반 토픽 예측 (S2 항목 2.7)

이 보고서는 `s2_title_classifier.py`가 실행 결과로 자동 생성했습니다(SEED={SEED}). 분류기는 LLM이 아니라 TF-IDF(1~2gram, 최소 문서빈도 5) + 선형 분류기(SGD, modified_huber)입니다.

## 1. 설정 (결과를 보기 전에 정한 값, 동결 부록에는 없던 항목)

| 항목 | 값 |
|---|---|
| 학습 표본 | F1X 중 {TOPIC_START}년 이후, 토픽 보유, 제목 있음: {len(L):,}편 |
| 학습 대상 토픽 | {len(keep)}개 (논문 {MIN_TOPIC_PAPERS}편 미만 토픽 제외: {dict(dropped) or '없음'}) |
| 검증 | 저널×연도 그룹 {N_FOLDS}겹 교차검증 |
| tau(예측확률 하한) | {TAU_MAIN} (격자 {TAU_GRID}) |
| 토픽 F1 하한 | {F1_FLOOR} |

- 예측확률은 modified_huber 점수를 정규화한 값이며 **보정되지 않았습니다**. tau는 확률이 아니라 점수 척도의 기준입니다.
- 위 임계값은 부록에 없던 값이므로 CHANGELOG_S2에 "동결 후 추가한 사전 명시"로 기록해야 합니다.

## 2. 교차검증 성능

- 정확도 {acc:.3f}, 토픽별 F1의 단순 평균(macro) {mac:.3f}, 논문 수 가중 평균 {wf:.3f}.
- F1이 하한({F1_FLOOR}) 이상인 토픽 {int(T.above_floor.sum())}개 / {len(T)}개.
- 같은 저널·연도 칸이 학습과 검증에 함께 들어가지 않으므로, 이 값은 새로운 저널·연도 칸으로의 일반화 성능에 해당합니다.

### 기간별 정확도

{tbl(by_period)}

### F1이 낮은 토픽 10개

{tbl(low.round(3), index=False)}

전체 토픽별 F1은 `title_topic_f1.csv`에 있습니다.

## 3. tau 격자별 적용률과 정확도 (교차검증 예측 기준)

{tbl(TT, index=False)}

## 4. 결측 논문 예측 (민감도 분석 전용)

- 토픽 결측 논문(제목 있음, {TOPIC_START}년 이후) {n_miss:,}편에 예측 토픽을 붙였습니다(`title_pred_missing_topic.csv`).
- `pmax ≥ {TAU_MAIN}`이고 토픽 F1 ≥ {F1_FLOOR}인 논문은 {int(M_.use_in_sensitivity.sum()):,}편({M_.use_in_sensitivity.mean():.1%})입니다.
- 이 예측은 주 분석에 쓰지 않습니다. 주 분석의 결측 보정은 성향 가중입니다.

## 5. 한계

1. **교차검증 성능은 결측 논문에 대한 정확도가 아닙니다.** 학습은 토픽이 있는 논문에서 했고, 결측 논문은 임베딩이 없거나 입력이 달라서 결측이 된 논문일 수 있습니다. 그렇다면 결측 논문의 제목 분포는 학습 분포와 다르고, 위 정확도는 낙관적일 수 있습니다. 이 스크립트로는 확인할 수 없습니다.
2. 토픽이 임베딩(제목 외 정보 포함 여부는 확인하지 못함)에서 나왔다면 제목만으로 복원할 수 있는 한계가 있습니다. F1이 낮은 토픽은 제목 정보가 약한 토픽입니다.
3. 토픽 18번(F1 전체 1편)은 학습 대상에서 제외됩니다.
4. 저널×연도 그룹 교차검증은 저널과 토픽의 강한 연관을 일부 제거하므로 일반 교차검증보다 낮게 나옵니다. 이는 의도한 설계입니다.
"""
(S2 / "title_classifier_report.md").write_text(md, encoding="utf-8")
print("저장: title_classifier_report.md, title_topic_f1.csv, title_pred_missing_topic.csv")
print(f"정확도 {acc:.3f} | macro F1 {mac:.3f} | F1 하한 이상 토픽 {int(T.above_floor.sum())}/{len(T)}")

학습 대상 76,984편, 토픽 149개 | 제외 토픽(논문 20편 미만) {}
  fold 1/5 완료 (검증 15,398편)
  fold 2/5 완료 (검증 15,396편)
  fold 3/5 완료 (검증 15,398편)
  fold 4/5 완료 (검증 15,395편)
  fold 5/5 완료 (검증 15,397편)
저장: title_classifier_report.md, title_topic_f1.csv, title_pred_missing_topic.csv
정확도 0.508 | macro F1 0.503 | F1 하한 이상 토픽 138/149


In [43]:
# -*- coding: utf-8 -*-
# =====================================================================================
# S2 항목 2.9 K 민감도: K in {75, 150, 300} x seed 5회 재클러스터링 (OpenAlex 호출 없음, 동결 부록 수정 없음)
# 산출: paper1/02_S2/K_sensitivity.csv (K, seed마다 1행 + 기준 topic150 1행)
# 읽는 것: transport-atlas-data/paper_index.json, paper_emb.bin (int8 양자화 임베딩), analysis_data/paper_semantic_features.csv(pid, emb_idx)
# 측정:
#   - ari_vs_topic150: 재클러스터링 라벨과 기존 topic150의 ARI. 기존 클러스터링과 같은 전처리·알고리즘이 아니면 '재현'이 아니라 '일치도'로만 읽을 것
#   - ari_pairwise_seed: 같은 K의 seed 쌍 평균 ARI(안정성)
#   - nmi_vs_topic150
#   - C1 후보 통계(주제 유효수 = 1/HHI, HHI)의 연도 추세 Spearman 부호와 기준(topic150)과의 부호 일치
# 이 스크립트가 하지 않는 것: 핵심 결과 C2, C3, C5(S5~S7에서 나오는 통계)의 부호 유지 점검. 이 3개는 해당 단계가 끝난 뒤 같은 재클러스터링 라벨로 다시 계산해야 한다.
# 가정(확인하지 못함): 행 구조 = 앞 4바이트 float32 scale + 나머지 int8 벡터. 행을 L2 정규화하므로 양의 scale이면 값에 영향 없음.
# 문법 검사만 했고 실제 자료로는 실행하지 않았습니다. 대용량이라 수 분~수십 분 걸릴 수 있습니다.
# =====================================================================================
import importlib.util, json, itertools, warnings
from pathlib import Path
import numpy as np, pandas as pd
from scipy.stats import spearmanr
warnings.filterwarnings("ignore")

KS = [75, 150, 300]; SEEDS = [2026, 2027, 2028, 2029, 2030]
TOPIC_START = 2000; YMAX = 2025

ROOT = Path.home() / "Research/Minnesota_Transport"
cands = [Path.cwd() / "s2_measurement.py", ROOT / "s2_measurement.py", ROOT / "paper1/s2_measurement.py",
         ROOT / "paper1/02_S2/s2_measurement.py", Path.home() / "Downloads/s2_measurement.py"]
MOD = next((c for c in cands if c.exists()), None); assert MOD is not None, f"s2_measurement.py를 찾지 못했습니다: {[str(c) for c in cands]}"
_sp = importlib.util.spec_from_file_location("s2", MOD); m = importlib.util.module_from_spec(_sp); _sp.loader.exec_module(m)
S2 = m.S2; S2.mkdir(parents=True, exist_ok=True)

edirs = [ROOT / "transport-atlas-data", ROOT.parent / "transport-atlas-data", Path.cwd() / "transport-atlas-data", Path.home() / "transport-atlas-data"]
ED = next((d for d in edirs if (d / "paper_emb.bin").exists() and (d / "paper_index.json").exists()), None)
assert ED is not None, f"임베딩 파일(paper_emb.bin, paper_index.json)을 찾지 못했습니다. 후보: {[str(d) for d in edirs]} -> edirs에 실제 위치를 추가하세요"
SF = ROOT / "analysis_data/paper_semantic_features.csv"; assert SF.exists(), f"{SF} 없음"

from sklearn.cluster import MiniBatchKMeans
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

meta = json.loads((ED / "paper_index.json").read_text(encoding="utf-8")); n, rb = meta["n_papers"], meta["row_bytes"]
Mb = np.fromfile(ED / "paper_emb.bin", dtype=np.uint8).reshape(n, rb)
def norm_pid(s):
    # 정수형 문자열로 통일: 7349, 7349.0, "7349.0", " 7349 " 모두 "7349"
    n = pd.to_numeric(s, errors="coerce")
    return n.astype("Int64").astype(str).where(n.notna(), None)

sf = pd.read_csv(SF, usecols=["emb_idx", "pid"]); print("paper_semantic_features pid dtype:", sf.pid.dtype, "| 예시:", sf.pid.head(3).tolist(), "| pid 결측", int(sf.pid.isna().sum()), "| emb_idx 결측", int(sf.emb_idx.isna().sum()))
sf["pid"] = norm_pid(sf.pid); sf = sf.dropna(subset=["pid", "emb_idx"])

F = m.load_F1X(); print("F1X pid dtype:", F.pid.dtype, "| 예시:", F.pid.head(3).tolist()); F["pid"] = norm_pid(F.pid)
X = F[(F.year >= TOPIC_START) & (F.year <= YMAX) & F.topic150.notna()][["pid", "year", "topic150"]].merge(sf.drop_duplicates("pid"), on="pid", how="inner")
X = X[(X.emb_idx >= 0) & (X.emb_idx < n)].reset_index(drop=True)
if len(X) == 0:
    nt = F[(F.year >= TOPIC_START) & F.topic150.notna()]
    raise SystemExit(f"병합 결과 0편입니다. pid 형식을 확인하세요. F1X(토픽 보유, {TOPIC_START}+) {len(nt):,}편, sf {len(sf):,}행 | F1X pid 예시 {nt.pid.head(3).tolist()} | sf pid 예시 {sf.pid.head(3).tolist()} | 공통 pid {len(set(nt.pid) & set(sf.pid))}개. "
                     "공통이 0이면 두 파일의 pid가 서로 다른 체계일 수 있으니 paper_ml_features.csv(pid, topic150)나 papers_master.csv의 pid 열을 대조하세요.")
print(f"임베딩이 있는 토픽 논문 {len(X):,}편 (F1X, {TOPIC_START}~{YMAX}) | 임베딩 행 {n:,}, 행 바이트 {rb}")
V = Mb[X.emb_idx.values.astype(int), 4:].copy().view(np.int8).astype(np.float32)
V /= np.maximum(np.linalg.norm(V, axis=1, keepdims=True), 1e-12)
ref = X.topic150.astype(int).values; years = sorted(X.year.unique())

def c1_stats(labels):
    rows = []
    for y in years:
        c = np.bincount(labels[X.year.values == y]); p = c[c > 0] / c.sum(); rows.append(float((p ** 2).sum()))
    hhi = np.array(rows); effn = 1.0 / hhi; yy = np.array(years, dtype=float)
    return float(spearmanr(yy, effn).correlation), float(spearmanr(yy, hhi).correlation)

base_eff, base_hhi = c1_stats(ref)
out = [dict(K="topic150(기준)", seed="", n_clusters=len(np.unique(ref)), ari_vs_topic150=1.0, nmi_vs_topic150=1.0, ari_pairwise_seed=np.nan,
            c1_effn_rho=base_eff, c1_hhi_rho=base_hhi, effn_sign_same=True, hhi_sign_same=True)]
for K in KS:
    labs = {}
    for sd in SEEDS:
        km = MiniBatchKMeans(n_clusters=K, random_state=sd, batch_size=4096, n_init=3, max_iter=100); labs[sd] = km.fit_predict(V)
        e, h = c1_stats(labs[sd])
        out.append(dict(K=K, seed=sd, n_clusters=len(np.unique(labs[sd])), ari_vs_topic150=adjusted_rand_score(ref, labs[sd]), nmi_vs_topic150=normalized_mutual_info_score(ref, labs[sd]),
                        ari_pairwise_seed=np.nan, c1_effn_rho=e, c1_hhi_rho=h, effn_sign_same=bool(np.sign(e) == np.sign(base_eff)), hhi_sign_same=bool(np.sign(h) == np.sign(base_hhi))))
        print(f"  K={K} seed={sd} 완료")
    pw = float(np.mean([adjusted_rand_score(labs[a], labs[b]) for a, b in itertools.combinations(SEEDS, 2)]))
    for r in out:
        if r["K"] == K: r["ari_pairwise_seed"] = pw

D = pd.DataFrame(out); D.to_csv(S2 / "K_sensitivity.csv", index=False, encoding="utf-8-sig")
print("\n저장: K_sensitivity.csv")
print(D.round(3).to_string(index=False))
g = D[D.K != "topic150(기준)"].groupby("K").agg(runs=("seed", "size"), ari_vs_ref=("ari_vs_topic150", "mean"), ari_seed=("ari_pairwise_seed", "first"), effn_same=("effn_sign_same", "mean"), hhi_same=("hhi_sign_same", "mean"))
print("\nK별 요약:\n", g.round(3).to_string())
print("※ 기준 C1 추세 rho: 유효수 %.3f, HHI %.3f. |rho|가 작으면 부호는 불안정합니다. C2, C3, C5의 부호 유지는 이 파일로 판단할 수 없습니다." % (base_eff, base_hhi))

paper_semantic_features pid dtype: float64 | 예시: [87546.0, 87963.0, 87992.0] | pid 결측 35 | emb_idx 결측 0
F1X pid dtype: object | 예시: ['7349', '7350', '7351']
임베딩이 있는 토픽 논문 76,984편 (F1X, 2000~2025) | 임베딩 행 95,166, 행 바이트 772
  K=75 seed=2026 완료
  K=75 seed=2027 완료
  K=75 seed=2028 완료
  K=75 seed=2029 완료
  K=75 seed=2030 완료
  K=150 seed=2026 완료
  K=150 seed=2027 완료
  K=150 seed=2028 완료
  K=150 seed=2029 완료
  K=150 seed=2030 완료
  K=300 seed=2026 완료
  K=300 seed=2027 완료
  K=300 seed=2028 완료
  K=300 seed=2029 완료
  K=300 seed=2030 완료

저장: K_sensitivity.csv
           K seed  n_clusters  ari_vs_topic150  nmi_vs_topic150  ari_pairwise_seed  c1_effn_rho  c1_hhi_rho  effn_sign_same  hhi_sign_same
topic150(기준)              149            1.000            1.000                NaN        0.880      -0.880            True           True
          75 2026          75            0.311            0.650              0.366        0.941      -0.941            True           True
          75 2027          7

In [44]:
#

In [46]:
# -*- coding: utf-8 -*-
# =====================================================================================
# 임베딩이 없는 논문(= 토픽 결측)의 특징 점검. 읽기 전용, 파일 저장 없음, OpenAlex 호출 없음.
# Jupyter 셀에 통째로 붙여 실행하세요(s2_measurement.py(v3)가 후보 경로에 있어야 함).
# 확인하는 것:
#   [1] 임베딩 유무 x 토픽 유무 교차표(양방향 일치), sf 쪽 pid 결측·pid_source·연도 범위
#   [2] 임베딩 유무별 DOI 결측, 제목 길이, 비연구 표시(clean_flags), 저널 구성
#   [3] 연도별 임베딩 보유율(전체 / 고정 14개 / 그 외)과 TRR 2005~2008 (2007년 토픽 보유율 하락과 겹치는지)
#   [4] 노트북에서 임베딩 생성 코드(초록 사용 여부) 검색: .ipynb_checkpoints 제외
# 결과로 원인을 단정하지 말고, '같이 움직인다/움직이지 않는다'로만 읽을 것.
# =====================================================================================
import importlib.util, json, re, warnings
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings("ignore"); pd.set_option("display.width", 220); pd.set_option("display.max_rows", 100)

ROOT = Path.home() / "Research/Minnesota_Transport"
cands = [Path.cwd() / "s2_measurement.py", ROOT / "s2_measurement.py", ROOT / "paper1/s2_measurement.py", ROOT / "paper1/02_S2/s2_measurement.py", Path.home() / "Downloads/s2_measurement.py"]
MOD = next((c for c in cands if c.exists()), None); assert MOD is not None, f"s2_measurement.py를 찾지 못했습니다: {[str(c) for c in cands]}"
_sp = importlib.util.spec_from_file_location("s2", MOD); m = importlib.util.module_from_spec(_sp); _sp.loader.exec_module(m)
hdr = m.hdr

def norm_pid(s):
    n = pd.to_numeric(s, errors="coerce"); return n.astype("Int64").astype(str).where(n.notna(), None)

SF = ROOT / "analysis_data/paper_semantic_features.csv"
sf = pd.read_csv(SF); F = m.load_F1X(); F["pid"] = norm_pid(F.pid); sf["pid_n"] = norm_pid(sf.pid)
w = F[F.year >= 2000].copy(); w["has_emb"] = w.pid.isin(set(sf.pid_n.dropna())); w["has_topic"] = w.topic150.notna()
w["tlen"] = w.title.fillna("").str.split().str.len(); w["no_doi"] = w.doi_norm.isna()

hdr("[1] 임베딩 x 토픽 교차표 (F1X, 2000+)")
print(pd.crosstab(w.has_emb, w.has_topic, margins=True).to_string())
print(f"\nsf {len(sf):,}행 | pid 결측 {int(sf.pid.isna().sum())} | F1X(2000+)와 매칭 {int(sf.pid_n.isin(set(w.pid)).sum()):,} | F1X에 없는 sf pid {int((~sf.pid_n.dropna().isin(set(F.pid))).sum()):,}")
print("pid_source:", sf.pid_source.value_counts(dropna=False).to_dict(), "| sf 연도 범위", int(sf.year.min()), "~", int(sf.year.max()))

hdr("[2] 임베딩 유무별 특징")
print("DOI 결측 비율:", w.groupby("has_emb").no_doi.mean().round(4).to_dict(), "| 제목 길이(단어):"); print(w.groupby("has_emb").tlen.describe().round(1).to_string())
cp = m.S2 / "clean_flags.csv"
if cp.exists():
    cf = pd.read_csv(cp, dtype={"pid": str}); cf["pid"] = norm_pid(cf.pid); x = w.merge(cf[["pid", "any_flag"]], on="pid", how="left")
    print("정리 규칙 표시(any_flag) 비율:", x.groupby("has_emb").any_flag.mean().round(4).to_dict(), "(표시는 전체의 0.49%뿐이라 비연구 항목 전체를 대표하지 않음)")
v = w.groupby(["venue_short", "has_emb"]).size().unstack(fill_value=0); v["emb_rate"] = v[True] / (v[True] + v[False]); v["n"] = v[True] + v[False]
print("\n저널별 임베딩 보유율(낮은 순 12):\n", v.sort_values("emb_rate").head(12).round(3).to_string())

hdr("[3] 연도별 임베딩 보유율")
fixed = set(json.loads((m.S0 / "PREREG.json").read_text(encoding="utf-8"))["event"]["fixed_venues"]); w["fixed"] = w.venue_short.isin(fixed)
t = w.groupby("year").has_emb.mean().to_frame("전체"); t["고정14"] = w[w.fixed].groupby("year").has_emb.mean(); t["그외"] = w[~w.fixed].groupby("year").has_emb.mean()
print(t.round(3).T.to_string())
print("\nTRR 2005~2008 임베딩 보유율:", w[(w.venue_short == "TRR") & w.year.between(2005, 2008)].groupby("year").has_emb.agg(["size", "mean"]).round(3).to_dict("index"))
print("(토픽 보유율과 같은 값이면 정의상 당연: 아래 [1]의 일치가 100%인지가 핵심)")

hdr("[4] 임베딩 생성 코드 검색(.ipynb_checkpoints 제외)")
rx = re.compile(r"(abstract_inverted_index|inverted_index|abstract|specter2?_?(encode|base)?|SentenceTransformer|AutoModel|adapters?|emb_idx\s*=|paper_emb\.bin)", re.I); n = 0; seen = {}
for p in sorted(ROOT.rglob("*.ipynb")):
    if ".ipynb_checkpoints" in str(p): continue
    try: cells = json.loads(p.read_text(encoding="utf-8"))["cells"]
    except Exception: continue
    for ci, c in enumerate(cells):
        for li, ln in enumerate("".join(c.get("source", [])).splitlines()):
            if rx.search(ln) and not ln.strip().startswith("#"):
                seen[p.name] = seen.get(p.name, 0) + 1
                if seen[p.name] <= 6 and n < 40: print(f"  [{p.name} | cell {ci} | line {li + 1}] {ln.strip()[:170]}"); n += 1
print("노트북별 일치 줄 수:", seen or "없음")
print("\n읽는 법: abstract 관련 줄이 임베딩 생성 셀에 있으면 입력에 초록이 쓰였을 가능성이 있고(셀을 직접 읽어 확인), 없고 제목만 쓰였다면 초록 부재는 원인이 아닙니다.")


[1] 임베딩 x 토픽 교차표 (F1X, 2000+)
has_topic  False   True    All
has_emb                       
False      14810      0  14810
True           0  76984  76984
All        14810  76984  91794

sf 95,166행 | pid 결측 35 | F1X(2000+)와 매칭 76,984 | F1X에 없는 sf pid 7,040
pid_source: {'doi': 88637, 'title': 6494, nan: 35} | sf 연도 범위 1967 ~ 2026

[2] 임베딩 유무별 특징
DOI 결측 비율: {False: 0.0083, True: 0.0024} | 제목 길이(단어):
           count  mean  std  min   25%   50%   75%   max
has_emb                                                 
False    14810.0  12.0  4.2  1.0   9.0  12.0  14.0  50.0
True     76984.0  12.3  4.0  1.0  10.0  12.0  15.0  73.0
정리 규칙 표시(any_flag) 비율: {False: 0.0057, True: 0.0034} (표시는 전체의 0.49%뿐이라 비연구 항목 전체를 대표하지 않음)

저널별 임베딩 보유율(낮은 순 12):
 has_emb      False  True  emb_rate     n
venue_short                             
VT Mag         453   507     0.528   960
RTBM           392   844     0.683  1236
CACIE          589  1371     0.699  1960
GEIT            66   157     0.704   223
JPT       

In [47]:
#